# 🧪 PyCaret: From Zero to Hero
## Low-Code Machine Learning End to End — A Progressive, Beginner-Friendly Masterclass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlmastery/class/blob/main/colabs/06_data_eval_mlops/pycaret_zero_to_hero.ipynb)
![Python](https://img.shields.io/badge/Python-3.11%2B-blue)
![PyCaret](https://img.shields.io/badge/PyCaret-3.3-blue)

---

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 20px; color: white; text-align: center; margin: 10px 0;">

### Welcome

One running example, a **household budget** dataset, carried through every part: can we predict which households will **overspend next month**, how much they will **spend**, and what their **monthly spending series** looks like ahead? Every part adds one capability of PyCaret and one habit of a careful data scientist.

</div>

## 🎯 Learning Objectives

1. **Explain the PyCaret lifecycle, `setup → compare → create → tune → ensemble → interpret → finalize → predict → save`, and what each call does under the hood.**
2. **Drive **setup** deliberately: imputation, encoding, scaling, transformations, feature selection, outliers, multicollinearity and class imbalance, each as a knob you understand.**
3. **Compare dozens of models on a small budget dataset in one call, and read the **comparison table** and its metrics critically.**
4. **Tune, bag, boost, blend, stack and **calibrate** models, and pick the decision threshold for the business cost.**
5. **Read PyCaret's **plot gallery** (AUC, PR, confusion, threshold, calibration, learning, feature, residuals) and know what each plot can and cannot tell you.**
6. **Use the **regression, clustering, anomaly and time-series** modules on the same budget data.**
7. **Explain predictions with SHAP-style interpretation, and run an honest error analysis.**
8. **Ship it: finalize, save, load, predict on new data, log experiments, check drift, and know the 2026 low-code landscape.**

## 📚 Table of Contents

| Part | Tasks | Topics | Difficulty |
|:----:|:-----:|--------|:----------:|
| 1 | 1-2 | Why low-code ML; the lifecycle; your first experiment | ⭐ |
| 2 | 3-5 | Setup, deliberately: every preprocessing knob; the data, analysed | ⭐⭐ |
| 3 | 6-8 | Classification: compare, metrics, the plot gallery | ⭐⭐ |
| 4 | 9-11 | Tune, ensemble, blend, stack, calibrate, threshold | ⭐⭐⭐ |
| 5 | 12-13 | Regression on next month's spend; residuals and the leakage trap | ⭐⭐ |
| 6 | 14-15 | Interpretability: SHAP, reason codes, what the model learned | ⭐⭐ |
| 7 | 16-17 | Clustering and anomaly detection on budget behaviour | ⭐⭐ |
| 8 | 18-19 | Time series forecasting with the pycaret.time_series module | ⭐⭐⭐ |
| 9 | 20-22 | MLOps: finalize, save, predict, experiment logging, drift, fairness | ⭐⭐⭐ |
| 10 | 23-24 | Error analysis and diagnostics | ⭐⭐ |
| 11 | 25-26 | SOTA 2026: where low-code fits next to AutoGluon, RAPIDS and foundation models | 🔬 |

## 🗺️ How to read

Every concept follows one loop: **🧠 Intuition → 📐 The Math (gently) → 💻 Code → 📊 Picture → ✅ Takeaway**. Read the intuition first, run the code second, and check the takeaway against what you saw. Callouts mark ⚠️ warnings, 🔑 new variables, and 🏆 best practices.

## ⏱️ Estimated Time

2.5 to 3 hours (about 12 minutes of model training on Colab CPU). Each part is self-contained after Part 2, so you can stop and resume.


In [ ]:
# PyCaret 3.3.2 (the latest stable release) supports Python 3.9-3.11 ONLY.
# On 3.12+ it installs and then refuses to import; on 3.13 its pinned deps have no wheels at all.
# So this notebook needs Colab's Python 3.11 image:
#   Runtime > Change runtime type > Runtime version > 2025.07 > Save
# NOTE: pip prints a red "dependency resolver" ERROR block about google-colab wanting pandas 2.2.2
# and plotnine wanting matplotlib 3.8. That is expected and harmless here. PyCaret pins the older
# versions on purpose, this notebook does not use those packages, and the install still succeeds.
import sys
assert sys.version_info[:2] == (3, 11), "Need Python 3.11, got " + sys.version.split()[0] + ". Set Runtime version to 2025.07."
!pip install -q --only-binary=:all: "pycaret[analysis,models]==3.3.2"
print("pip finished on Python", sys.version.split()[0])
print("If this was the first run: Runtime > Restart session, then run this cell again.")

pip finished on Python 3.11.13
If this was the first run: Runtime > Restart session, then run this cell again.


In [ ]:
!pip install pycaret

# Part 0 · Setup and the Data ⭐

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 One dataset for the whole course

Real projects rarely start with a clean CSV. They start with an export like the one below: one row per **household-month**, with income, category spends, a free-text note, a survey score nobody trusts, a referral code that is often missing, and a few surprise expenses. The generator plants real structure (seasonality, city tiers, an interaction between dining and entertainment, missing values) so that every technique in this course has something honest to find.

Two tables are built from it:

- **`hh`**, one row per household: this month's features, **next month's** outcomes as targets. `overspent_next` (classification) and `spend_next` (regression).
- **`ts`**, the monthly spending series of every household, for forecasting.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Why next month

Predicting *this* month's total from *this* month's categories is a leak: the total is their sum, and any model scores a perfect R². Predicting **next** month from what you know now is the real question. Part 4 shows the leak on purpose.

</div>


In [ ]:
#!pip install -q pycaret[analysis,models]   # PyCaret 3.3; ~3 min on Colab (Python 3.11 or 3.12)
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm   # imported before scikit-learn on purpose: on Windows the two ship different OpenMP runtimes and the other order crashes LightGBM fits (harmless on Colab)
from sklearn.model_selection import train_test_split

plt.style.use("seaborn-v0_8-whitegrid"); sns.set_palette("husl")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)

# ------------------------------------------------------------------ the household budget dataset
CITY_TIERS = ["metro", "city", "town"]
NOTES = [
    "quiet month, cooked at home",
    "hosted family for a week",
    "car repair and a new phone",
    "holiday travel with the kids",
    "cut the streaming subscriptions",
    "medical bills came in",
    "birthday party and gifts",
    "worked overtime, ate out a lot",
]


def make_budget(n_households: int = 1500, months: int = 24, seed: int = 42) -> pd.DataFrame:
    # One row per household-month with income, category spends, a note and two targets.
    rng = np.random.default_rng(seed)
    hh = pd.DataFrame({
        "household_id": np.arange(n_households),
        "city_tier": rng.choice(CITY_TIERS, n_households, p=[0.35, 0.4, 0.25]),
        "household_size": rng.integers(1, 6, n_households),
        "owns_car": rng.random(n_households) < 0.55,
        "base_income": rng.lognormal(mean=8.4, sigma=0.35, size=n_households).round(-1),
    })
    tier_rent = {"metro": 0.34, "city": 0.27, "town": 0.20}
    rows = []
    for m in range(months):
        month = pd.Timestamp("2024-01-01") + pd.DateOffset(months=m)
        season = 1.0 + 0.12 * np.sin(2 * np.pi * (m % 12) / 12 - 1.2) + (0.18 if m % 12 == 11 else 0.0)
        income = hh.base_income * (1 + 0.004 * m) * rng.normal(1.0, 0.03, n_households)
        rent = income * hh.city_tier.map(tier_rent).values * rng.normal(1.0, 0.05, n_households)
        groceries = (180 + 95 * hh.household_size) * season * rng.normal(1.0, 0.12, n_households)
        utilities = (60 + 25 * hh.household_size) * (1.25 if month.month in (1, 2, 7, 8) else 1.0) * rng.normal(1.0, 0.1, n_households)
        transport = np.where(hh.owns_car, 210, 90) * season * rng.normal(1.0, 0.15, n_households)
        dining = np.clip(rng.gamma(2.0, 70, n_households) * season, 0, None)
        entertainment = np.clip(rng.gamma(1.8, 60, n_households) * season + 0.35 * dining, 0, None)
        surprise = rng.random(n_households) < 0.06
        other = np.where(surprise, rng.uniform(300, 1200, n_households), rng.gamma(1.5, 40, n_households))
        total = rent + groceries + utilities + transport + dining + entertainment + other
        note = rng.choice(NOTES, n_households)
        note = np.where(surprise, rng.choice(NOTES[2:4] + NOTES[5:6], n_households), note)
        rows.append(pd.DataFrame({
            "household_id": hh.household_id,
            "month": month,
            "city_tier": hh.city_tier,
            "household_size": hh.household_size,
            "owns_car": hh.owns_car,
            "income": income.round(0),
            "rent": rent.round(0),
            "groceries": groceries.round(0),
            "utilities": utilities.round(0),
            "transport": transport.round(0),
            "dining": dining.round(0),
            "entertainment": entertainment.round(0),
            "other": other.round(0),
            "note": note,
            "total_spend": total.round(0),
        }))
    budget = pd.concat(rows, ignore_index=True)
    budget["overspent"] = (budget.total_spend > 0.65 * budget.income).astype(int)  # about 20% of household-months
    # two noise columns and a little missingness, as real exports have
    budget["survey_score"] = rng.integers(1, 6, len(budget))
    budget["referral_code"] = rng.choice(["A1", "B2", "C3", None], len(budget), p=[0.3, 0.3, 0.3, 0.1])
    mask = rng.random(len(budget)) < 0.04
    budget.loc[mask, "dining"] = np.nan
    return budget


def household_table(budget: pd.DataFrame) -> pd.DataFrame:
    # One row per household: this month's features, NEXT month's outcomes as the targets.
    # Features come from the second-to-last month (spends, income, a note) plus
    # history aggregates over all earlier months. Targets come from the last month:
    # `spend_next` (regression) and `overspent_next` (classification). Predicting
    # this month's total from this month's categories would be a leak (the total
    # is their sum); the notebooks use that as a teaching trap.
    months = sorted(budget.month.unique())
    feat_month, target_month = months[-2], months[-1]
    feats = budget[budget.month == feat_month].drop(columns=["month"]).copy()
    hist = budget[budget.month < feat_month].groupby("household_id").agg(
        avg_spend_prev=("total_spend", "mean"),
        overspend_rate_prev=("overspent", "mean"),
        max_dining_prev=("dining", "max"),
    )
    targets = budget[budget.month == target_month][["household_id", "total_spend", "overspent"]].rename(
        columns={"total_spend": "spend_next", "overspent": "overspent_next"})
    table = feats.merge(hist, on="household_id", how="left").merge(targets, on="household_id")
    return table.reset_index(drop=True)


def monthly_series(budget: pd.DataFrame) -> pd.DataFrame:
    # Long format for time-series libraries: item_id, timestamp, target.
    ts = budget[["household_id", "month", "total_spend"]].rename(
        columns={"household_id": "item_id", "month": "timestamp", "total_spend": "target"})
    return ts.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

budget = make_budget(n_households=1500, months=24, seed=RANDOM_STATE)   # one row per household-month
hh = household_table(budget)                                            # one row per household
ts = monthly_series(budget)                                             # item_id, timestamp, target

LABEL_CLS, LABEL_REG = "overspent_next", "spend_next"
ID_COL = "household_id"
FEATURES = [c for c in hh.columns if c not in (ID_COL, LABEL_CLS, LABEL_REG)]
train_df, test_df = train_test_split(hh, test_size=0.25, random_state=RANDOM_STATE, stratify=hh[LABEL_CLS])
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)

print(f"budget: {budget.shape[0]:,} household-months  |  hh: {hh.shape[0]:,} households x {len(FEATURES)} features  |  ts: {ts.item_id.nunique():,} series x {ts.groupby('item_id').size().iloc[0]} months")
print(f"train {len(train_df):,} / test {len(test_df):,}  |  positive rate ({LABEL_CLS}): train {train_df[LABEL_CLS].mean():.1%}, test {test_df[LABEL_CLS].mean():.1%}")
print("All libraries imported successfully!")


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 0.1 Two helpers you will see everywhere

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: one report card for every model

Metrics are only comparable when they are computed the same way on the same split. `report_card` computes the full family for a classifier (threshold, ranking, probabilistic) or a regressor (scale, relative, robust) and returns a tidy one-row table, so every part of this course reports models the same way. `pretty` styles any DataFrame for reading, and `timer` measures wall time so speed claims are numbers, not adjectives.

</div>


In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, brier_score_loss, matthews_corrcoef,
                             mean_absolute_error, mean_squared_error, r2_score, median_absolute_error)
import time as _time
from contextlib import contextmanager

def report_card(y_true, y_pred=None, y_prob=None, name="model", task="classification"):
    # One tidy row of metrics. Classification needs y_pred (labels) and, ideally, y_prob (P(class 1)).
    y_true = np.asarray(y_true)
    if task == "classification":
        y_pred = np.asarray(y_pred)
        row = {"model": name,
               "accuracy": accuracy_score(y_true, y_pred),
               "balanced_acc": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0),
               "recall": recall_score(y_true, y_pred, zero_division=0),
               "f1": f1_score(y_true, y_pred, zero_division=0),
               "mcc": matthews_corrcoef(y_true, y_pred)}
        if y_prob is not None:
            y_prob = np.asarray(y_prob)
            row.update({"roc_auc": roc_auc_score(y_true, y_prob),
                        "pr_auc": average_precision_score(y_true, y_prob),
                        "log_loss": log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
                        "brier": brier_score_loss(y_true, y_prob)})
    else:
        y_pred = np.asarray(y_pred, dtype=float)
        err = y_true - y_pred
        row = {"model": name,
               "MAE": mean_absolute_error(y_true, y_pred),
               "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
               "MedAE": median_absolute_error(y_true, y_pred),
               "MAPE_%": float(np.mean(np.abs(err / np.clip(np.abs(y_true), 1e-9, None))) * 100),
               "R2": r2_score(y_true, y_pred),
               "bias": float(err.mean())}
    return pd.DataFrame([row]).set_index("model")

def pretty(df, fmt="{:.3f}", highlight="max", subset=None):
    # A styled DataFrame: three decimals, the best value per column highlighted.
    sty = df.style.format({c: fmt for c in df.columns if np.issubdtype(df[c].dtype, np.number)}, na_rep="-")
    cols = subset if subset is not None else [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
    if cols and highlight == "max":
        sty = sty.highlight_max(subset=cols, color="#d4edda")
    elif cols and highlight == "min":
        sty = sty.highlight_min(subset=cols, color="#d4edda")
    return sty

@contextmanager
def timer(label="block"):
    # Wall-clock timing you can read: with timer('fit'): ...
    t0 = _time.perf_counter()
    yield
    print(f"⏱️ {label}: {_time.perf_counter() - t0:.2f} s")

# a tiny self-test so the helpers never surprise you later
_demo = report_card([0, 1, 1, 0], [0, 1, 0, 0], [0.2, 0.9, 0.4, 0.1], name="demo")
print(_demo.round(3).to_string())
print(report_card([100., 200.], [110., 190.], name="demo", task="regression").round(2).to_string())


## 0.2 First look at the data

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Before any model, three questions

1. **What is one row?** A household, described by its latest month and its history, with next month's outcomes.
2. **What is the target's balance?** About one household in six overspends next month. Accuracy alone will lie; Part 3 fixes that.
3. **What could leak?** `total_spend` and `overspent` are *this* month's values, legitimate inputs for a *next-month* prediction. The category columns sum to `total_spend`, which is only a leak if the target were this month's total.

</div>


In [ ]:
print(hh.head(3).T.to_string())
print()
print(hh[FEATURES].describe(include="all").T.head(12).to_string())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
hh[LABEL_CLS].value_counts().rename({0: "stays within", 1: "overspends"}).plot.bar(ax=axes[0], color=["#2196F3", "#dc3545"])
axes[0].set_title("Next month: overspends?"); axes[0].set_ylabel("households"); axes[0].tick_params(axis="x", rotation=0)
axes[1].hist(hh[LABEL_REG], bins=40, color="#764ba2"); axes[1].set_title("Next month's total spend"); axes[1].set_xlabel("spend")
sns.boxplot(data=hh, x="city_tier", y="rent", ax=axes[2], order=["metro", "city", "town"]); axes[2].set_title("Rent by city tier")
plt.tight_layout(); plt.show()


# Part 1 · Why Low-Code ML; the Lifecycle; Your First Experiment ⭐

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

PyCaret is a thin, opinionated layer over scikit-learn. Every call you make builds or uses a scikit-learn `Pipeline`, runs cross-validation, and hands you the result as a DataFrame. Nothing is hidden; it is only pre-decided. This Part shows the **lifecycle** (Task 1) with a hand-built scikit-learn baseline next to it, then runs the **first experiment** end to end (Task 2): `setup`, `compare_models`, `predict_model`, and the same call in the object-oriented form.

The point of the Part is not the score. It is to know, for each line, what PyCaret did instead of you.

</div>


# Task 1 · The Lifecycle, and a Baseline Built by Hand

## 1.1 What "low-code" buys, and what it costs

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

A tabular machine-learning project repeats the same twelve decisions every time: how to impute, how to encode, whether to scale, how to split, how many folds, which models, which metric, how to tune, whether to ensemble, whether to calibrate, how to save, how to predict. A **low-code library** makes a sensible default for each decision and exposes it as a keyword argument. You write one line per stage instead of forty, and you can still overrule any default.

The cost is that a default you did not notice is still a decision you made. Part 2 goes through every one of them. Part 1 first shows the stages and what each does under the hood.

</div>

## 1.2 The lifecycle

```
   setup ──► compare_models ──► create_model ──► tune_model ──► ensemble / blend / stack
     │                                                                    │
     │                                                                    ▼
     └──────────────────────────────────────────────── calibrate ──► interpret ──► finalize
                                                                                     │
                                                              save_model ◄── predict_model
```

| Call | What it does under the hood | Returns |
|---|---|---|
| `setup(data, target, ...)` | infers column types, splits train/test (or takes `test_data=`), builds a scikit-learn `Pipeline` of preprocessing steps, fits it on the train partition, chooses the fold generator | an experiment object; `pull()` gives the summary table |
| `compare_models(include=, fold=, sort=)` | for each model: `cross_validate(Pipeline + model)` on the train partition with every registered metric; sorts the means | the best model (or `n_select` of them); `pull()` gives the table |
| `create_model("rf")` | one model, same cross-validation, per-fold table | the fitted model |
| `tune_model(model)` | `RandomizedSearchCV` (or optuna / skopt) over a built-in grid, scored by `optimize=` | the tuned model |
| `ensemble_model` / `blend_models` / `stack_models` | `BaggingClassifier` / `AdaBoostClassifier` / `VotingClassifier` / `StackingClassifier` around your models | the ensemble |
| `calibrate_model(model)` | `CalibratedClassifierCV` | the calibrated model |
| `interpret_model(model)` | SHAP explainer on the transformed features | a plot |
| `finalize_model(model)` | refits the pipeline + model on train **and** test partitions | the final pipeline |
| `predict_model(model, data=)` | runs `data` through the fitted pipeline, appends `prediction_label` / `prediction_score` | a DataFrame |
| `save_model` / `load_model` | `joblib` dump / load of the whole pipeline | a file / a pipeline |

Every item in the middle column is a scikit-learn class you could call yourself. The next cell confirms that the library is installed and counts how many decisions `setup` alone exposes.


In [ ]:
#

import inspect
import pycaret
from pycaret.classification import ClassificationExperiment

# Safely inspect setup parameters
try:
    # Fall back to inspecting module-level setup helper or the class setup method
    from pycaret.classification import setup
    sig = inspect.signature(setup)
except Exception:
    try:
        sig = inspect.signature(ClassificationExperiment().setup)
    except Exception:
        sig = inspect.signature(ClassificationExperiment)

print(f"PyCaret {pycaret.__version__}  |  ClassificationExperiment.setup() exposes {len(sig.parameters)} keyword arguments")
knob_names = [p for p in sig.parameters if p not in ("data", "target", "data_func")]
print("a sample of them:", ", ".join(knob_names[:12]), "...")
print("preprocessing knobs used in Part 2:", ", ".join(k for k in ["numeric_imputation", "categorical_imputation", "normalize",
      "transformation", "remove_multicollinearity", "remove_outliers", "feature_selection", "polynomial_features",
      "bin_numeric_features", "fix_imbalance", "ignore_features", "date_features"] if k in sig.parameters))

## 1.3 The same thing by hand: scikit-learn in ten lines

Before the first PyCaret call, here is the baseline it has to beat: a `ColumnTransformer` (median imputation for numbers, one-hot for the two categorical columns) feeding a `HistGradientBoostingClassifier`, scored with 3-fold stratified cross-validation on `train_df` and then once on `test_df`. Every later Part can be measured against this card.

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Two numbers, two meanings

The cell prints a **cross-validated AUC** (three fits on `train_df`, each scored on the fold it did not see) and a **test AUC** (one fit on all of `train_df`, scored on `test_df`). They are close here but they are not the same quantity, and the whole notebook keeps them apart: cross-validation is for choosing, the test set is for reporting.

</div>


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

DROP = [ID_COL, LABEL_REG, "note"]                       # id, the regression target, free text (2.1 explains `note`)
FEATS = [c for c in FEATURES if c != "note"]
cat_cols = ["city_tier", "referral_code"]
num_cols = [c for c in FEATS if c not in cat_cols]       # includes the bool `owns_car`; HistGB reads it as 0/1

sk_pipe = Pipeline([
    ("prep", ColumnTransformer([("num", SimpleImputer(strategy="median"), num_cols),
                                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])),
    ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
])
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
with timer("hand-built pipeline: 3-fold CV + one fit"):
    cv_auc = cross_val_score(sk_pipe, train_df[FEATS], train_df[LABEL_CLS], cv=cv3, scoring="roc_auc")
    sk_pipe.fit(train_df[FEATS], train_df[LABEL_CLS])
sk_prob = sk_pipe.predict_proba(test_df[FEATS])[:, 1]
baseline_card = report_card(test_df[LABEL_CLS], (sk_prob >= 0.5).astype(int), sk_prob, name="hand-built HistGB")
print(f"cross-validated AUC on train_df: {cv_auc.mean():.4f} +/- {cv_auc.std(ddof=1):.4f}   |   test AUC on test_df: {baseline_card['roc_auc'].iloc[0]:.4f}")
pretty(baseline_card)


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: keep a hand-built baseline next to every AutoML result

Ten lines of scikit-learn, one gradient-boosting model, no tuning. If a low-code tool cannot beat this card on the same test set, the extra machinery has not earned its place, and if it beats it by a lot, something is probably leaking (Part 5 shows what that looks like). `baseline_card` stays in memory for the rest of the notebook for exactly this comparison.

</div>

## 1.4 Task 1 Summary

**✅ Key Takeaways**
- PyCaret's lifecycle is `setup → compare → create → tune → ensemble → calibrate → interpret → finalize → predict → save`; each step wraps a scikit-learn class.
- `setup` alone exposes dozens of keyword arguments; every one is a decision made for you unless you overrule it.
- Cross-validated scores choose models; the held-out `test_df` reports them. The two numbers are kept apart from here on.
- A ten-line hand-built pipeline is the bar every later Part has to clear.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `DROP`, `FEATS` | columns removed before every experiment (`household_id`, `spend_next`, `note`); the remaining feature list |
| `num_cols`, `cat_cols` | the numeric and categorical feature names |
| `sk_pipe`, `cv3` | the hand-built scikit-learn pipeline and its 3-fold stratified splitter |
| `cv_auc`, `sk_prob`, `baseline_card` | its per-fold CV AUC, its test probabilities, its `report_card` row |

**➡️ Next Up:** Task 2 runs the same problem through PyCaret, one call per lifecycle stage.


# Task 2 · Your First Experiment

## 2.1 `setup`: the call that decides everything

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

`setup` looks at every column and decides what it is (number, category, date, text, or the target), splits the rows into a train and a test partition, builds a preprocessing pipeline from its defaults, fits that pipeline on the train partition, and chooses how the cross-validation folds will be cut. It returns an experiment object and prints a summary. With `verbose=False` and `html=False` it prints nothing, and `pull()` returns the summary as a DataFrame, which is the form this notebook uses everywhere.

Three columns are dropped first. `household_id` is an identifier, not a feature. `spend_next` is the *other* target (Part 5). `note` is free text: PyCaret has no reason to know that, so it would treat the column as a categorical variable. Here the notes happen to take only eight distinct values, so it would one-hot encode them into eight columns; a real export with thousands of distinct notes would push it past `max_encoding_ohe=25` and into an ordinal code that means nothing. Task 4 shows both, and the `text_features=` knob that is meant for text.

</div>


In [ ]:
from pycaret.classification import setup

with timer("setup (first experiment)"):
    # Initialize the OOP experiment object
    s = ClassificationExperiment()
    # Call setup to configure the experiment
    s.setup(
        data=train_df.drop(columns=DROP),
        target=LABEL_CLS,
        session_id=RANDOM_STATE,
        fold=3,
        n_jobs=1,
        verbose=False
    )
    from pycaret.classification import set_current_experiment
    set_current_experiment(s)

## 2.2 The summary, row by row

| Row | Value here | Meaning |
|---|---|---|
| **Session id** | 42 | the random seed for the split, the folds and every model (`session_id=RANDOM_STATE`) |
| **Target / Target type** | `overspent_next`, Binary | PyCaret detected two classes; with three or more it says Multiclass and switches metric averaging |
| **Original data shape** | (1125, 19) | rows and columns handed in, target included |
| **Transformed data shape** | (1125, 23) | after preprocessing: the two categorical columns became six one-hot columns (3 city tiers + 3 referral codes), so 19 − 2 + 6 = 23 |
| **Transformed train / test set shape** | (787, 23) / (338, 23) | PyCaret's **own** 70/30 split of the 1,125 rows it was given; `train_size=0.7` by default |
| **Numeric features / Categorical features** | 15 / 2 | inferred from dtypes; the bool `owns_car` is neither and is passed through as is |
| **Rows with missing values** | 11.9% | any row with at least one missing cell (`dining` or `referral_code`) |
| **Preprocess** | True | the pipeline is built; `preprocess=False` hands the raw columns to the models |
| **Imputation type** | simple | `SimpleImputer`; `"iterative"` fits a model per column instead |
| **Numeric / Categorical imputation** | mean / mode | the defaults Task 4 revisits |
| **Maximum one-hot encoding** | 25 | categories with more distinct values than this get an ordinal code instead of dummies |
| **Encoding method** | None | the encoder for those high-cardinality columns (default: ordinal by frequency) |
| **Fold Generator / Fold Number** | StratifiedKFold / 3 | how every `compare_models`, `create_model` and `tune_model` cuts its folds |
| **CPU Jobs / Use GPU** | 1 / False | parallelism; `use_gpu=True` is Part 11's topic |
| **Log Experiment / Experiment Name** | False / clf-default-name | MLflow logging, Part 9 |
| **USI** | a hash | the unique session identifier used in log file names |

The next cell checks the rows that can be recomputed from `train_df` by hand.


In [ ]:
from pycaret.classification import get_config
X_in = train_df.drop(columns=DROP)
n_num = X_in.drop(columns=[LABEL_CLS]).select_dtypes("number").shape[1]
n_cat = X_in.select_dtypes("object").shape[1]
n_bool = X_in.select_dtypes("bool").shape[1]
one_hot_cols = sum(X_in[c].nunique() for c in cat_cols)
print(f"numeric {n_num}  |  categorical {n_cat}  |  bool {n_bool}   (bool columns are passed through unchanged)")
print(f"rows with any missing value: {X_in.isna().any(axis=1).mean():.1%}")
print(f"transformed columns: {X_in.shape[1]} - {n_cat} categorical + {one_hot_cols} one-hot = {X_in.shape[1] - n_cat + one_hot_cols}")
print(f"internal train partition: int(0.7 x {len(X_in)}) = {int(0.7 * len(X_in))} rows  ->  X_train {get_config('X_train').shape}, X_test {get_config('X_test').shape}")
print(f"positive rate: given {X_in[LABEL_CLS].mean():.3f} | internal train {get_config('y_train').mean():.3f} | internal test {get_config('y_test').mean():.3f}   (data_split_stratify=True)")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ The 338 "test" rows are not `test_df`

`setup` was given `train_df` only, so it carved its own 30% hold-out from those 1,125 households. The notebook's real held-out set, `test_df` (375 households), is still outside the experiment. This is the correct default when you have one table and no split of your own, and it is the wrong thing here, because every Part reports on `test_df`. Task 5 fixes it with `test_data=`. Until then, this Part scores `test_df` explicitly through `predict_model(..., data=test_df...)`.

</div>

## 2.3 `compare_models`: every model, one call

`models()` lists the library. This notebook always passes a short `include=` list and `fold=3` so the call stays under half a minute; without `include`, `turbo=True` (the default) already skips the slowest models.


In [ ]:
from pycaret.classification import models, compare_models, pull

library = models()
print(f"{len(library)} classifiers in the library:", ", ".join(library.index))

INCLUDE = ["lr", "dt", "rf", "et", "lightgbm", "gbc", "knn", "nb"]
with timer(f"compare_models: {len(INCLUDE)} models x 3 folds"):
    best = compare_models(include=INCLUDE, fold=3, verbose=False)

cmp_first = pull()
print("\nbest by the default sort (Accuracy):", type(best).__name__)
pretty(cmp_first.drop(columns=["Model"]), subset=[c for c in cmp_first.columns if c not in ("Model", "TT (Sec)")])

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Every number in that table is cross-validation on the training partition

`compare_models` fits each model three times on two-thirds of the 787 internal training rows and scores the remaining third. It never touches the 338 internal test rows, let alone `test_df`. The table ranks hypotheses; the held-out set decides. Also note the default sort: **Accuracy**, which on a target with about 17% positives rewards the model that says "stays within" most often. Part 3 sorts by AUC and adds PR-AUC.

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
names = cmp_first["Model"].str.replace(" Classifier", "").str.replace("Light Gradient Boosting Machine", "LightGBM")
order = cmp_first["AUC"].sort_values(ascending=False).index
ax.bar(names[order], cmp_first.loc[order, "AUC"], color="#2196F3", label="PyCaret CV AUC (mean of 3 folds)")
ax.axhline(cv_auc.mean(), color="#dc3545", ls="--", lw=2, label=f"hand-built HistGB CV AUC = {cv_auc.mean():.3f}")
ax.set_ylim(0.6, 1.0); ax.set_ylabel("cross-validated AUC"); ax.set_xlabel("")
ax.set_title("First compare_models run vs the hand-built baseline"); ax.legend(loc="lower left")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()
n_beat = int((cmp_first["AUC"] > cv_auc.mean()).sum())
print(f"{n_beat} of {len(cmp_first)} PyCaret models beat the hand-built baseline's CV AUC; the spread across models is {cmp_first['AUC'].max() - cmp_first['AUC'].min():.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

One bar per model, sorted by cross-validated AUC, with the hand-built gradient-boosting pipeline as the dashed line. Most models land within a few hundredths of each other and of the baseline: on 787 rows of structured features, the model family matters less than the features. The single decision tree is the clear loser because it outputs only a handful of distinct scores, which ruins a ranking metric. Note that the CV numbers are not exactly comparable: PyCaret's folds were cut on 787 rows, the baseline's on all 1,125.

**The common misread:** "the top bar is the best model". The bars are means of three folds; the fold-to-fold spread (`create_model` prints it) is about as wide as the gaps between neighbours.

</div>

## 2.4 `predict_model` on the notebook's held-out set

`predict_model(model, data=...)` pushes any DataFrame with the same columns through the fitted pipeline and appends two columns: `prediction_label` and `prediction_score`. If the target column is present it also prints a metric row, which `pull()` returns.


In [ ]:
from pycaret.classification import predict_model, pull
import numpy as np
import pandas as pd

preds = predict_model(best, data=test_df.drop(columns=DROP), verbose=False)
print("new columns:", [c for c in preds.columns if c.startswith("prediction")])
print(preds[[LABEL_CLS, "prediction_label", "prediction_score"]].head(6).to_string())
holdout_row = pull()
print("\nPyCaret's own metric row on test_df:")
print(holdout_row.to_string(index=False))

# prediction_score is the probability of the PREDICTED class; recover P(overspends) from it ...
p1_from_score = np.where(preds["prediction_label"] == 1, preds["prediction_score"], 1 - preds["prediction_score"])
# ... or ask for it directly
preds_raw = predict_model(best, data=test_df.drop(columns=DROP), raw_score=True, verbose=False)
print("\nwith raw_score=True the columns are:", [c for c in preds_raw.columns if c.startswith("prediction")])
print("recovered P(class 1) == prediction_score_1:", bool(np.allclose(p1_from_score, preds_raw["prediction_score_1"])))

first_cards = pd.concat([baseline_card,
                         report_card(preds_raw[LABEL_CLS], preds_raw["prediction_label"], preds_raw["prediction_score_1"],
                                     name=f"PyCaret best ({type(best).__name__})")])
pretty(first_cards)

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `prediction_score` is the confidence in the *predicted* class

A row with `prediction_label = 0` and `prediction_score = 0.9` means "90% sure it stays within budget", which is a 10% chance of overspending. Feed that column to `roc_auc_score` and the ranking is scrambled for every predicted-negative row. `raw_score=True` replaces it with one column per class, `prediction_score_0` and `prediction_score_1`; the cell above checks that `prediction_score_1` equals the value recovered by hand. Every ranking, probabilistic or calibration metric in this notebook reads `prediction_score_1`.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: one `report_card` per model, on the same rows

The table above has the hand-built baseline and PyCaret's winner on the same 375 households with the same eleven metrics. That is the only kind of comparison that means anything. PyCaret's own metric row (`pull()` after `predict_model`) agrees with `report_card` on accuracy and AUC; it is a fine quick check, but it does not carry PR-AUC, log loss or Brier, and it changes shape between modules.

</div>

## 2.5 The same experiment, object-oriented

The functional API (`setup`, `compare_models`, ...) keeps one **current experiment** in the module. The OOP form, `ClassificationExperiment()`, holds the same state in an object, so several experiments can coexist (Part 3 uses one for a side experiment) and nothing global is touched. Same methods, same arguments.


In [ ]:
exp_oop = ClassificationExperiment()
with timer("OOP experiment: setup + one model"):
    exp_oop.setup(train_df.drop(columns=DROP), target=LABEL_CLS, session_id=RANDOM_STATE, fold=3, n_jobs=1, verbose=False, html=False)
    lr_oop = exp_oop.create_model("lr", verbose=False)
lr_folds = exp_oop.pull()
print("per-fold table from create_model (OOP form):"); print(lr_folds.round(4).to_string())

# the functional experiment is untouched
print("\nfunctional experiment still holds", get_config("X_train").shape[0], "training rows;",
      "OOP experiment holds", exp_oop.get_config("X_train").shape[0])
print("same class underneath:", type(s).__name__, "==", type(exp_oop).__name__)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading a per-fold table

`create_model` prints one row per fold plus **Mean** and **Std**. The Std row is the number the comparison table hides: two models whose means differ by less than their Std are not distinguishable on this data. Keep this in mind whenever a `compare_models` table looks decisive.

</div>

## 2.6 Task 2 Summary

**✅ Key Takeaways**
- `setup` infers types, splits, builds and fits a preprocessing pipeline, and fixes the fold generator; `pull()` returns its summary, every row of which is a decision.
- Given `train_df` alone, `setup` makes its own 70/30 split; the notebook's `test_df` is outside it until `test_data=` is used (Task 5).
- `compare_models` is cross-validation on the training partition, sorted by Accuracy by default.
- `prediction_score` is the confidence in the predicted class; `raw_score=True` gives `prediction_score_1` = $P(\text{overspends})$.
- `ClassificationExperiment()` is the same machinery as an object; the functional API is a module-level instance of it.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `s` | the current (functional) experiment; a `ClassificationExperiment` |
| `summary` | `pull()` after `setup`, the summary table |
| `INCLUDE`, `best`, `cmp_first` | the short model list, the winner by Accuracy, the comparison table |
| `preds`, `preds_raw`, `holdout_row` | `predict_model` output without and with `raw_score=True`; PyCaret's metric row on `test_df` |
| `first_cards` | `report_card` rows: hand-built baseline vs PyCaret's winner on `test_df` |
| `exp_oop`, `lr_oop`, `lr_folds` | the OOP experiment, its logistic regression, the per-fold table |

**➡️ Next Up:** Part 2 looks at the data properly, then turns every `setup` default into a deliberate choice.


# Part 2 · Setup, Deliberately: Every Preprocessing Knob; the Data, Analysed ⭐⭐

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

Part 1 accepted every default. Part 2 earns them. First the data is analysed the way a careful analyst would before touching a model (Task 3): balance, distributions, seasonality, missingness, correlations, the free-text column. Each finding points at one `setup` argument. Then every preprocessing knob is turned on its own, as a mini-experiment, with the transformed training table inspected before and after (Task 4). Finally the pipeline object and the train/test split are opened up, the `test_data=` handover is made, and the leak that comes from setting up on the full table is shown with numbers (Task 5).

By the end of the Part, the `setup` call that every later Part uses has no accidental defaults in it.

</div>


# Task 3 · The Data, Analysed

## 3.1 Balance and the two target distributions

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Why analyse before `setup`

`setup` will impute, encode and split without asking. Whether its choices are good depends on facts about the table: how skewed the money columns are, which columns are missing and why, which columns say the same thing twice, how rare the positive class is. Five pictures answer those questions; each answer is a `setup` argument in Task 4.

</div>


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
counts = train_df[LABEL_CLS].value_counts().sort_index()
axes[0].bar(["stays within (0)", "overspends (1)"], counts.values, color=["#2196F3", "#dc3545"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 8, f"{v} ({v / counts.sum():.1%})", ha="center")
axes[0].set_title("overspent_next in train_df"); axes[0].set_ylabel("households"); axes[0].set_ylim(0, counts.max() * 1.15)
axes[1].hist(train_df["income"], bins=40, color="#764ba2", alpha=0.85, label=f"skew = {train_df['income'].skew():.2f}")
axes[1].set_title("income (this month)"); axes[1].set_xlabel("income"); axes[1].set_ylabel("households"); axes[1].legend()
axes[2].hist(train_df[LABEL_REG], bins=40, color="#11998e", alpha=0.85, label=f"skew = {train_df[LABEL_REG].skew():.2f}")
axes[2].set_title("spend_next (regression target)"); axes[2].set_xlabel("next month's total spend"); axes[2].legend()
plt.tight_layout(); plt.show()
print(f"positive rate {counts[1] / counts.sum():.3f}  ->  a model that always says 'stays within' has accuracy {counts[0] / counts.sum():.3f}")
print(f"income: median {train_df.income.median():,.0f}, mean {train_df.income.mean():,.0f}, max {train_df.income.max():,.0f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

**Left:** about one household in six overspends next month, so the classes are imbalanced. Accuracy's floor is the share of the majority class, printed under the plot; Part 3 uses metrics that do not reward that floor, and `fix_imbalance` (Task 4) is the `setup` knob for it. **Middle and right:** both money distributions have a long right tail (positive skew: the mean sits to the right of the median). Tree models do not care; linear models, k-nearest neighbours and anything with a distance in it do, which is what `normalize` and `transformation` are for.

**The common misread:** treating the skew as a data error. Incomes really are log-normal; the tail is signal, and the fix is a transform, not a deletion.

</div>

## 3.2 Where the money goes, by city tier


In [ ]:
SPEND_CATS = ["rent", "groceries", "utilities", "transport", "dining", "entertainment", "other"]
shares = train_df[SPEND_CATS].div(train_df["income"], axis=0)
share_by_tier = shares.groupby(train_df["city_tier"]).mean().loc[["metro", "city", "town"]]

fig, ax = plt.subplots(figsize=(10, 4.5))
share_by_tier.plot.bar(stacked=True, ax=ax, colormap="viridis", width=0.6)
for i, tier in enumerate(share_by_tier.index):
    ax.text(i, share_by_tier.loc[tier].sum() + 0.01, f"{share_by_tier.loc[tier].sum():.0%} of income", ha="center")
ax.set_ylabel("mean share of income"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
ax.set_title("Category spend as a share of income, by city tier (train_df)"); ax.legend(title="category", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()
print("rent share by tier:", share_by_tier["rent"].round(3).to_dict())
print("overspend rate by tier:", train_df.groupby("city_tier")[LABEL_CLS].mean().round(3).to_dict())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Each bar is a city tier; the segments are the mean share of income spent on each category. Rent is the biggest segment and it shrinks from metro to town, which is the only thing that separates the tiers; the other categories barely move. The total height is the share of income spent, and the overspend rates printed below follow the rent share. `city_tier` therefore carries real signal, and it is a category with three values, so one-hot encoding (the default) is the right treatment.

**The common misread:** reading the stacked heights as absolute money. Metro households earn about the same as town households in this generator; the tiers differ in rent share, not in income.

</div>

## 3.3 Seasonality in the raw monthly table

`hh` has one row per household and no dates, but it was built from `budget`, the household-month table. The seasonality lives there, and it matters for two reasons: Part 8 forecasts it, and the feature month (`hh`'s "this month") sits at a particular point of the year.


In [ ]:
monthly = budget.groupby("month")["total_spend"].mean()
months = sorted(budget.month.unique()); feat_month, target_month = months[-2], months[-1]

fig, ax = plt.subplots(figsize=(12, 4.3))
ax.plot(monthly.index, monthly.values, marker="o", color="#2196F3", lw=2, label="mean total_spend, all households")
dec = monthly[monthly.index.month == 12]
ax.scatter(dec.index, dec.values, s=140, color="#dc3545", zorder=5, label="December")
ax.axvline(feat_month, color="#28a745", ls="--", lw=1.5, label=f"feature month ({feat_month:%b %Y})")
ax.axvline(target_month, color="#764ba2", ls="--", lw=1.5, label=f"target month ({target_month:%b %Y})")
ax.set_title("Mean monthly total spend across 1,500 households (budget)"); ax.set_xlabel("month"); ax.set_ylabel("mean total spend")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()
lift = dec.mean() / monthly[monthly.index.month != 12].mean() - 1
print(f"December spend is {lift:+.1%} vs other months; the feature month is {feat_month:%B}, the target month is {target_month:%B}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two years of mean monthly spend: a smooth yearly wave plus a December spike (the red dots), riding on a slow upward drift. The dashed lines mark the month whose columns are the features and the month whose outcome is the target. Both are late in the year, so "this month's spend" already sits on the high side of the wave; a model trained on this pair of months has learned one particular season, which is worth remembering when it is applied to a different one (Part 9 checks for drift).

**The common misread:** expecting `hh` to contain this picture. It does not; the seasonality is compressed into the history aggregates (`avg_spend_prev`, `overspend_rate_prev`, `max_dining_prev`) and into the fact that the feature month is fixed.

</div>

## 3.4 Missingness and correlation


In [ ]:
miss = train_df[FEATURES].isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
num_for_corr = train_df.drop(columns=[ID_COL, "note"]).select_dtypes("number")
corr = num_for_corr.corr()

fig, axes = plt.subplots(1, 2, figsize=(17, 6.2), gridspec_kw={"width_ratios": [1, 2.2]})
axes[0].barh(miss.index, miss.values * 100, color="#ffc107")
for i, v in enumerate(miss.values):
    axes[0].text(v * 100 + 0.2, i, f"{v:.1%}", va="center")
axes[0].set_xlabel("% of households missing"); axes[0].set_title("Columns with missing values (train_df)"); axes[0].set_xlim(0, max(miss.values) * 130)
sns.heatmap(corr, ax=axes[1], cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink": 0.7},
            xticklabels=True, yticklabels=True)
axes[1].set_title("Pearson correlation of numeric columns, both targets included")
plt.tight_layout(); plt.show()

pairs = corr.abs().where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(ascending=False)
print("feature pairs with |r| >= 0.85:"); print(pairs[pairs >= 0.85].round(3).to_string())
print("\ncorrelation with the targets:")
print(pd.DataFrame({LABEL_CLS: corr[LABEL_CLS], LABEL_REG: corr[LABEL_REG]}).drop(index=[LABEL_CLS, LABEL_REG]).abs().max(axis=1).sort_values(ascending=False).head(6).round(3).to_string())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

**Left:** only two columns have gaps. `referral_code` is missing for about one household in ten (a real "no referral", not a data error) and `dining` for about one in twenty-five (a genuine unknown). These are different kinds of missing and Task 4 treats them differently. **Right:** the heatmap has a block of strongly correlated money columns. `avg_spend_prev` and `rent`, `household_size` and `utilities` or `groceries`, `total_spend` and `avg_spend_prev`: these say the same thing twice. Trees shrug; linear models and importance plots do not, so `remove_multicollinearity` is a knob worth understanding. The last rows of the print-out show which features correlate most with the targets; `overspent` (this month's outcome) and `total_spend` lead, which is expected since households repeat their behaviour.

**The common misread:** a high correlation with the *target* read as leakage. `overspent` this month is a legitimate input for predicting next month; leakage would be a column computed *from* next month (Part 5).

</div>

## 3.5 The free-text column


In [ ]:
note_stats = train_df.groupby("note").agg(households=("note", "size"), overspend_rate=(LABEL_CLS, "mean")).sort_values("overspend_rate", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(note_stats.index, note_stats["households"], color="#2196F3"); axes[0].set_title("`note` value counts (train_df)"); axes[0].set_xlabel("households"); axes[0].invert_yaxis()
axes[1].barh(note_stats.index, note_stats["overspend_rate"], color="#dc3545")
axes[1].axvline(train_df[LABEL_CLS].mean(), color="k", ls="--", lw=1, label=f"overall rate {train_df[LABEL_CLS].mean():.2f}")
axes[1].set_title("overspend rate next month, by this month's note"); axes[1].set_xlabel("P(overspent_next)"); axes[1].invert_yaxis(); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()
print(note_stats.round(3).to_string())
print(f"\ndistinct notes: {train_df.note.nunique()}  (a real export would have thousands)")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

**Left:** the eight notes are roughly equally common; three of them ("car repair", "holiday travel", "medical bills") are slightly more frequent because the generator attaches them to surprise-expense months. **Right:** the overspend rate by note differs from the overall rate by only a few points, and a note describes *this* month while the target is *next* month, so the column carries weak signal at best. That, plus the fact that a real notes column would be thousands of distinct strings, is why `note` is dropped from the default experiments. Task 4 shows what PyCaret would do with it (eight dummies here; an ordinal code once the count passes 25) and the `text_features=` route for real text.

**The common misread:** "PyCaret handles text". It handles a column you *declare* as text (`text_features=["note"]`, TF-IDF or bag-of-words); undeclared, a text column is just a category with many levels.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Three findings that change the `setup` call

1. Income is right-skewed (skew above 1): a `transformation` for distance- and linear-based models.
2. Several feature pairs correlate above 0.85: `remove_multicollinearity` if coefficients or importances will be read.
3. `referral_code` missing means "none", `dining` missing means "unknown": two different imputation choices, not one default.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: every `setup` argument should answer a question the analysis raised

Turning knobs without a reason is how preprocessing pipelines grow to fifteen steps nobody can explain. The list above is the reason for each knob used later in this notebook; the ones not on the list (`polynomial_features`, `bin_numeric_features`, `remove_outliers`) are shown in Task 4 so you know what they do, and left off by default because nothing here asks for them.

</div>

## 3.6 Task 3 Summary

**✅ Key Takeaways**
- About one household in six overspends; the money columns are right-skewed; rent share separates the city tiers.
- `budget` has a yearly wave plus a December spike; `hh`'s features come from a fixed late-year month.
- Two columns are missing for different reasons; several feature pairs correlate above 0.85.
- `note` has weak next-month signal and, undeclared, would be treated as a category.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `SPEND_CATS`, `shares`, `share_by_tier` | the seven spend categories, their share of income, the mean share per tier |
| `monthly`, `feat_month`, `target_month` | mean spend per month from `budget`; the feature and target months |
| `miss`, `corr`, `pairs` | missing share per column, the correlation matrix, the sorted feature pairs |
| `note_stats` | households and overspend rate per note |

**➡️ Next Up:** Task 4 turns each of those findings into a `setup` knob and shows what it changes.


# Task 4 · The Knobs, One at a Time

## 4.1 A mini-experiment per knob

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 How to see what a knob does

`get_config("X_train_transformed")` is the training partition **after** the pipeline: exactly what the models see. `get_config("pipeline")` is the pipeline itself. So the recipe for every knob is the same: call `setup` with that one argument changed, compare `X_train_transformed` with the default run (rows, columns, values), and read which rows of `pull()` changed. The helper below does the bookkeeping; the default run is the reference.

</div>


In [ ]:
BASE = dict(session_id=RANDOM_STATE, fold=3, n_jobs=1, verbose=False, html=False)   # the notebook's constant setup arguments
KNOB_LOG = []

with timer("reference run (all defaults)"):
    setup(train_df.drop(columns=DROP), target=LABEL_CLS, **BASE)
summary0 = pull().set_index("Description")["Value"].astype(str)
X0 = get_config("X_train_transformed")
steps0 = [name for name, _ in get_config("pipeline").steps]
KNOB_LOG.append({"knob": "defaults", "rows": X0.shape[0], "columns": X0.shape[1], "pipeline steps": ", ".join(steps0)})

def knob(label, data=None, target=LABEL_CLS, **kw):
    # One mini-experiment: setup() with the notebook's constants plus the knob(s) in kw. Prints the change.
    t0 = _time.perf_counter()
    setup(train_df.drop(columns=DROP) if data is None else data, target=target, **BASE, **kw)
    summ = pull().set_index("Description")["Value"].astype(str)
    Xt = get_config("X_train_transformed")
    steps = [name for name, _ in get_config("pipeline").steps]
    KNOB_LOG.append({"knob": label, "rows": Xt.shape[0], "columns": Xt.shape[1], "pipeline steps": ", ".join(steps)})
    print(f"[{label}]  X_train_transformed {X0.shape} -> {Xt.shape}   |  new pipeline steps: {[s_ for s_ in steps if s_ not in steps0] or 'none'}   ({_time.perf_counter() - t0:.1f} s)")
    for k in summ.index:
        if k not in ("USI", "Session id") and summ[k] != summary0.get(k, "-"):
            print(f"      pull(): {k}: {summary0.get(k, '-')} -> {summ[k]}")
    return Xt

print("reference:", X0.shape, "| pipeline steps:", steps0)
print(X0.dtypes.value_counts().to_dict())


## 4.2 Imputation: `numeric_imputation`, `categorical_imputation`

| Argument | Accepts | Fitted on |
|---|---|---|
| `numeric_imputation` | `"mean"` (default), `"median"`, `"mode"`, `"knn"`, `"drop"`, or a number to fill with | the training partition |
| `categorical_imputation` | `"mode"` (default), `"drop"`, or any string to fill with | the training partition |
| `imputation_type="iterative"` | fits a model per column (`numeric_iterative_imputer`, `iterative_imputation_iters`) | slower; for gaps that depend on other columns |

Task 3 found two different kinds of missing. For `dining` a **median** is the robust guess for a skewed column. For `referral_code`, "missing" *means* "no referral", so it deserves its own category, not the most common code.


In [ ]:
X_imp = knob("imputation: median + 'none' category", numeric_imputation="median", categorical_imputation="none")

imp_num = get_config("pipeline").named_steps["numerical_imputer"].transformer
fitted = pd.Series(imp_num.statistics_, index=get_config("pipeline").named_steps["numerical_imputer"].include)
print(f"\nfitted fill value for dining: {fitted['dining']:.1f}  (train-partition median {get_config('X_train')['dining'].median():.1f}; the default mean would be {get_config('X_train')['dining'].mean():.1f})")
print("referral_code columns, default run :", [c for c in X0.columns if c.startswith("referral")])
print("referral_code columns, this run    :", [c for c in X_imp.columns if c.startswith("referral")])
mode_code = train_df["referral_code"].mode()[0]
print(f"\nunder the default, the {train_df['referral_code'].isna().mean():.1%} of households with no referral were silently labelled '{mode_code}' (the mode)")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Mode imputation invents a fact

The default run had three `referral_code` dummies and no way to tell a household that used code A1 from one that used none: one in ten households was quietly assigned the most common code. `categorical_imputation="none"` adds a fourth column, `referral_code_none`, and the model can now learn that "no referral" is its own kind of household. Whenever a gap has a meaning, give it a name instead of a guess.

</div>

## 4.3 Scaling and shape: `normalize`, `normalize_method`, `transformation`

| Argument | Options | Changes |
|---|---|---|
| `normalize=True` + `normalize_method` | `"zscore"` (default: mean 0, std 1), `"minmax"` (0 to 1), `"maxabs"` (−1 to 1), `"robust"` (median 0, IQR 1) | the **axis**: every column on the same scale |
| `transformation=True` + `transformation_method` | `"yeo-johnson"` (default), `"quantile"` | the **shape**: pulls in the long tail so the column looks Gaussian |

Both are fitted on the training partition and applied to everything else. Distance-based models (`knn`), linear models with regularisation (`lr`, `ridge`) and anything solved by gradient descent need the first; the second helps the same models when a column is skewed. Trees are indifferent to both.


In [ ]:
X_z = knob("normalize (z-score)", normalize=True)
X_rob = knob("normalize (robust)", normalize=True, normalize_method="robust")
X_yj = knob("transformation (yeo-johnson)", transformation=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (title, col) in zip(axes, [("raw income", X0["income"]), ("z-scored income (normalize)", X_z["income"]), ("yeo-johnson income (transformation)", X_yj["income"])]):
    ax.hist(col, bins=40, color="#764ba2", alpha=0.85, label=f"skew = {pd.Series(col).skew():.2f}")
    ax.set_title(title); ax.set_xlabel("value"); ax.set_ylabel("households"); ax.legend()
plt.tight_layout(); plt.show()
print(f"z-score: mean {X_z['income'].mean():.3f}, std {X_z['income'].std():.3f}   |   robust: median {X_rob['income'].median():.3f}, IQR {X_rob['income'].quantile(0.75) - X_rob['income'].quantile(0.25):.3f}")
print(f"skew of income: raw {X0['income'].skew():.2f} -> z-scored {X_z['income'].skew():.2f} (unchanged) -> yeo-johnson {X_yj['income'].skew():.2f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Three histograms of the same column. **Scaling** (middle) relabels the x-axis and leaves the shape, and the skew, exactly as it was. **Transformation** (right) changes the shape: the long right tail is folded in and the skew drops to about zero. They answer different problems: scaling makes columns comparable to each other, transformation makes one column better behaved. They can be combined (`normalize=True, transformation=True`), and PyCaret applies the transformation first.

**The common misread:** "I normalized, so the skew is handled". It is not; `normalize` cannot change skew, only `transformation` can.

</div>

## 4.4 Redundancy: `remove_multicollinearity`, `multicollinearity_threshold`

The rule PyCaret implements: for every feature, collect the features whose Pearson correlation with it is at or above the threshold; keep the one with the highest correlation to the target; drop the rest. Task 3 listed the candidate pairs.


In [ ]:
X_mc = knob("remove_multicollinearity (0.9)", remove_multicollinearity=True, multicollinearity_threshold=0.9)
dropped = sorted(set(X0.columns) - set(X_mc.columns))
print("\ndropped:", dropped)
corr_t = X0.astype(float).corr().abs()                     # the same matrix the transformer saw (dummies and bools included)
y_corr = X0.astype(float).corrwith(get_config("y_train").astype(float)).abs()
print("each dropped column, its partners at |r| >= 0.9, and who was kept (highest |r| with the target wins):")
for c in dropped:
    partners = corr_t[c].drop(index=c); partners = partners[partners >= 0.9]
    group = [c] + list(partners.index)
    print(f"   {c:>16}: partners {partners.round(3).to_dict()}  |  |r| with target: { {g: round(y_corr[g], 3) for g in group} }")
print("the transformer's own record:", sorted(set(get_config("pipeline").named_steps["remove_multicollinearity"].transformer.drop_)))


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ The knob drops what it drops, not what you expect

`owns_car` disappeared. Not because a car is unimportant, but because it correlates above 0.9 with `transport` (car owners spend more on transport) and `transport` correlates a little more with the target. Always print the dropped set; a dropped column that you meant to explain in a report is a surprise you want before the meeting, not during it. Tree ensembles do not need this knob at all; use it when coefficients, importances or SHAP values will be read.

</div>

## 4.5 Rows, not columns: `remove_outliers`, `outliers_method`, `outliers_threshold`

An **outlier** is a row whose combination of values is far from every other row. PyCaret's default detector is an Isolation Forest (`"iforest"`; also `"ee"` for elliptic envelope and `"lof"` for local outlier factor), and `outliers_threshold` is the share of training rows it will remove (default 0.05).


In [ ]:
X_out = knob("remove_outliers (iforest, 5%)", remove_outliers=True)
print(f"\ntraining rows: {X0.shape[0]} -> {X_out.shape[0]}  (removed {X0.shape[0] - X_out.shape[0]} = {(X0.shape[0] - X_out.shape[0]) / X0.shape[0]:.1%})")
print(f"internal test rows: {get_config('X_test_transformed').shape[0]}  (unchanged: the step only acts when the pipeline is FIT)")
print(f"positive rate among training rows: before {get_config('y_train').mean():.3f} -> after {get_config('y_train_transformed').mean():.3f}")
print("detector:", get_config("pipeline").named_steps["remove_outliers"].transformer)


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Outlier removal is a training-time step; prediction sees every row

The internal test partition kept all its rows. A pipeline cannot refuse to score a household, so the outliers are removed only while fitting, and at prediction time the model meets the very rows it never trained on. Look at the positive-rate line above before using this knob: if the removed 5% are richer in overspenders than the rest, the detector has been deleting the rare class. Here the difference is small, but that is a number to check, not to assume.

</div>

## 4.6 Fewer columns: `feature_selection`, `feature_selection_method`, `n_features_to_select`

| `feature_selection_method` | Under the hood | Character |
|---|---|---|
| `"classic"` (default) | `SelectFromModel` with `feature_selection_estimator` (LightGBM by default) | keeps the columns the model splits on most |
| `"univariate"` | `SelectKBest` with an F-test per column | fast; ignores interactions |
| `"sequential"` | `SequentialFeatureSelector` | forward search; slow but honest |

`n_features_to_select` is a count (integer) or a share of columns (float, default 0.2).


In [ ]:
X_fs = knob("feature_selection: classic, 8", feature_selection=True, n_features_to_select=8)
kept_classic = list(X_fs.columns)
X_fu = knob("feature_selection: univariate, 8", feature_selection=True, feature_selection_method="univariate", n_features_to_select=8)
kept_uni = list(X_fu.columns)
print("\nclassic (LightGBM importance) keeps :", kept_classic)
print("univariate (F-test) keeps          :", kept_uni)
print(f"agree on {len(set(kept_classic) & set(kept_uni))} of 8:", sorted(set(kept_classic) & set(kept_uni)))


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Two selectors, two answers

The two methods agree on only part of the list. The F-test scores each column alone against the target, so it favours the columns with the strongest marginal correlation (Task 3's list). LightGBM's importance counts how often a column was used in a split, which rewards columns that are useful *in combination* and penalises a column whose information a neighbour already carries. Neither is wrong; they measure different things, and with 22 columns on 787 rows there is little reason to select at all.

**The common misread:** treating the kept list as "the important features". It is the list one particular selector keeps at one particular budget; Part 6 measures importance properly.

</div>

## 4.7 More columns: `polynomial_features`, `bin_numeric_features`


In [ ]:
X_poly = knob("polynomial_features (degree 2)", polynomial_features=True, polynomial_degree=2)
print("\nfirst new columns:", [c for c in X_poly.columns if c not in X0.columns][:6])
same = bool(np.allclose(X_poly["city_tier_city^2"], X_poly["city_tier_city"]))
print(f"'city_tier_city^2' is identical to 'city_tier_city': {same}   (squaring a 0/1 column adds nothing)")

X_bin = knob("bin_numeric_features=['income']", bin_numeric_features=["income"])
binner = get_config("pipeline").named_steps["bin_numeric_features"].transformer
print(f"\nincome became {X_bin['income'].nunique()} ordinal bins {sorted(X_bin['income'].unique())}; strategy={binner.strategy}; edges: {np.round(binner.bin_edges_[0], 0)}")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Degree 2 on 22 columns is 275 columns, many of them junk

`polynomial_features` multiplies every pair of columns, including the one-hot dummies, whose squares are copies of themselves and whose cross products are mostly zero. The column count went from 22 to 275 on 787 rows, which is a recipe for overfitting in a linear model and wasted time in a tree. Use it on a **short, hand-picked** list of numeric columns (Part 6 shows that `dining` × `entertainment` is the interaction worth having), not on the whole table.

</div>

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Binning throws away order inside a bin, on purpose

`bin_numeric_features` replaces `income` with five k-means bins (the edges are printed). A model can no longer tell two households apart inside the same bin, which is a loss for a tree, and a gain for a linear model when the effect of income is not linear (the cut-offs act as a piecewise-constant curve). It is also how a monotone rule becomes a lookup table: useful when the model has to be explained to someone who thinks in brackets.

</div>

## 4.8 Class balance: `fix_imbalance`, `fix_imbalance_method`

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

With one positive in six, a model that hedges towards "stays within" is rewarded by accuracy. **SMOTE** (Synthetic Minority Over-sampling Technique) draws new positive rows on the line segments between a positive and its nearest positive neighbours until the classes are equal. It needs the `imbalanced-learn` package; `fix_imbalance_method` accepts its name (`"SMOTE"`, the default) or any `imblearn` sampler object. Like outlier removal it acts only when the pipeline is fit, and PyCaret fits it inside each cross-validation fold, so the held-out fold stays real.

</div>


In [ ]:
try:
    import imblearn
    HAVE_IMBLEARN = True
    print("imbalanced-learn", imblearn.__version__)
except ImportError:
    HAVE_IMBLEARN = False
    print("imbalanced-learn is not installed: uncomment the line below, or leave fix_imbalance=False and use class weights / threshold tuning (Part 4)")
    # !pip install imbalanced-learn -q

if HAVE_IMBLEARN:
    X_sm = knob("fix_imbalance (SMOTE)", fix_imbalance=True)
    y_sm = get_config("y_train_transformed")
    print(f"\ntraining rows {X0.shape[0]} -> {X_sm.shape[0]}; positive rate {get_config('y_train').mean():.3f} -> {y_sm.mean():.3f}; internal test rows still {get_config('X_test_transformed').shape[0]}")
    frac_tier = int(((X_sm['city_tier_metro'] % 1) != 0).sum()); frac_size = int(((X_sm['household_size'] % 1) != 0).sum())
    print(f"synthetic rows with a fractional city_tier_metro: {frac_tier}; with a fractional household_size: {frac_size}")
    from imblearn.under_sampling import RandomUnderSampler
    X_us = knob("fix_imbalance (RandomUnderSampler)", fix_imbalance=True, fix_imbalance_method=RandomUnderSampler(random_state=RANDOM_STATE))
    print(f"\nunder-sampling instead: {X0.shape[0]} -> {X_us.shape[0]} rows, positive rate {get_config('y_train_transformed').mean():.3f}")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ SMOTE interpolates *after* encoding, and it does not know what a dummy is

The synthetic rows have `city_tier_metro = 0.25` and `household_size = 2.6`: a quarter-metro household with two-and-a-half people. Trees cope, linear models cope, but any explanation of those rows is nonsense, and the more categorical columns you have the worse it gets. Two cleaner options for a 17% positive rate: leave the rows alone and move the **decision threshold** (Part 4, Task 11), or pass a sampler that does not invent rows (`RandomUnderSampler`, shown above, which threw away most of the majority class instead). Resampling changes the training distribution, so the probabilities of a SMOTE-trained model are not calibrated to the real 17% either; Part 4 measures that.

</div>

## 4.9 Leaving columns out, and text done properly: `ignore_features`, `keep_features`, `text_features`

- `ignore_features=[...]`: the columns are dropped before the pipeline; they never reach `X_train`. Cleaner than dropping them in pandas when the same DataFrame is reused.
- `keep_features=[...]`: the opposite protection; those columns survive `feature_selection` and `remove_multicollinearity`.
- `text_features=[...]` + `text_features_method` (`"tf-idf"` default, or `"bow"`): the column is vectorised into word columns instead of being treated as a category.


In [ ]:
with_note = train_df.drop(columns=[ID_COL, LABEL_REG])           # `note` left in on purpose

X_ign = knob("ignore_features=['note', 'survey_score']", data=with_note, ignore_features=["note", "survey_score"])
print("   X_train (before the pipeline) columns:", get_config("X_train").shape[1], "| 'note' in X_train:", "note" in get_config("X_train").columns)

X_note = knob("note left in (treated as a category)", data=with_note)
print("   note columns:", [c for c in X_note.columns if c.startswith("note")][:3], "... total", sum(c.startswith("note") for c in X_note.columns))

X_txt = knob("text_features=['note'] (tf-idf)", data=with_note, text_features=["note"])
print("   word columns:", [c for c in X_txt.columns if c.startswith("note")][:8], "... total", sum(c.startswith("note") for c in X_txt.columns))
print("   pipeline steps:", [name for name, _ in get_config("pipeline").steps])


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Three fates for one column

Ignored, `note` never enters `X_train`. Left in, it becomes eight dummies (one per distinct note) because eight is below `max_encoding_ohe=25`; with thousands of distinct notes it would become one ordinal column ranked by frequency, a number with no meaning. Declared as text, it becomes one TF-IDF column per word ("bills", "car", "cooked", ...), which is what a text column should be, and a `text_embedding` step appears in the pipeline.

**The common misread:** that PyCaret detected the text. It did not; `text_features` is always an explicit declaration.

</div>

## 4.10 Dates: `date_features`, `create_date_columns`

`hh` has no date column, so this mini-experiment uses a slice of `budget`, whose `month` column is a real datetime. PyCaret detects datetime columns automatically; `date_features=` forces the treatment for a column stored as text. `create_date_columns` picks which parts to extract (default `["day", "month", "year"]`; any pandas datetime attribute such as `"quarter"`, `"dayofweek"` or `"is_month_end"` works).


In [ ]:
budget_slice = budget.sample(2000, random_state=RANDOM_STATE)[["month", "income", "groceries", "household_size", "overspent"]]
X_date = knob("date_features (month, year, quarter)", data=budget_slice, target="overspent",
              date_features=["month"], create_date_columns=["month", "year", "quarter"])
print("\ncolumns:", list(X_date.columns))
print(X_date.head(3).to_string())
print("\nthe original datetime column is gone; only its parts remain. To keep a trend, add a 'months since start' column yourself before setup.")


## 4.11 The knob table

The log kept by `knob()` becomes a table: what each argument did to the training partition.


In [ ]:
knob_table = pd.DataFrame(KNOB_LOG).set_index("knob")
knob_table["delta columns"] = knob_table["columns"] - X0.shape[1]
knob_table["delta rows"] = knob_table["rows"] - X0.shape[0]
print(knob_table[["rows", "columns", "delta rows", "delta columns", "pipeline steps"]].to_string())

fig, ax = plt.subplots(figsize=(11, 6))
sub = knob_table.drop(index=[i for i in knob_table.index if "date_features" in i or "note left in" in i or "text_features" in i or "ignore_features" in i])
colors = ["#2196F3" if r == X0.shape[0] else "#dc3545" for r in sub["rows"]]
ax.barh(sub.index, sub["columns"], color=colors)
for i, (c, r) in enumerate(zip(sub["columns"], sub["rows"])):
    ax.text(c + 2, i, f"{c} cols x {r} rows", va="center", fontsize=9)
ax.axvline(X0.shape[1], color="k", ls="--", lw=1, label=f"defaults: {X0.shape[1]} columns")
ax.set_xscale("log"); ax.set_xlabel("columns in X_train_transformed (log scale)"); ax.invert_yaxis()
ax.set_title("What each setup knob did to the training partition (red = row count changed)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"{len(KNOB_LOG)} mini-experiments logged")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

One bar per knob, the length being the number of columns the models see (log scale, because `polynomial_features` is off the chart otherwise). Most knobs leave the shape alone and change values (imputation, scaling, transformation, binning). Two shrink the columns (multicollinearity, feature selection), one explodes them (polynomial), and the red bars are the two that change the **row** count: outlier removal takes rows away, SMOTE adds them.

**The common misread:** that a knob which leaves the shape unchanged did nothing. `normalize` and `transformation` changed every value in fifteen columns and the shape not at all.

</div>

| Knob | What it changed here | Use it when |
|---|---|---|
| `numeric_imputation="median"` | fill value for `dining` moved from the mean to the median | the column is skewed |
| `categorical_imputation="none"` | a fourth `referral_code_none` dummy | "missing" has a meaning |
| `normalize=True` | every numeric column to mean 0 / std 1; shape unchanged | `lr`, `knn`, `svm`, neural nets; anything with a distance or a penalty |
| `transformation=True` | income skew from above 1 to about 0 | skewed columns and a linear or distance model |
| `remove_multicollinearity=True` | dropped 5 columns, `owns_car` among them | coefficients / importances will be read |
| `remove_outliers=True` | 5% of training rows gone; test rows untouched | a few rows are known to be corrupt, not merely rare |
| `feature_selection=True` | 22 → 8 columns; two methods agree only partly | hundreds of columns, or a model that must be small |
| `polynomial_features=True` | 22 → 275 columns | a short list of numeric columns whose interaction matters |
| `bin_numeric_features` | income → 5 ordinal bins | a non-linear effect in a linear model; bracket-style explanations |
| `fix_imbalance=True` | training rows to a 50/50 mix, some of them fractional | recall on the rare class matters and threshold tuning is not enough |
| `ignore_features` | column never enters `X_train` | identifiers, leaks, columns not available at prediction time |
| `text_features` | one TF-IDF column per word | the column really is text |
| `date_features` | datetime → its parts | any date column; add your own trend column first |

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: change one knob, look at `X_train_transformed`, then decide

Every row in the table above came from one `setup` call and one look at the transformed table. That habit catches the surprises this Task showed: the mode-imputed referral codes, the dropped `owns_car`, the fractional households, the 275 polynomial columns. Twenty seconds per knob is cheap next to a week of debugging a pipeline you never inspected.

</div>

## 4.12 Task 4 Summary

**✅ Key Takeaways**
- `get_config("X_train_transformed")` is what the models see; compare it with the default run to learn what a knob does.
- Imputation defaults (mean, mode) are guesses; a skewed column wants the median and a meaningful gap wants its own category.
- `normalize` changes the axis, `transformation` changes the shape; only the second fixes skew.
- `remove_outliers` and `fix_imbalance` act at fit time only; the test partition keeps every row.
- `remove_multicollinearity` keeps the partner most correlated with the target, which can drop a column you meant to keep.
- `polynomial_features` on the full table explodes the column count; SMOTE produces fractional dummies; `text_features` must be declared.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `BASE` | the constant `setup` arguments (`session_id`, `fold`, `n_jobs`, `verbose`, `html`) |
| `knob(label, data=None, target=..., **kw)` | runs one mini-experiment, logs it, prints what changed, returns `X_train_transformed` |
| `X0`, `summary0`, `steps0` | the reference run's transformed table, summary and pipeline steps |
| `X_imp`, `X_z`, `X_rob`, `X_yj`, `X_mc`, `X_out`, `X_fs`, `X_fu`, `X_poly`, `X_bin`, `X_sm`, `X_us`, `X_ign`, `X_note`, `X_txt`, `X_date` | the transformed table under each knob |
| `HAVE_IMBLEARN` | whether `imbalanced-learn` is installed |
| `KNOB_LOG`, `knob_table` | the log of every mini-experiment and its table |

**➡️ Next Up:** Task 5 opens the pipeline object, explains the split, hands PyCaret the notebook's own test set, and shows the leak that comes from not doing so.


# Task 5 · The Pipeline Object and the Split

## 5.1 What `setup` built

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

Everything Task 4 did lives in one object: a scikit-learn `Pipeline` whose steps are `TransformerWrapper`s around ordinary scikit-learn and `category_encoders` transformers. `get_config("pipeline")` returns it. When a model is trained, PyCaret clones this pipeline, appends the model as the last step, and fits the whole thing; that is why `predict_model` and `save_model` work on raw rows. The experiment object also exposes the data at every stage: `dataset` (as given), `dataset_transformed` (after the pipeline), and the `train` / `test` halves of each.

</div>


In [ ]:
with timer("setup with the knobs this notebook keeps"):
    s = setup(train_df.drop(columns=DROP), target=LABEL_CLS, normalize=True, **BASE)   # each functional setup() is a NEW experiment object
pipe = get_config("pipeline")

print("pipeline steps:")
for name, step in pipe.steps:
    inner = step.transformer if hasattr(step, "transformer") else step
    cols = getattr(step, "include", None)
    print(f"   {name:>22}: {type(inner).__name__:<18} on {len(cols) if cols else 'all'} column(s)")

print("\ndata at each stage (rows x columns):")
for attr in ["dataset", "dataset_transformed", "train", "train_transformed", "test", "test_transformed"]:
    print(f"   s.{attr:<20} {getattr(s, attr).shape}")
print("\nX_train_transformed dtypes:", get_config("X_train_transformed").dtypes.value_counts().to_dict())
print("target column is not in X_train_transformed:", LABEL_CLS not in get_config("X_train_transformed").columns)


## 5.2 How the split is made

| Argument | Default | Meaning |
|---|---|---|
| `train_size` | 0.7 | share of the given rows that become the train partition; the rest is PyCaret's internal test partition |
| `data_split_stratify` | True | keep the class ratio equal in both partitions (a `StratifiedShuffleSplit` on the target); a list of column names stratifies on those instead |
| `data_split_shuffle` | True | shuffle before splitting; set False for time-ordered rows |
| `fold_strategy` | `"stratifiedkfold"` | the cross-validation generator: `"kfold"`, `"groupkfold"` (with `fold_groups`), `"timeseries"`, or any scikit-learn splitter object |
| `fold` | 10 | number of folds; this notebook uses 3 |
| `fold_shuffle` | False | whether the fold generator shuffles; with `False` the folds are contiguous blocks of the (already shuffled) train partition |
| `test_data` | None | **your own** test partition; `train_size` is then ignored |
| `index` | True | keep the DataFrame index; `False` resets it to a fresh `RangeIndex` |

The split is made **before** the pipeline is fitted, so the pipeline's statistics come from the train partition only. The next cell proves that with the imputer.


In [ ]:
fg = get_config("fold_generator")
print("fold generator:", fg)
print(f"train partition {get_config('X_train').shape[0]} rows = int({int(len(train_df))} x 0.7); internal test {get_config('X_test').shape[0]} rows")
print(f"positive rate: train {get_config('y_train').mean():.3f} | internal test {get_config('y_test').mean():.3f}   (stratified)")

imp = pipe.named_steps["numerical_imputer"]
fitted_mean = pd.Series(imp.transformer.statistics_, index=imp.include)["dining"]
print(f"\nimputer's fitted mean for dining: {fitted_mean:.3f}")
print(f"   mean of dining in the TRAIN partition : {get_config('X_train')['dining'].mean():.3f}   <- matches")
print(f"   mean of dining in all rows given      : {train_df['dining'].mean():.3f}   <- does not: the internal test rows were not used")


## 5.3 Handing PyCaret the notebook's own test set: `test_data=`

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `test_data=` needs `index=False` when both frames start at 0

`train_df` and `test_df` were both `reset_index(drop=True)` in Part 0, so each has a `RangeIndex` starting at 0. PyCaret concatenates the two frames and keeps the index by default (`index=True`), which produces duplicate labels and this error:

`ValueError: Invalid value for the index parameter. There are duplicate indices in the dataset. Use index=False to reset the index to RangeIndex.`

`index=False` gives the concatenated table a fresh index. Every `setup` in this notebook that passes `test_data=` also passes `index=False`. The next cell triggers the error on purpose, then does it right.

</div>


In [ ]:
try:
    setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP), normalize=True, **BASE)
    print("no error (the two frames had distinct indices)")
except ValueError as e:
    print("ValueError:", str(e)[:140])

with timer("setup with test_data=test_df, index=False"):
    exp = setup(train_df.drop(columns=DROP), target=LABEL_CLS,
                test_data=test_df.drop(columns=DROP),     # the notebook's own held-out set
                normalize=True, index=False, **BASE)

summary_td = pull().set_index("Description")["Value"]
for key in ["Original data shape", "Transformed train set shape", "Transformed test set shape", "Fold Generator", "Fold Number", "Normalize"]:
    print(f"{key:>30}: {summary_td[key]}")
print(f"\nX_train rows: {get_config('X_train').shape[0]} == len(train_df) {len(train_df)}  |  X_test rows: {get_config('X_test').shape[0]} == len(test_df) {len(test_df)}")
print(f"held-out positive rate inside the experiment: {get_config('y_test').mean():.3f} == test_df: {test_df[LABEL_CLS].mean():.3f}")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: one split, made once, passed in

With `test_data=test_df`, `predict_model(model)` with no `data=` scores exactly the 375 households every other Part reports on, `plot_model` draws its held-out plots on them, and `finalize_model` (Part 9) knows which rows to add back. Without it, PyCaret's internal test partition is 338 different households from inside `train_df`, and "test AUC" means two different things in two Parts of the same notebook. The `setup` call above is the one Parts 3 and 4 open with.

</div>

## 5.4 The leak: `setup` on the full table

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

The tempting shortcut is `setup(hh, ...)`: hand PyCaret the whole table and let it split. PyCaret does split honestly, but its split is not yours. About 70% of the households in `test_df` end up inside PyCaret's train partition, so when you later score `test_df` with `predict_model(..., data=test_df)`, most of those rows were seen during training. A tree-based model remembers rows; the score goes up; nothing warns you.

</div>


In [ ]:
from pycaret.classification import create_model
leak_rows = []
for label, table in [("honest: setup(train_df), score test_df", train_df), ("LEAK: setup(hh), score test_df", hh)]:
    setup(table.drop(columns=DROP), target=LABEL_CLS, normalize=True, **BASE)
    seen = set(table.loc[get_config("X_train").index, ID_COL]) & set(test_df[ID_COL])   # index=True keeps the table's own index
    for mid in ["dt", "rf"]:
        m = create_model(mid, verbose=False)
        p = predict_model(m, data=test_df.drop(columns=DROP), raw_score=True, verbose=False)
        leak_rows.append({"run": label, "model": mid, "test_df households seen in training": len(seen),
                          "test AUC": roc_auc_score(p[LABEL_CLS], p["prediction_score_1"]),
                          "test accuracy": accuracy_score(p[LABEL_CLS], p["prediction_label"])})
leak_table = pd.DataFrame(leak_rows).set_index(["run", "model"])
print(leak_table.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.3))
w = 0.35; x = np.arange(2)
honest = leak_table.xs("honest: setup(train_df), score test_df")["test AUC"].values
leaky = leak_table.xs("LEAK: setup(hh), score test_df")["test AUC"].values
ax.bar(x - w / 2, honest, w, color="#2196F3", label="honest: setup on train_df")
ax.bar(x + w / 2, leaky, w, color="#dc3545", label="leak: setup on the full table")
for i in range(2):
    ax.text(x[i] - w / 2, honest[i] + 0.005, f"{honest[i]:.3f}", ha="center"); ax.text(x[i] + w / 2, leaky[i] + 0.005, f"{leaky[i]:.3f}", ha="center")
ax.set_xticks(x); ax.set_xticklabels(["decision tree", "random forest"]); ax.set_ylabel("AUC on test_df"); ax.set_ylim(0.5, 1.05)
ax.set_title("The same test set, scored by models that did or did not train on most of it"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()
print(f"inflation from the leak: decision tree {leaky[0] - honest[0]:+.3f} AUC, random forest {leaky[1] - honest[1]:+.3f} AUC")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two models, each trained twice: on `train_df` (blue) and on the full `hh` (red), both scored on `test_df`. The red bars are higher, and the decision tree's is much higher: a single tree memorises the rows it saw, and it saw most of the test set. The random forest is inflated too, by less, because bagging averages out some of the memorisation. Neither red number is a measurement of anything; both look like good news.

**The common misread:** "PyCaret split the data, so it is fine". PyCaret's split protects PyCaret's internal test partition. It cannot protect a test set it was never told about.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ The quieter version of the same leak

The dramatic form above is scoring rows the model trained on. The quiet form is preprocessing on the full table *before* `setup`: filling `dining` with the mean of all 1,500 households, scaling income with the overall standard deviation, or fitting SMOTE on everything. Each one lets a little of the test set into the training statistics; none of them changes a shape or raises an error. The rule that avoids both: split first, then hand `setup` the train rows and `test_data=` the test rows, and let the pipeline fit itself on the train partition only, as 5.2 showed it does.

</div>

## 5.5 Task 5 Summary

**✅ Key Takeaways**
- `get_config("pipeline")` is a scikit-learn `Pipeline` of wrapped transformers; models are appended to a clone of it, which is why `predict_model` takes raw rows.
- The split happens before the pipeline is fitted; the imputer's statistics come from the train partition only.
- `test_data=test_df` with `index=False` makes the experiment's held-out set the notebook's; every Part from here on opens `setup` this way.
- `setup` on the full table and scoring your own test set is a leak PyCaret cannot see; a single tree makes it obvious, a forest hides it.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `pipe` | the preprocessing pipeline of the `normalize=True` experiment |
| `fg` | its fold generator (`StratifiedKFold`, 3 folds, no shuffle) |
| `exp` | the experiment opened with `test_data=test_df`, `index=False`, `normalize=True`: the form Parts 3 and 4 use |
| `summary_td` | its `pull()` summary |
| `leak_table`, `honest`, `leaky` | test AUC / accuracy of `dt` and `rf` trained on `train_df` vs on the full table |

**➡️ Next Up:** Part 3 opens the same experiment and reads everything `compare_models` produces: the table column by column, the metric families, and the plot gallery.


In [ ]:
# leave the current experiment in the recommended state for the next Part
exp = setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP),
            normalize=True, index=False, **BASE)
print("Parts 1-2 done. Current experiment: train", get_config("X_train").shape, "| held-out", get_config("X_test").shape,
      "| knobs on:", {k: summary_td[k] for k in ["Normalize"] if k in summary_td.index})


# Part 3 · Classification: Compare, Metrics, the Plot Gallery ⭐⭐

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

Part 1 ran `compare_models` once and took the winner on faith. This Part slows down and reads everything the call produces: **the comparison table** column by column (Task 6), the **four metric families** and where accuracy lies on a 17%-positive target (Task 7), and the **plot gallery** with what each picture can and cannot tell you (Task 8).

Parts 3 and 4 share one experiment, opened below with `test_data=test_df`, so every "test" number from here to the end of Part 4 is computed on the notebook's own held-out set, the same 375 households every other Part reports on.

</div>


# Task 6 · `compare_models` and the Comparison Table

## 6.1 One experiment for two Parts

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 What `compare_models` does

It is a loop, nothing more: for every model in its library, **fit the preprocessing pipeline plus that model inside each cross-validation fold**, score the held-out fold with every registered metric, average the folds, and sort. The *train partition* of the experiment is the only data it touches. The held-out partition (`test_data`) waits for `predict_model`.

Three arguments control the cost: `include` (which models), `fold` (how many fits per model), and `turbo` (skip the slow models when `include` is not given).

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `index=False` and `normalize=True` are deliberate

`train_df` and `test_df` both start their index at 0. With PyCaret's default `index=True`, the two partitions would carry duplicate index labels, so `index=False` resets them. `normalize=True` (z-scores; a Part 2 knob) is here for speed as much as for accuracy: the next cell shows why.

</div>


In [ ]:
from pycaret.classification import *
from pycaret.classification import (ClassificationExperiment, setup, compare_models, create_model, tune_model,
                                    ensemble_model, blend_models, stack_models, calibrate_model, optimize_threshold,
                                    plot_model, predict_model, add_metric, remove_metric, get_metrics, pull, get_config)
import inspect

DROP = [ID_COL, LABEL_REG, "note"]      # id, the regression target, and free text (Part 2 explains `note`)
sig = inspect.signature(setup).parameters
assert {"test_data", "session_id", "normalize", "index", "n_jobs"} <= set(sig), "argument names changed"

with timer("setup (Parts 3-4)"):
    exp = setup(train_df.drop(columns=DROP), target=LABEL_CLS,
                test_data=test_df.drop(columns=DROP),   # the notebook's own held-out set
                normalize=True, index=False,
                fold=3, session_id=RANDOM_STATE, n_jobs=1, verbose=False, html=False)

summary = pull().set_index("Description")["Value"]
for key in ["Target", "Original data shape", "Transformed data shape", "Transformed train set shape",
            "Transformed test set shape", "Fold Generator", "Fold Number", "Normalize", "CPU Jobs"]:
    if key in summary.index:
        print(f"{key:>30}: {summary[key]}")
print(f"\nheld-out rows inside the experiment: {len(get_config('X_test'))}  ==  len(test_df): {len(test_df)}")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: the split lives outside PyCaret

`test_data=` hands PyCaret the split the notebook already made. Without it, `setup` would carve its own 70/30 split from `train_df`, and the "test" numbers in this Part would be on 338 different households than every other Part reports on. One split, made once, reused everywhere.

</div>

## 6.2 Why `normalize=True` before comparing

The next cell fits scikit-learn's `LogisticRegression` (PyCaret's `lr`, with `max_iter=1000`) twice on the numeric columns of the training partition: raw, then z-scored. Watch `n_iter_`, the number of solver iterations used.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X_raw = get_config("X_train").select_dtypes("number").fillna(0)
y_raw = get_config("y_train")
X_scaled = StandardScaler().fit_transform(X_raw)

rows = []
for name, X in [("raw money columns", X_raw), ("z-scored", X_scaled)]:
    t0 = _time.perf_counter()
    m = LogisticRegression(max_iter=1000).fit(X, y_raw)
    rows.append({"features": name, "solver iterations": int(m.n_iter_[0]), "fit seconds": _time.perf_counter() - t0})
lr_speed = pd.DataFrame(rows).set_index("features")
print(lr_speed.round(3).to_string())
print(f"\nraw fit hit the max_iter cap: {lr_speed.loc['raw money columns', 'solver iterations'] >= 1000}")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ A slow `lr` is a warning sign, not a property of logistic regression

On the raw columns (income in the thousands, `overspend_rate_prev` between 0 and 1) the solver ran out of its 1000 iterations without converging; on z-scored features it needs a few dozen. The `TT (Sec)` column of the comparison table would have shown `lr` as the slowest model for no good reason, and every later call that contains it (`blend_models`, `stack_models`, the learning-curve plot) would inherit the cost. Scaling is free for tree models, so this experiment turns it on for everyone.

</div>

## 6.3 The comparison, sorted by AUC


In [ ]:
INCLUDE = ["lr", "dt", "rf", "et", "lightgbm", "gbc", "knn", "nb"]   # short list: the stage must stay fast
with timer(f"compare_models on {len(INCLUDE)} models x 3 folds"):
    top3 = compare_models(include=INCLUDE, fold=3, sort="AUC", n_select=3, verbose=False)

cmp = pull()                       # the table behind the call, as a DataFrame
print(cmp.to_string())
print("\nn_select=3 returned:", [type(m).__name__ for m in top3])
best = top3[0]
pretty(cmp.drop(columns=["Model"]), subset=[c for c in cmp.columns if c not in ("Model", "TT (Sec)")])


## 6.4 The table, column by column

| Column | What it is | Needs | Reads as |
|---|---|---|---|
| **Accuracy** | share of households labelled correctly at the 0.5 threshold | labels | "always stays within" already scores about 0.83 here, so treat 0.89 with suspicion |
| **AUC** | ROC area: the probability that a random overspender scores above a random non-overspender | scores | 0.5 is a coin flip, 1.0 is a perfect ranking; threshold-free |
| **Recall** | of the households that overspend, the share caught | labels | the cost of a miss lives here |
| **Prec.** | of the households flagged, the share that overspend | labels | the cost of a false alarm lives here |
| **F1** | harmonic mean of precision and recall | labels | one number for the positive class; ignores true negatives |
| **Kappa** | agreement with the truth beyond chance | labels | 0 = chance-level, 1 = perfect; corrects accuracy for the base rate |
| **MCC** | correlation between predicted and true labels | labels | uses all four confusion-matrix cells; the most honest single label metric on imbalanced data |
| **TT (Sec)** | wall-clock seconds for the 3 fits | the machine | changes with the CPU and its load; never compare across runs |

For a **binary** target PyCaret computes Recall, Prec. and F1 for the positive class (`average="binary"`), which is what you want; for multiclass it switches to weighted averages. Task 7 checks that claim against `report_card`.

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Every number above is cross-validation on the training partition

`compare_models` never touches `test_data`. The table ranks models by their fold-averaged score on 1,125 training households; the 375 held-out households are still untouched. That is exactly right (the test set must stay unused until the choice is made) and it is why the "best" row is a hypothesis, not a result.

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
plot_df = cmp[["AUC", "F1"]].copy()
plot_df.index = cmp["Model"].str.replace(" Classifier", "").str.replace("Light Gradient Boosting Machine", "LightGBM")
plot_df.plot.bar(ax=ax, color=["#2196F3", "#764ba2"], width=0.75)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="AUC of a coin flip")
ax.set_ylim(0.4, 1.0); ax.set_ylabel("cross-validated score (mean of 3 folds)"); ax.set_xlabel("")
ax.set_title("compare_models: AUC and F1 per model (sorted by AUC)"); ax.legend(loc="upper right")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()
gap = cmp["AUC"].max() - cmp["AUC"].min()
print(f"AUC spread across models: {gap:.3f}   |   F1 spread: {cmp['F1'].max() - cmp['F1'].min():.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two bars per model: **AUC** (ranking quality) and **F1** (label quality at the default 0.5 threshold). The AUC bars are tall and close together; the F1 bars are shorter and spread wider. The decision tree is the outlier on both: a single tree ranks badly because it produces only a handful of distinct scores. The order of models changes between the two bars, which is the point: "best" depends on the metric you sort by.

**The common misread:** treating the AUC gap between neighbours as a decision. With 3 folds of 375 rows each, differences in the second decimal are inside fold-to-fold noise (`create_model` prints the per-fold std; Task 9 uses it).

</div>

## 6.5 Adding your own metric: `add_metric`

The table has no **PR-AUC** (average precision), the ranking metric that focuses on the positive class. `add_metric` registers any scikit-learn scoring function so that every later `compare_models`, `create_model` and `tune_model` reports it:

- `id` and `name`: the row key and the column header,
- `score_func`: a function `(y_true, y_pred_or_proba) -> float`,
- `target`: `"pred"` (labels), `"pred_proba"` (probabilities) or `"threshold"` (decision function),
- `greater_is_better`: lets `sort=` and `optimize=` know which way is up.


In [ ]:
from sklearn.metrics import average_precision_score

if "pr_auc" in get_metrics().index:       # idempotent: re-running the cell must not fail
    remove_metric("pr_auc")
add_metric("pr_auc", "PR-AUC", average_precision_score, target="pred_proba", greater_is_better=True)

metrics_now = get_metrics()[["Display Name", "Target", "Greater is Better", "Custom"]]
print(metrics_now.to_string())

with timer("compare_models on the top 3, now with PR-AUC"):
    best_pr = compare_models(include=top3, fold=3, sort="PR-AUC", verbose=False)
cmp_pr = pull()
print(); print(cmp_pr.to_string())
print("\nsorted by PR-AUC, the winner is:", cmp_pr.iloc[0]["Model"])


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: register the metric you will decide by, then `sort=` by it

`compare_models(include=top3, ...)` accepts fitted model objects as well as ids, so re-scoring a short list with a new metric costs three fits, not eight. Once the metric is registered it also becomes a legal `optimize=` target for `tune_model` and `optimize_threshold` in Part 4, so the whole pipeline optimises the number you care about. `get_metrics()` shows the registry; `remove_metric("pr_auc")` takes it back.

</div>

## 6.6 Task 6 Summary

**✅ Key Takeaways**
- `compare_models` is a cross-validation loop on the train partition; `test_data` is not touched until `predict_model`.
- Accuracy, Recall, Prec., F1, Kappa and MCC are label metrics at the 0.5 threshold; AUC (and PR-AUC) are ranking metrics; `TT (Sec)` is machine time.
- Unscaled columns made `lr` hit its iteration cap; `normalize=True` fixed the speed, and the table's ordering changes with `sort=`.
- `add_metric` registers a custom scorer for every later call; `compare_models(include=[fitted models])` re-scores cheaply.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `exp` | the experiment opened with `test_data=test_df`; used by every Part 3-4 call |
| `DROP` | columns removed before `setup` (`household_id`, `spend_next`, `note`) |
| `top3`, `best` | the three best models by CV AUC, and the first of them |
| `cmp`, `cmp_pr` | the comparison tables (`pull()`), before and after `add_metric` |
| `best_pr` | the best of `top3` when sorted by PR-AUC |

**➡️ Next Up:** Task 7 puts the four metric families side by side on the held-out set and shows where accuracy lies.


# Task 7 · The Metric Families

## 7.1 Four questions, four families

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

A classifier outputs a **score** (here, a probability of overspending). Every metric asks a different question of that score:

| Family | Question | Needs | Metrics |
|---|---|---|---|
| **Threshold** | after cutting at 0.5, how good are the labels? | labels | accuracy, balanced accuracy, precision, recall, F1, MCC |
| **Ranking** | do overspenders score above non-overspenders? | the order of scores | ROC-AUC, PR-AUC |
| **Probabilistic** | how far are the probabilities from the truth, per household? | the score values | log loss, Brier |
| **Calibration** | when the model says 30%, does it happen 30% of the time? | scores grouped in bins | reliability diagram, ECE (Task 11) |

Threshold metrics change when the cut-off moves (Task 11); ranking metrics do not; probabilistic metrics punish confident mistakes.

</div>

## 7.2 The math, gently

With $P$ positives and $N$ negatives, $TP$ correctly flagged, $FP$ wrongly flagged, $FN$ missed:

$$ \text{Precision} = \frac{TP}{TP + FP}, \qquad \text{Recall} = \frac{TP}{TP + FN}, \qquad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

$$ \text{ROC-AUC} = \Pr\big(s(\text{random positive}) > s(\text{random negative})\big), \qquad \text{PR-AUC} \approx \sum_k (R_k - R_{k-1})\, P_k $$

$$ \text{LogLoss} = -\frac{1}{n}\sum_i \big[y_i \log p_i + (1-y_i)\log(1-p_i)\big], \qquad \text{Brier} = \frac{1}{n}\sum_i (p_i - y_i)^2 $$

The baselines matter: a model that flags nobody has accuracy $N/(P+N)$; a random ranking has ROC-AUC $0.5$ and PR-AUC equal to the positive rate $P/(P+N)$.

## 7.3 The top three on the held-out set

`predict_model(model)` with no `data=` scores the experiment's held-out partition, which is `test_df` thanks to `test_data=`. `raw_score=True` adds one probability column per class; `prediction_score_1` is $P(\text{overspends})$.


In [ ]:
def holdout_probs(model, **kw):
    # P(class 1) and the true labels on the experiment's held-out set (= test_df)
    p = predict_model(model, raw_score=True, verbose=False, **kw)
    return p[LABEL_CLS].values, p["prediction_label"].values, p["prediction_score_1"].values

cards, curves = [], {}
for m in top3:
    name = type(m).__name__.replace("Classifier", "")
    y_true, y_lab, y_prob = holdout_probs(m)
    cards.append(report_card(y_true, y_lab, y_prob, name=name))
    curves[name] = (y_true, y_prob)

# two baselines every table should carry
prior = y_true.mean()
cards.append(report_card(y_true, np.zeros_like(y_true), np.full(len(y_true), prior), name="always 'stays within'"))
rng = np.random.default_rng(RANDOM_STATE)
cards.append(report_card(y_true, (rng.random(len(y_true)) < prior).astype(int), rng.random(len(y_true)), name="random guess"))

scores_test = pd.concat(cards)
print(f"positive rate on the held-out set: {prior:.3f}")
pretty(scores_test, subset=[c for c in scores_test.columns if c not in ("log_loss", "brier")])


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Read the rows from the bottom up. The **"always stays within"** row has accuracy equal to one minus the positive rate, recall 0, and PR-AUC equal to the positive rate: that is the floor. The real models beat it clearly on **recall, F1, MCC and PR-AUC**, and only modestly on **accuracy**, because accuracy mostly rewards the easy 83% of households. **ROC-AUC** is high for every real model because most households are easy to rank; **PR-AUC** is lower and spreads the models out, because it only looks at how the 63 overspenders are ranked. The two probabilistic columns, printed below, agree on the order but not on the scale: log loss grows without bound for a confident wrong answer, Brier is capped at 1.

**The common misread:** "0.9 accuracy means the model is right 90% of the time" when the base rate is 0.83; the honest comparison is `balanced_acc`, `mcc`, or the lift over the baseline row.

</div>


In [ ]:
print(scores_test[["log_loss", "brier", "roc_auc", "pr_auc"]].round(4).to_string())

# does PyCaret's own hold-out row agree with report_card? (positive-class precision/recall/F1)
_ = predict_model(best, verbose=False)
pc_row = pull().iloc[0]
rc_row = scores_test.iloc[0]
check = pd.DataFrame({"pycaret pull()": [pc_row["Prec."], pc_row["Recall"], pc_row["F1"], pc_row["AUC"]],
                      "report_card": [rc_row["precision"], rc_row["recall"], rc_row["f1"], rc_row["roc_auc"]]},
                     index=["precision", "recall", "f1", "roc_auc"])
print("\n" + check.round(4).to_string())
print("\nagree to 3 decimals:", bool(np.allclose(check["pycaret pull()"], check["report_card"], atol=1e-3)))


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: verify the library's metric once, then trust it

`pull()` after `predict_model` gives PyCaret's own held-out row. It matches `report_card` to three decimals, which confirms that for a binary target the Recall, Prec. and F1 columns are positive-class metrics. Do this check once per library and version; it takes one cell and removes a whole class of "why do our numbers differ" meetings.

</div>

## 7.4 ROC and PR curves, three models on one figure


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for (name, (yt, yp)), color in zip(curves.items(), ["#2196F3", "#764ba2", "#28a745"]):
    fpr, tpr, _ = roc_curve(yt, yp)
    prec, rec, _ = precision_recall_curve(yt, yp)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={roc_auc_score(yt, yp):.3f})")
    axes[1].plot(rec, prec, color=color, lw=2, label=f"{name} (AP={average_precision_score(yt, yp):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="random (AUC=0.5)")
axes[1].axhline(prior, color="k", ls="--", lw=1, label=f"random (AP={prior:.3f})")
axes[0].set(xlabel="false positive rate", ylabel="true positive rate (recall)", title="ROC curves (held-out set)")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-Recall curves (held-out set)", ylim=(0, 1.02))
for ax in axes: ax.legend(loc="lower left" if ax is axes[1] else "lower right")
plt.tight_layout(); plt.show()
print("ROC: closer to the top-left is better.  PR: closer to the top-right is better; the dashed line is the positive rate.")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left, the ROC curves hug the top-left corner and are hard to tell apart: with 312 negatives, the false-positive rate moves slowly, so every model looks good. Right, the PR curves separate: precision falls fast as recall rises past about 0.6, because the remaining overspenders look like ordinary households. The PR plot is where the decision (Task 11) actually lives, since it shows the trade-off at every threshold.

**The common misread:** picking the model with the best ROC-AUC when the job is to find a rare class. On imbalanced data, rank by PR-AUC (or by the precision at the recall the business needs).

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `prediction_score` without `raw_score=True` is not $P(\text{overspends})$

By default `predict_model` returns `prediction_score`, the probability of the **predicted** class: 0.9 can mean "90% sure it overspends" or "90% sure it does not". Every ranking, probabilistic or calibration metric needs the positive-class column, so this notebook always passes `raw_score=True` and reads `prediction_score_1`.

</div>

## 7.5 Task 7 Summary

**✅ Key Takeaways**
- Threshold metrics need labels, ranking metrics need order, probabilistic metrics need the values; each family answers a different question.
- On a 17%-positive target, accuracy's floor is 0.83 and PR-AUC's floor is 0.17; report lift over the baseline rows.
- ROC-AUC barely separates the models here; PR-AUC and F1 do.
- PyCaret's binary Recall/Prec./F1 are positive-class metrics, verified against `report_card`.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `holdout_probs(model)` | `(y_true, y_label, P(class 1))` on the held-out set via `predict_model(raw_score=True)` |
| `scores_test` | `report_card` rows for the top 3 plus two baselines |
| `curves` | `{name: (y_true, y_prob)}` for drawing ROC/PR curves |
| `prior` | positive rate on the held-out set |

**➡️ Next Up:** Task 8 opens the plot gallery and says what each picture can and cannot tell you.


# Task 8 · The Plot Gallery

## 8.1 `plot_model`, and how to render it anywhere

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

`plot_model(model, plot=<name>)` draws one diagnostic picture with matplotlib (most plots go through the `yellowbrick` library). In a notebook the figure appears inline. In a script, a scheduled job, or a CI run there is no display, so this notebook uses the route that works everywhere: `save=True` writes a PNG into the working directory and returns its file name; the helper below reads that file back and shows it with `plt.imshow`. `evaluate_model(model)` shows the same gallery as an interactive widget with one button per plot; it needs a live notebook kernel, so it is mentioned here and not run.

</div>

| `plot=` | Picture | Needs |
|---|---|---|
| `auc` | ROC curve per class | probabilities |
| `pr` | precision-recall curve | probabilities |
| `confusion_matrix` | the four counts at the 0.5 threshold | labels |
| `class_report` | precision / recall / F1 / support per class as a heatmap | labels |
| `threshold` | precision, recall, F1 and queue rate as the threshold moves | probabilities, repeated fits |
| `calibration` | reliability diagram | probabilities |
| `learning` | score vs training-set size | repeated fits |
| `feature` | importances or coefficients | a model that exposes them |
| `boundary` | decision regions in 2-D | two features (PCA otherwise) |
| `vc`, `manifold`, `rfe`, `dimension`, `lift`, `gain`, `ks`, `error` | more in the docs | varies |


In [ ]:
import os, tempfile, pathlib, shutil

PLOT_DIR = pathlib.Path(tempfile.mkdtemp(prefix="pycaret_plots_"))

def show_plot(model, plot, ax=None, exp=None, title=None, **kw):
    # Render one plot_model() picture inline, in any runtime: save=True -> PNG -> imshow.
    fn = (exp.plot_model if exp is not None else plot_model)
    cwd = os.getcwd(); os.chdir(PLOT_DIR)
    try:
        fname = fn(model, plot=plot, save=True, verbose=False, **kw)
    finally:
        os.chdir(cwd)
    img = plt.imread(PLOT_DIR / fname)
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.imshow(img); ax.axis("off"); ax.set_title(title or f"plot_model(plot='{plot}')")
    if own:
        plt.tight_layout(); plt.show()
    return fname

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
with timer("auc + pr"):
    f1 = show_plot(best, "auc", ax=axes[0]); f2 = show_plot(best, "pr", ax=axes[1])
plt.tight_layout(); plt.show()
print("saved:", f1, "|", f2, " in", PLOT_DIR)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `auc` and `pr`

Both are drawn on the experiment's held-out partition. **Can tell you:** how well the model ranks households, independent of any threshold; the ROC plot also shows the micro and macro averages, which coincide for a binary target. **Cannot tell you:** what happens at the threshold you will deploy, or whether the probabilities mean anything (a model can rank perfectly and still say "20%" for households that overspend 60% of the time).

**The common misread:** reading the `auc` plot's "class 0" and "class 1" curves as two different models. For a binary target they are mirror images with the same area.

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
show_plot(best, "confusion_matrix", ax=axes[0]); show_plot(best, "class_report", ax=axes[1])
plt.tight_layout(); plt.show()
from sklearn.metrics import confusion_matrix
y_true, y_lab, _ = holdout_probs(best)
tn, fp, fn, tp = confusion_matrix(y_true, y_lab, labels=[0, 1]).ravel()
print(f"confusion matrix at 0.5:  TN={tn}  FP={fp}  FN={fn}  TP={tp}   ->  missed overspenders: {fn} of {fn + tp}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `confusion_matrix` and `class_report`

**Can tell you:** the four counts at the default threshold and, in the report, per-class precision, recall, F1 and support, which makes the minority class's weakness visible (its recall is the lowest cell). **Cannot tell you:** anything about other thresholds; both pictures change completely when the cut-off moves (Task 11), and neither shows *which* households were missed (Part 10 does that).

**The common misread:** a diagonal-heavy confusion matrix read as "good": with 312 negatives, the top-left cell is large for any model, including one that flags nobody. Look at the bottom row (the actual overspenders) instead.

</div>


In [ ]:
with timer("threshold plot (n_trials=10 refits instead of the default 50)"):
    fname = show_plot(best, "threshold", plot_kwargs={"n_trials": 10})
print("yellowbrick's DiscriminationThreshold refits the model n_trials times on random splits; saved", fname)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `threshold`

Precision rises and recall falls as the cut-off moves right; F1 peaks somewhere between; the **queue rate** line is the share of households that would be flagged, which is the workload of whoever acts on the prediction. The bands come from refitting on `n_trials` random splits (the default 50 costs five times more than the 10 used here). **Can tell you:** roughly where F1 peaks and how much recall a stricter threshold costs. **Cannot tell you:** the business-optimal cut-off, because it does not know that a missed overspend costs more than a false alarm; Task 11 adds that.

**The common misread:** taking the dashed "best F1 threshold" as a recommendation. It is the F1 optimum on random splits of the training partition, not on your cost structure.

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
with timer("calibration + learning"):
    show_plot(best, "calibration", ax=axes[0]); show_plot(best, "learning", ax=axes[1])
plt.tight_layout(); plt.show()
print("learning curve = 10 training sizes x 3 folds = 30 fits; calibration = one pass over the held-out set")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `calibration` and `learning`

**Calibration (left):** households are binned by predicted probability; each point is the bin's actual overspend rate. Points on the diagonal mean "30% means 30%". **Can tell you:** whether the probabilities can be used as probabilities (for expected-cost decisions, Task 11). **Cannot tell you:** ranking quality; a badly calibrated model can still have a perfect AUC.

**Learning curve (right):** training and cross-validation score against training-set size. A wide gap that closes slowly means variance (more data would help); two low, converged lines mean bias (a richer model or better features would help). **Can tell you:** whether collecting more households is worth it. **Cannot tell you:** which features to add.

**The common misread:** in the calibration plot, a point far from the diagonal in the rightmost bin. Very few households get a probability above 0.9, so that point is an average of a handful of cases; the histogram of scores matters as much as the curve.

</div>


In [ ]:
# the feature plot needs coef_ or feature_importances_; Naive Bayes and KNN have neither
imp_model = next((m for m in top3 if hasattr(m, "coef_") or hasattr(m, "feature_importances_")), None)
if imp_model is None:
    imp_model = create_model("rf", fold=3, verbose=False)
show_plot(imp_model, "feature", title=f"plot_model(plot='feature') on {type(imp_model).__name__}")
cols = get_config("X_train_transformed").columns
if hasattr(imp_model, "coef_"):
    coefs = pd.Series(imp_model.coef_[0], index=cols).sort_values(key=np.abs, ascending=False)
    print(f"top 5 |coefficient| on z-scored features ({type(imp_model).__name__}):")
else:
    coefs = pd.Series(imp_model.feature_importances_, index=cols).sort_values(ascending=False)
    print(f"top 5 split-based importances ({type(imp_model).__name__}):")
print(coefs.head(5).round(3).to_string())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `feature`

For a linear model this plot shows the **coefficients**; for a tree ensemble it shows `feature_importances_` (split-based). **Can tell you:** which columns the model leans on and, for a linear model on standardized features, in which direction (the sign). **Cannot tell you:** causation, or importance *on the test set* (Part 6 computes permutation importance for that). The picture is only comparable across features because `normalize=True` put every column on the same scale; on raw columns a coefficient on income (thousands) and one on `overspend_rate_prev` (0 to 1) cannot be compared.

**The common misread:** a one-hot column such as `city_tier_town` with a large coefficient read as "town matters most". It is the effect of being in that tier *relative to the dropped tier*, not the importance of the whole variable.

</div>

## 8.2 The decision boundary needs its own, two-feature experiment

`plot_model(plot="boundary")` projects the features to two PCA components, which makes the axes unreadable. A cleaner picture comes from a **second experiment on two named features**. The functional API holds one current experiment at a time, so the side experiment uses the OOP form, `ClassificationExperiment()`, which leaves `exp` untouched for the rest of Parts 3-4.


In [ ]:
PAIR = ["total_spend", "income"]
side = ClassificationExperiment()
with timer("side experiment on two features"):
    side.setup(train_df[PAIR + [LABEL_CLS]], target=LABEL_CLS, test_data=test_df[PAIR + [LABEL_CLS]],
               normalize=True, index=False, fold=3, session_id=RANDOM_STATE, n_jobs=1, verbose=False, html=False)
    side_rf = side.create_model("rf", verbose=False)
    side_cv = side.pull().loc["Mean"]
show_plot(side_rf, "boundary", exp=side, title=f"boundary: random forest on {PAIR[0]} x {PAIR[1]}")
print(f"two-feature random forest, CV AUC {side_cv['AUC']:.3f} vs {cmp.loc['rf', 'AUC']:.3f} with all features")
print("main experiment still intact:", get_config("X_train").shape[1], "features")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `boundary`

The axes are the two z-scored features; the shaded regions are the forest's predicted class; dots are the held-out households. The boundary is roughly a line of constant `total_spend / income` (the 0.65 rule that defines overspending), with the tree ensemble's characteristic staircase edges. **Can tell you:** the shape of the rule a model learned, and where the classes overlap (the mixed band along the boundary is where every model's errors live). **Cannot tell you:** how the full model behaves, since two features give a lower AUC than all of them.

**The common misread:** a jagged boundary read as overfitting. Staircases are how axis-aligned trees draw any diagonal; the test dots inside the wrong region are the evidence that matters.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: OOP experiments for anything "on the side"

`ClassificationExperiment()` objects are independent: their own pipeline, folds, metrics and model containers. Use one whenever you need a second `setup` (a feature subset, a different target, an ablation) without losing the main experiment. The functional API is the same class behind a module-level current experiment; `get_config("X_train").shape` above proves it was untouched.

</div>

## 8.3 Task 8 Summary

**✅ Key Takeaways**
- `plot_model(..., save=True)` plus `plt.imread` renders every plot in any runtime; `evaluate_model` is the interactive version for a live kernel.
- `auc`/`pr` are threshold-free, `confusion_matrix`/`class_report` are threshold-bound, `threshold` shows the trade-off, `calibration` checks the probabilities, `learning` diagnoses bias vs variance, `feature` shows what the model leans on.
- `plot_kwargs` reaches the yellowbrick visualizer (`n_trials` cut the threshold plot's cost fivefold).
- A second experiment lives in a `ClassificationExperiment()` object and does not disturb the functional one.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `show_plot(model, plot, ...)` | renders a `plot_model` picture inline via a saved PNG; `exp=` selects an OOP experiment |
| `PLOT_DIR` | temp directory holding the PNGs |
| `side`, `side_rf` | the two-feature OOP experiment and its random forest |
| `coefs` | `best`'s coefficients on z-scored features, sorted by magnitude |

**➡️ Next Up:** Part 4 tunes, bags, boosts, blends, stacks and calibrates, then chooses the threshold by cost.


# Part 4 · Tune, Ensemble, Blend, Stack, Calibrate, Threshold ⭐⭐⭐

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

Part 3 chose a model. Part 4 makes it better in three different senses, and keeps the three apart:

- **Better hyperparameters** (`tune_model`, Task 9) change what one model can fit.
- **Better combinations** (`ensemble_model`, `blend_models`, `stack_models`, Task 10) average away the errors of several models.
- **Better decisions** (`calibrate_model`, `optimize_threshold`, a cost matrix, Task 11) leave the ranking alone and change what the score *means* and where it is cut.

The working model for the Part is LightGBM: it fits in under a second, PyCaret's tuning grid for it is rich, and its raw probabilities are usually over-confident, so calibration has something to fix. The experiment (`exp`, with `test_data=test_df`) continues from Part 3.

</div>


# Task 9 · `tune_model`

## 9.1 Hyperparameters, and the search over them

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

**Parameters** are what training sets (the trees' split values). **Hyperparameters** are what you set before training (how many trees, how deep, how fast to learn). `tune_model` draws `n_iter` random combinations from PyCaret's built-in grid for that model, cross-validates each one with `fold` folds, and keeps the best by `optimize`. That is `n_iter × fold` fits, so `n_iter=10, fold=3` is 30 fits.

`choose_better=True` adds a safety net: if the tuned model scores worse than the original on the same folds, the **original** is returned. The next cells show why that net is needed.

</div>


In [ ]:
with timer("create_model('lightgbm')"):
    lgbm = create_model("lightgbm", fold=3, verbose=False)
lgbm_cv = pull()
print("baseline LightGBM, per fold:"); print(lgbm_cv.round(4).to_string())

with timer("tune_model: n_iter=10 random draws x 3 folds"):
    lgbm_t, tuner = tune_model(lgbm, n_iter=10, fold=3, optimize="AUC", choose_better=True,
                               return_tuner=True, verbose=False)
tuned_cv = pull()
print("\ntuned (or kept) LightGBM, per fold:"); print(tuned_cv.round(4).to_string())

changed = {k: (lgbm.get_params()[k], lgbm_t.get_params()[k])
           for k in lgbm.get_params() if lgbm.get_params()[k] != lgbm_t.get_params()[k]}
kept_original = lgbm_t is lgbm or not changed
print("\nbest params found by the search:", {k: v for k, v in tuner.best_params_.items()})
print("returned model is the ORIGINAL (choose_better kept it)" if kept_original else f"returned model differs in: {changed}")


## 9.2 `search_library`, `search_algorithm`, `custom_grid`

| `search_library` | `search_algorithm` options | Needs | Character |
|---|---|---|---|
| `"scikit-learn"` (default) | `"random"` (default), `"grid"` | nothing extra | random draws from the built-in distributions; grid enumerates `custom_grid` |
| `"optuna"` | `"random"`, `"tpe"` (default) | `pip install optuna` | Tree-structured Parzen Estimator: each draw learns from the previous ones |
| `"scikit-optimize"` | `"bayesian"` | `pip install scikit-optimize` | Gaussian-process Bayesian optimisation |
| `"tune-sklearn"` | `"random"`, `"grid"`, `"bayesian"`, `"hyperopt"`, `"optuna"`, `"bohb"` | `pip install tune-sklearn ray[tune]` | distributed search with early stopping |

`custom_grid={"param": [values]}` replaces the built-in space; with `search_algorithm="grid"` every combination is tried and `n_iter` is ignored. The next cell checks which libraries this runtime has and runs a tiny custom grid.


In [ ]:
avail = {}
for lib, mod in [("scikit-learn", "sklearn"), ("optuna", "optuna"), ("scikit-optimize", "skopt"), ("tune-sklearn", "tune_sklearn")]:
    try:
        __import__(mod); avail[lib] = "installed"
    except ImportError:
        avail[lib] = "not installed"
print("search libraries:", avail)

# !pip install optuna -q     # then: tune_model(lgbm, search_library="optuna", search_algorithm="tpe", n_iter=20)
grid = {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1], "num_leaves": [7, 31]}   # 8 combinations
with timer("custom_grid, exhaustive (8 combos x 3 folds)"):
    lgbm_g = tune_model(lgbm, custom_grid=grid, search_algorithm="grid", fold=3, optimize="AUC",
                        choose_better=False, verbose=False)
grid_cv = pull()
print(f"\ngrid winner: {{k: lgbm_g.get_params()[k] for k in grid}}")
print(f"CV AUC  baseline {lgbm_cv.loc['Mean', 'AUC']:.4f}  |  random search {tuned_cv.loc['Mean', 'AUC']:.4f}  |  custom grid {grid_cv.loc['Mean', 'AUC']:.4f}")


In [ ]:
folds = np.arange(3)
before = lgbm_cv.iloc[:3]["AUC"].values; after = tuned_cv.iloc[:3]["AUC"].values; grid_f = grid_cv.iloc[:3]["AUC"].values
fig, ax = plt.subplots(figsize=(10, 4.5))
w = 0.26
ax.bar(folds - w, before, w, color="#2196F3", label=f"baseline (mean {before.mean():.3f})")
ax.bar(folds, after, w, color="#764ba2", label=f"random search, choose_better (mean {after.mean():.3f})")
ax.bar(folds + w, grid_f, w, color="#28a745", label=f"custom grid (mean {grid_f.mean():.3f})")
for i in folds:
    ax.plot([i - w, i, i + w], [before[i], after[i], grid_f[i]], color="gray", lw=1, marker="o", ms=3)
ax.set_xticks(folds); ax.set_xticklabels([f"fold {i}" for i in folds]); ax.set_ylabel("AUC on the held-out fold")
lo = min(before.min(), after.min(), grid_f.min()); ax.set_ylim(lo - 0.03, 1.0)
ax.set_title("Per-fold AUC before and after tuning (paired by fold)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"fold-to-fold std of the baseline: {before.std(ddof=1):.4f}   |   mean gain from tuning: {after.mean() - before.mean():+.4f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Three bars per fold, joined by a grey line so the eye pairs them. The folds differ from each other by more than the tuned model differs from the baseline: the noise between folds is larger than the signal from tuning. Whether a bar rises or falls after tuning depends on the fold as much as on the hyperparameters.

**The common misread:** "tuning improved AUC by 0.01" from the Mean row alone. Compare the gain with the fold std printed under the plot; a gain smaller than the std is not yet evidence.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ A tuned model can be worse, and `choose_better` is why you get to find out

Ten random draws from a large space often miss the defaults, which the LightGBM authors chose well. Without `choose_better=True`, `tune_model` returns the best of the ten draws even when all ten lose to the original. With it, the printout above says plainly whether the original was kept. Do not switch the net off to "make the tuning show"; raise `n_iter`, or move to `optuna`'s TPE sampler, which spends later draws near the good earlier ones.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: tune the metric you decide by, on the folds you compare by

`optimize="AUC"` here because Part 3 ranked by AUC. Registered custom metrics (`"PR-AUC"` from Task 6) are valid too. Keep `fold` the same as in `compare_models`, so the tuned score is comparable to the table it came from, and set `return_tuner=True` when you want to inspect `tuner.best_params_` or `tuner.cv_results_` afterwards.

</div>

## 9.3 Task 9 Summary

**✅ Key Takeaways**
- `tune_model` = `n_iter` random draws x `fold` fits, scored by `optimize`; `choose_better=True` returns the original if the search loses.
- The fold-to-fold std sets the bar: a gain below it is noise.
- `custom_grid` + `search_algorithm="grid"` enumerates your own space; `optuna` / `scikit-optimize` / `tune-sklearn` are optional smarter searchers.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `lgbm`, `lgbm_cv` | baseline LightGBM and its per-fold table |
| `lgbm_t`, `tuner`, `tuned_cv` | model returned by the random search (original if `choose_better` kept it), the fitted searcher, its folds |
| `lgbm_g`, `grid_cv` | winner of the 8-combination custom grid and its folds |
| `avail` | which search libraries this runtime has |

**➡️ Next Up:** Task 10 combines models instead of tuning one.


# Task 10 · Bagging, Boosting, Blending, Stacking

## 10.1 Four ways to combine

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

| Method | PyCaret call | Members | How they are combined | Fixes |
|---|---|---|---|---|
| **Bagging** | `ensemble_model(m, method="Bagging")` | `n_estimators` copies of one model on bootstrap samples | average the probabilities | variance (a jumpy model such as a deep tree) |
| **Boosting** | `ensemble_model(m, method="Boosting")` | copies of one model, each fit on the previous one's mistakes (AdaBoost) | weighted vote | bias (a weak model that underfits) |
| **Blending** | `blend_models([m1, m2, ...], method="soft")` | different models | average probabilities (`soft`) or majority vote (`hard`) | errors that different families make in different places |
| **Stacking** | `stack_models([m1, m2, ...], meta_model=...)` | different models plus a **meta-model** | the meta-model learns weights from out-of-fold predictions | the same, with learned instead of equal weights |

</div>

## 10.2 Stacking, as a diagram

```
                 ┌──────────── out-of-fold predictions (meta_model_fold = 5) ────────────┐
   features ──►  │  lr   ──► P_lr(overspend)   ─┐                                          │
   features ──►  │  rf   ──► P_rf(overspend)   ─┼──►  meta-model (logistic regression) ──► final P
   features ──►  │  lgbm ──► P_lgbm(overspend) ─┘        learns one weight per member       │
                 └───────────────────────────────────────────────────────────────────────────┘
   restack=True would also feed the original features into the meta-model (off by default)
```

The out-of-fold step matters: if the meta-model saw predictions the members made on their own training rows, it would learn to trust an over-fitted member. `meta_model_fold` controls that inner cross-validation.


In [ ]:
def cv_mean(label):
    r = pull().loc["Mean"]
    return {"approach": label, "CV AUC": r["AUC"], "CV F1": r["F1"], "CV PR-AUC": r.get("PR-AUC", np.nan), "CV MCC": r["MCC"]}

rows = []
with timer("single decision tree, bagged, boosted"):
    dt = create_model("dt", fold=3, verbose=False);                         rows.append(cv_mean("decision tree"))
    dt_bag = ensemble_model(dt, method="Bagging", fold=3, verbose=False);   rows.append(cv_mean("bagged tree (x10)"))
    dt_boost = ensemble_model(dt, method="Boosting", fold=3, verbose=False); rows.append(cv_mean("boosted tree (AdaBoost x10)"))
print(type(dt_bag).__name__, "|", type(dt_boost).__name__)
print(pd.DataFrame(rows).set_index("approach").round(4).to_string())


In [ ]:
with timer("members + blend + stack"):
    lr = create_model("lr", fold=3, verbose=False);   rows.append(cv_mean("logistic regression"))
    rf = create_model("rf", fold=3, verbose=False);   rows.append(cv_mean("random forest"))
    _ = create_model(lgbm, fold=3, verbose=False);    rows.append(cv_mean("lightgbm"))
    blended = blend_models([lr, rf, lgbm], method="soft", fold=3, verbose=False);         rows.append(cv_mean("blend soft [lr, rf, lgbm]"))
    stacked = stack_models([lr, rf, lgbm], meta_model=lr, fold=3, verbose=False);        rows.append(cv_mean("stack [lr, rf, lgbm] -> lr"))

ens_table = pd.DataFrame(rows).set_index("approach")
print(type(blended).__name__, "|", type(stacked).__name__)
pretty(ens_table)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading the table

The first three rows are the textbook picture: a single decision tree ranks poorly, **bagging** it lifts AUC by a wide margin (ten trees average out the jumpiness of one), and **boosting** it lifts it too but by a different route. The next three rows are the members. The **blend** lands at or near the best member on every column: averaging three good models rarely hurts and often helps a little. The **stack** is close to the blend; its logistic meta-model has only three inputs and learns weights that are not far from equal.

**The common misread:** expecting stacking to beat blending by a clear margin. With three well-correlated members and 1,125 rows, the learned weights have almost nothing to learn; stacking earns its extra cost when members disagree a lot or when there are many of them.

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ["#dc3545", "#ffc107", "#ffc107", "#2196F3", "#2196F3", "#2196F3", "#764ba2", "#28a745"]
ax.barh(ens_table.index, ens_table["CV AUC"], color=colors[:len(ens_table)])
for i, v in enumerate(ens_table["CV AUC"]):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(ens_table["CV AUC"].min() - 0.05, 1.0); ax.set_xlabel("cross-validated AUC (mean of 3 folds)")
ax.set_title("Single models vs bagged / boosted / blended / stacked"); ax.invert_yaxis()
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#dc3545", label="single tree"), Patch(color="#ffc107", label="bag / boost of it"),
                   Patch(color="#2196F3", label="members"), Patch(color="#764ba2", label="blend"), Patch(color="#28a745", label="stack")],
          loc="lower right")
plt.tight_layout(); plt.show()
print("largest single gain in the table:", (ens_table["CV AUC"].diff().abs().idxmax()), "step")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

One bar per approach, in the order they were built. The big jump is from the red bar (single tree) to the yellow ones: combining copies of a weak model is where ensembling pays most. The blue, purple and green bars sit within a narrow band: combining strong models moves the needle by hundredths at most.

**The common misread:** concluding that ensembles "do not work" from the top of the chart. They worked enormously at the bottom; the ceiling near the top is the data, not the method.

</div>

## 10.3 The best two on the held-out set


In [ ]:
candidates = {"blend soft [lr, rf, lgbm]": blended, "stack [lr, rf, lgbm] -> lr": stacked,
              "random forest": rf, "logistic regression": lr, "lightgbm": lgbm}
top2_names = ens_table.loc[list(candidates)].sort_values("CV AUC", ascending=False).index[:2].tolist()
cards = []
for name in top2_names:
    y_true, y_lab, y_prob = holdout_probs(candidates[name])
    cards.append(report_card(y_true, y_lab, y_prob, name=name))
ens_test = pd.concat(cards)
print("best two by CV AUC:", top2_names)
pretty(ens_test, subset=[c for c in ens_test.columns if c not in ("log_loss", "brier")])


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Costs that the table hides

- `blend_models(method="soft")` needs every member to expose `predict_proba`; an SVM without probability estimates breaks it (`method="hard"` votes on labels instead).
- A stack is `1 + len(members)` models at prediction time, and `stack_models` fits each member `meta_model_fold` extra times to get out-of-fold predictions. Its latency and its training cost both scale with the member list.
- `ensemble_model` and the blend/stack calls take `choose_better=True` as well: with it, they return the best *member* if the combination loses.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: blend diverse members, not similar ones

A blend of three tree ensembles shares their blind spots. `[lr, rf, lgbm]` mixes a linear model with a bagged and a boosted forest, which is why the blend matches its best member in AUC while lowering log loss: the members' probability errors partly cancel. Check member diversity with the correlation of their held-out probabilities before paying for a stack.

</div>

## 10.4 Task 10 Summary

**✅ Key Takeaways**
- Bagging fixes variance, boosting fixes bias, both apply to one base model; blending and stacking combine different models.
- The gain is huge for a weak base model and small for strong members; the ceiling is the data.
- Stacking needs out-of-fold member predictions (`meta_model_fold`); `restack=True` adds the raw features to the meta-model.
- The held-out set is opened only for the final two candidates, once.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `dt`, `dt_bag`, `dt_boost` | single tree, its bagged and boosted ensembles |
| `lr`, `rf` | the other blend/stack members (`lgbm` from Task 9) |
| `blended`, `stacked` | `VotingClassifier` (soft) and `StackingClassifier` (meta: logistic regression) |
| `ens_table`, `ens_test` | CV comparison of all approaches; held-out `report_card` for the best two |
| `cv_mean(label)` | helper: the `Mean` row of the last `pull()` as a dict |

**➡️ Next Up:** Task 11 leaves the ranking alone and fixes what the probabilities mean and where to cut them.


# Task 11 · Calibrate, then Choose the Threshold by Cost

## 11.1 Calibration

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

A model is **calibrated** when, among all households it scores at 0.3, about 30% overspend. Ranking metrics do not care (any monotone reshaping of the scores leaves AUC unchanged), but a cost decision does: "expected cost of ignoring this household" is only meaningful if the probability is real.

`calibrate_model(model, method=...)` wraps the fitted model in scikit-learn's `CalibratedClassifierCV`: it fits the model on `calibrate_fold` inner folds, learns a mapping from raw score to calibrated probability on the held-out parts, and averages. Two mappings:

- `"sigmoid"` (Platt scaling): a logistic curve with 2 parameters; robust on small data.
- `"isotonic"`: a free-form monotone step function; more flexible, needs more data.

</div>

## 11.2 The math, gently

The **reliability diagram** bins households by predicted probability and plots, per bin, the mean prediction against the observed rate. The **expected calibration error** summarises the gap:

$$ \text{ECE} = \sum_{b=1}^{B} \frac{n_b}{n}\,\big|\,\bar{y}_b - \bar{p}_b\,\big| $$

The **Brier score** decomposes into calibration plus refinement, which is why calibrating usually lowers it while leaving AUC alone.


In [ ]:
from sklearn.calibration import calibration_curve

def ece(y_true, y_prob, n_bins=10):
    # expected calibration error with equal-width bins
    bins = np.clip((y_prob * n_bins).astype(int), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any():
            total += m.mean() * abs(y_true[m].mean() - y_prob[m].mean())
    return total

with timer("calibrate_model x2 (sigmoid, isotonic)"):
    lgbm_sig = calibrate_model(lgbm_t, method="sigmoid", fold=3, verbose=False)
    lgbm_iso = calibrate_model(lgbm_t, method="isotonic", fold=3, verbose=False)
print(type(lgbm_sig).__name__, "wrapping", type(lgbm_sig.estimator).__name__)

cal_models = {"raw LightGBM": lgbm_t, "sigmoid": lgbm_sig, "isotonic": lgbm_iso}
cal_probs, cal_rows = {}, []
for name, m in cal_models.items():
    y_true, y_lab, y_prob = holdout_probs(m)
    cal_probs[name] = y_prob
    cal_rows.append({"model": name, "roc_auc": roc_auc_score(y_true, y_prob), "log_loss": log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
                     "brier": brier_score_loss(y_true, y_prob), "ECE": ece(y_true, y_prob)})
cal_table = pd.DataFrame(cal_rows).set_index("model")
pretty(cal_table, highlight="min", subset=["log_loss", "brier", "ECE"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for (name, p), color in zip(cal_probs.items(), ["#dc3545", "#2196F3", "#28a745"]):
    frac_pos, mean_pred = calibration_curve(y_true, p, n_bins=8, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", color=color, lw=2, label=f"{name} (ECE={cal_table.loc[name, 'ECE']:.3f})")
    axes[1].hist(p, bins=25, range=(0, 1), histtype="step", lw=2, color=color, label=name)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
axes[0].set(xlabel="mean predicted probability (bin)", ylabel="observed overspend rate (bin)", title="Reliability diagram, held-out set (8 quantile bins)")
axes[1].set(xlabel="predicted P(overspends)", ylabel="households", title="Distribution of scores", yscale="log")
axes[0].legend(loc="upper left"); axes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()
print("AUC is unchanged by calibration:", cal_table["roc_auc"].round(4).to_dict())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: the raw LightGBM curve bends away from the diagonal in the middle bins (it is over-confident where the classes overlap); both calibrated curves sit closer to it, and the table shows the lower log loss, Brier and ECE that go with that, at an unchanged AUC. Right: the score histograms on a log axis. Calibration reshapes the distribution (sigmoid smoothly, isotonic in steps) without reordering the households.

**The common misread:** treating the isotonic curve's tight fit as "better". With 1,125 training rows and 3 calibration folds it is fitting a step function to a few hundred points per fold; on a new month it can be worse than sigmoid. Prefer sigmoid below a few thousand rows unless the reliability diagram is clearly S-shaped.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Calibration does not change the ranking, so it does not change AUC

If a calibrated model shows a different AUC from its raw version, something else changed: usually the model was refit on different folds (`CalibratedClassifierCV` refits `calibrate_fold` copies and averages them, which is a mild ensemble). The printed AUCs above differ in the third decimal for exactly that reason.

</div>

## 11.3 The threshold: F1 first, cost second

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

0.5 is not a decision, it is a default. Every label metric in the comparison table was computed at that default. The right cut-off depends on what a mistake costs: suppose a **missed overspend costs 5** (a household goes into debt) and a **false alarm costs 1** (an unnecessary nudge). The expected cost per household at threshold $t$ is

$$ \text{cost}(t) = \frac{5 \cdot FN(t) + 1 \cdot FP(t)}{n} $$

and the threshold that minimises it is far below 0.5, because a miss is five times worse than an alarm.

The rule: **sweep thresholds on validation predictions, choose one, then report on the held-out set once.** The validation predictions here are *out-of-fold*: each training household is scored by a model that did not see it.

</div>


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict, StratifiedKFold

X_tr, y_tr = get_config("X_train_transformed"), get_config("y_train_transformed").values
work_model = lgbm_sig                      # the calibrated working model
with timer("out-of-fold probabilities on the training partition"):
    oof = cross_val_predict(clone(work_model), X_tr, y_tr, method="predict_proba",
                            cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))[:, 1]

COST_FN, COST_FP = 5.0, 1.0
grid_t = np.round(np.linspace(0.02, 0.98, 97), 2)
sweep = []
for t in grid_t:
    pred = (oof >= t).astype(int)
    fn_ = ((pred == 0) & (y_tr == 1)).sum(); fp_ = ((pred == 1) & (y_tr == 0)).sum()
    sweep.append({"threshold": t, "F1": f1_score(y_tr, pred, zero_division=0), "recall": recall_score(y_tr, pred, zero_division=0),
                  "precision": precision_score(y_tr, pred, zero_division=0), "flag rate": pred.mean(),
                  "expected cost": (COST_FN * fn_ + COST_FP * fp_) / len(y_tr)})
sweep = pd.DataFrame(sweep).set_index("threshold")
t_f1 = float(sweep["F1"].idxmax()); t_cost = float(sweep["expected cost"].idxmin())
print(f"F1-optimal threshold on OOF: {t_f1:.2f} (F1={sweep['F1'].max():.3f})")
print(f"cost-optimal threshold on OOF (miss=5, alarm=1): {t_cost:.2f} (cost/household={sweep['expected cost'].min():.3f}; at 0.5 it is {sweep.loc[0.50, 'expected cost'] if 0.50 in sweep.index else np.interp(0.5, sweep.index, sweep['expected cost']):.3f})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sweep[["precision", "recall", "F1", "flag rate"]].plot(ax=axes[0], lw=2, color=["#2196F3", "#28a745", "#764ba2", "gray"])
axes[0].axvline(0.5, color="k", ls=":", lw=1, label="default 0.5"); axes[0].axvline(t_f1, color="#764ba2", ls="--", lw=1, label=f"F1-optimal {t_f1:.2f}")
axes[0].set(xlabel="threshold", ylabel="score on out-of-fold predictions", title="Label metrics vs threshold (validation folds)"); axes[0].legend(loc="center right")
axes[1].plot(sweep.index, sweep["expected cost"], color="#dc3545", lw=2, label="expected cost / household")
axes[1].axvline(0.5, color="k", ls=":", lw=1, label="default 0.5"); axes[1].axvline(t_cost, color="#dc3545", ls="--", lw=1, label=f"cost-optimal {t_cost:.2f}")
axes[1].set(xlabel="threshold", ylabel=f"(5 x misses + 1 x false alarms) / n", title="Expected cost vs threshold (validation folds)"); axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()
print(f"moving from 0.5 to {t_cost:.2f} raises the flag rate from {np.interp(0.5, sweep.index, sweep['flag rate']):.1%} to {sweep.loc[t_cost, 'flag rate']:.1%} of households")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: as the threshold rises, recall (green) falls and precision (blue) rises; F1 (purple) peaks below 0.5; the grey flag-rate line is the workload. Right: the cost curve has its minimum well to the left of 0.5, because each miss weighs five alarms. The two dashed lines do not coincide: **the F1 optimum and the cost optimum are different decisions**, and only the second one knows the price of a mistake.

**The common misread:** picking the F1 threshold "because F1 balances precision and recall". F1 weighs them equally; the business rarely does.

</div>

## 11.4 PyCaret's `optimize_threshold`, bounded

`optimize_threshold(model, optimize=<metric>)` runs a full cross-validation of the model at every threshold that scipy's SHGO optimiser tries, then draws a plotly chart. Two things to know before calling it:

- Its cost is `(number of thresholds tried) x fold` fits; with the default optimiser settings that is dozens of thresholds. The cell below passes SHGO's `n=` and `options=` through `**shgo_kwargs` to keep it to a handful.
- The chart is plotly's `fig.show()`, which needs a browser or a notebook renderer. `return_data=True` returns the table behind the chart, so the notebook redraws it with matplotlib and works in any runtime.


In [ ]:
import plotly.graph_objects as go
from contextlib import contextmanager

@contextmanager
def quiet_plotly():
    # optimize_threshold() ends with fig.show(); in a non-interactive run that would try to open a browser
    original = go.Figure.show
    go.Figure.show = lambda self, *a, **k: None
    try:
        yield
    finally:
        go.Figure.show = original

with timer("optimize_threshold(optimize='F1'), bounded SHGO"), quiet_plotly():
    thr_data, lgbm_thr = optimize_threshold(lgbm_t, optimize="F1", return_data=True, verbose=False,
                                            n=8, options={"maxiter": 1, "f_tol": 1e-2},
                                            minimizer_kwargs={"options": {"maxiter": 3, "ftol": 1e-2}})
t_pycaret = float(lgbm_thr.probability_threshold)
print(type(lgbm_thr).__name__, "with probability_threshold =", round(t_pycaret, 4))
print(f"thresholds evaluated: {thr_data['probability_threshold'].nunique()}  (each one = a full 3-fold cross-validation)")

wide = thr_data.pivot_table(index="probability_threshold", columns="variable", values="value")
fig, ax = plt.subplots(figsize=(10, 4.5))
for col, color in zip(["Prec.", "Recall", "F1", "Accuracy"], ["#2196F3", "#28a745", "#764ba2", "gray"]):
    if col in wide: ax.plot(wide.index, wide[col], marker="o", lw=2, color=color, label=col)
ax.axvline(t_pycaret, color="#764ba2", ls="--", lw=1, label=f"optimize_threshold pick {t_pycaret:.3f}")
ax.axvline(t_f1, color="k", ls=":", lw=1, label=f"hand-made OOF F1 pick {t_f1:.2f}")
ax.set(xlabel="probability threshold", ylabel="CV mean score", title="optimize_threshold(): CV metrics at the thresholds SHGO tried"); ax.legend(loc="center right")
plt.tight_layout(); plt.show()
print(f"built-in pick {t_pycaret:.3f} vs hand-made OOF F1 pick {t_f1:.2f}: gap {abs(t_pycaret - t_f1):.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Each dot is one full cross-validation at one threshold; the optimiser concentrated its dots where F1 is high. Its pick and the hand-made out-of-fold pick land in the same low region, which is reassuring: two different routes to the same answer. The dots are sparse, though, so the built-in curve is only as fine as the number of CVs you paid for.

**The common misread:** treating the returned object as a new model. `CustomProbabilityThresholdClassifier` is the same LightGBM with a stored cut-off; `predict_model(lgbm_thr)` applies it, and `finalize_model` / `save_model` keep it (Part 9).

</div>

## 11.5 Report on the held-out set, once


In [ ]:
def cost_on_holdout(y_true, y_pred):
    fn_ = ((y_pred == 0) & (y_true == 1)).sum(); fp_ = ((y_pred == 1) & (y_true == 0)).sum()
    return (COST_FN * fn_ + COST_FP * fp_) / len(y_true), int(fn_), int(fp_)

final_rows, cost_rows = [], []
for label, t in [("default 0.5", 0.5), (f"F1-optimal {t_f1:.2f}", t_f1), (f"cost-optimal {t_cost:.2f}", t_cost)]:
    y_true, y_lab, y_prob = holdout_probs(work_model, probability_threshold=t)
    final_rows.append(report_card(y_true, y_lab, y_prob, name=label))
    c, fn_, fp_ = cost_on_holdout(y_true, y_lab)
    cost_rows.append({"threshold": label, "misses (FN)": fn_, "false alarms (FP)": fp_, "flagged": int(y_lab.sum()), "cost / household": c})
final_test = pd.concat(final_rows)
cost_test = pd.DataFrame(cost_rows).set_index("threshold")
print(cost_test.round(3).to_string())
pretty(final_test[["accuracy", "balanced_acc", "precision", "recall", "f1", "mcc", "roc_auc", "pr_auc"]])


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading the two tables

The cost table is the decision: at 0.5 the model misses a large share of the 63 overspenders; at the cost-optimal threshold it misses far fewer, flags more households, and the cost per household drops. In the metric table, `roc_auc` and `pr_auc` are identical across rows (the ranking never changed) while **accuracy falls** as the threshold drops. That is the correct outcome: the cheapest decision is not the most accurate one, because accuracy prices a miss and an alarm the same.

**The common misread:** rejecting the cost-optimal threshold because accuracy or precision went down. Those columns answer a question nobody asked once the cost matrix exists.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: tune the threshold on validation folds, report on test once

The sweep used out-of-fold predictions on the training partition; the held-out set was scored exactly once per threshold, after the choice was made. Sweeping thresholds on the test set and reporting the best one is a leak: the reported number would be optimistic by about the fold-to-fold noise you saw in Task 9. Calibrate first, then choose the threshold: a calibrated probability makes `cost(t)` an honest expectation, and the same threshold keeps its meaning next month.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ The threshold belongs to the model it was tuned on

`t_cost` was chosen for the sigmoid-calibrated LightGBM. Applying it to the raw model, or to the blend, gives a different flag rate and a different cost, because their score distributions differ (the histograms in 11.2). Whenever the model changes, the threshold sweep runs again.

</div>

## 11.6 Task 11 Summary

**✅ Key Takeaways**
- `calibrate_model` wraps the model in `CalibratedClassifierCV`; sigmoid for small data, isotonic when there is enough; AUC unchanged, log loss / Brier / ECE improved.
- The threshold is a decision: sweep it on out-of-fold predictions, pick by expected cost, report on the held-out set once.
- `optimize_threshold` does the sweep with full cross-validations per point (bound it with `n=`/`options=`) and returns a thresholded classifier that `predict_model` and `save_model` respect.
- The F1 optimum and the cost optimum differ; accuracy goes down at the cheapest threshold, and that is fine.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `lgbm_sig`, `lgbm_iso` | sigmoid- and isotonic-calibrated versions of `lgbm_t` |
| `cal_table`, `cal_probs`, `ece()` | calibration metrics per version, their held-out probabilities, the ECE helper |
| `work_model`, `oof` | the calibrated working model and its out-of-fold probabilities on the training partition |
| `sweep`, `t_f1`, `t_cost` | the threshold sweep table and the F1- and cost-optimal thresholds |
| `COST_FN`, `COST_FP` | the cost matrix (miss = 5, false alarm = 1) |
| `lgbm_thr`, `thr_data`, `t_pycaret` | `optimize_threshold`'s thresholded classifier, its data, its pick |
| `final_test`, `cost_test` | held-out metrics and costs at the three thresholds |

**➡️ Next Up:** Part 5 switches to regression on next month's spend, and walks into the leakage trap on purpose.


In [ ]:
shutil.rmtree(PLOT_DIR, ignore_errors=True)   # the gallery PNGs are already rendered above
print("Parts 3-4 done. Working model:", type(work_model).__name__, "| cost-optimal threshold:", round(t_cost, 2),
      "| held-out cost/household at that threshold:", round(cost_test.iloc[-1]["cost / household"], 3))


# Part 5 · Regression on Next Month's Spend; Residuals and the Leakage Trap ⭐⭐

<div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

Parts 3 and 4 asked *whether* a household will overspend. This Part asks *how much* it will spend: `spend_next`, a number, not a class. The lifecycle is the same (`setup → compare_models → predict_model`), the module is `pycaret.regression`, and the metrics change to the regression family (Task 12). Then the Part walks into the most common mistake in applied ML on purpose: a target that is a function of its own features, and shows how to recognise a leak from the numbers alone (Task 13).

Part 5 opens its own experiment, a `RegressionExperiment` with `test_data=test_df`, so every held-out number here is on the same 375 households as every other Part.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Why this Part uses `RegressionExperiment()` and not `from pycaret.regression import *`

`pycaret.classification` and `pycaret.regression` export functions with the **same names**: `setup`, `compare_models`, `create_model`, `plot_model`, `predict_model`, `pull`. A star-import from the regression module would silently replace the classification functions from Parts 3 and 4, and `pull()` would then return whichever module was imported last. The object-oriented class owns its own state, so the two experiments live side by side. Every `reg.method(...)` below is `method(...)` in the functional form.

</div>


# Task 12 · Regression: Compare, Metrics, Residuals

## 12.1 From a class to an amount

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

A **regression** model predicts a number. Everything about its evaluation follows from one quantity, the **residual** $r_i = y_i - \hat{y}_i$, the signed gap between what happened and what was predicted. Metrics summarise the residuals in different ways (absolute, squared, relative); diagnostic plots look at where the residuals are large and whether they depend on something the model should have used.

Two things change under the hood compared with classification: the cross-validation folds are plain `KFold` (there is no class to stratify on), and the default sort of the comparison table is **R²** rather than accuracy. This Part sorts by **MAE** instead, because a mean absolute error in currency units is what a budgeting product would report.

</div>


In [ ]:
from pycaret.regression import RegressionExperiment
import os, inspect, pathlib, tempfile, shutil

DROP_REG = [ID_COL, LABEL_CLS, "note"]      # id, the classification target, and free text (Part 2 explains `note`)
sig = inspect.signature(RegressionExperiment.setup).parameters
assert {"test_data", "transform_target", "transform_target_method", "index", "normalize"} <= set(sig), "argument names changed"

reg = RegressionExperiment()
with timer("regression setup"):
    reg.setup(train_df.drop(columns=DROP_REG), target=LABEL_REG,
              test_data=test_df.drop(columns=DROP_REG),      # the notebook's own held-out set
              normalize=True, index=False,
              fold=3, session_id=RANDOM_STATE, n_jobs=4, verbose=False, html=False)

summary = reg.pull().set_index("Description")["Value"]
for key in ["Target", "Target type", "Original data shape", "Transformed train set shape", "Transformed test set shape",
            "Numeric imputation", "Normalize", "Fold Generator", "Fold Number"]:
    if key in summary.index:
        print(f"{key:>30}: {summary[key]}")
print(f"\nheld-out rows inside the experiment: {len(reg.get_config('X_test'))}  ==  len(test_df): {len(test_df)}")
print(f"target on the training partition: mean {reg.get_config('y_train').mean():,.0f}, std {reg.get_config('y_train').std():,.0f}, "
      f"min {reg.get_config('y_train').min():,.0f}, max {reg.get_config('y_train').max():,.0f}")


## 12.2 The regression metrics

### 📐 The Math (gently)

With $n$ households, true values $y_i$, predictions $\hat{y}_i$ and residuals $r_i = y_i - \hat{y}_i$:

$$ \text{MAE} = \frac{1}{n}\sum_i |r_i|, \qquad \text{MSE} = \frac{1}{n}\sum_i r_i^2, \qquad \text{RMSE} = \sqrt{\text{MSE}} $$

$$ R^2 = 1 - \frac{\sum_i r_i^2}{\sum_i (y_i - \bar{y})^2}, \qquad \text{RMSLE} = \sqrt{\frac{1}{n}\sum_i \big(\log(1+\hat{y}_i) - \log(1+y_i)\big)^2}, \qquad \text{MAPE} = \frac{1}{n}\sum_i \frac{|r_i|}{|y_i|} $$

| Column | Units | Punishes | Read it as |
|---|---|---|---|
| **MAE** | currency | every unit of error equally | "typically off by this much"; robust to a few wild misses |
| **MSE** | currency² | large misses quadratically | hard to read; the thing least-squares models minimise |
| **RMSE** | currency | large misses | MAE's heavier-tailed cousin; RMSE ≫ MAE means a few very bad predictions |
| **R²** | none | relative to predicting the mean | 0 = no better than the mean, 1 = perfect, negative = worse than the mean |
| **RMSLE** | none | relative error, under-prediction more than over-prediction | for targets spanning orders of magnitude |
| **MAPE** | fraction | relative error | "off by 8%" is easy to explain, and it explodes when $y_i$ is near zero |

The baseline for every column is a model that predicts the training mean: MAE equal to the mean absolute deviation, R² of zero. A second, sharper baseline exists in this dataset: **predict that next month equals this month** (`total_spend`). Section 12.4 tests both.

## 12.3 The comparison, sorted by MAE


In [ ]:
INCLUDE_REG = ["lr", "ridge", "lasso", "rf", "et", "lightgbm", "gbr", "knn"]
with timer(f"compare_models on {len(INCLUDE_REG)} regressors x 3 folds"):
    reg_top3 = reg.compare_models(include=INCLUDE_REG, fold=3, sort="MAE", n_select=3, verbose=False)
reg_cmp = reg.pull()
reg_best = reg_top3[0]
print(reg_cmp.to_string())
print("\nn_select=3 returned:", [type(m).__name__ for m in reg_top3])

y_tr = reg.get_config("y_train")
mae_mean_baseline = float((y_tr - y_tr.mean()).abs().mean())
print(f"MAE of predicting the training mean (the floor): {mae_mean_baseline:,.1f}")
pretty(reg_cmp.drop(columns=["Model"]), highlight="min", subset=["MAE", "MSE", "RMSE", "RMSLE", "MAPE"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
labels = reg_cmp["Model"].str.replace(" Regressor", "").str.replace(" Regression", "").str.replace("Light Gradient Boosting Machine", "LightGBM")
axes[0].bar(labels, reg_cmp["MAE"], color="#f5576c")
axes[0].axhline(mae_mean_baseline, color="gray", ls="--", lw=1, label=f"predict the mean (MAE {mae_mean_baseline:,.0f})")
axes[0].set_title("compare_models: cross-validated MAE per model (sorted)"); axes[0].set_ylabel("MAE (currency)")
axes[0].set_ylim(0, mae_mean_baseline * 1.1); axes[0].tick_params(axis="x", rotation=25); axes[0].legend(loc="upper left")
axes[1].bar(labels, reg_cmp["R2"], color="#764ba2")
axes[1].axhline(0, color="gray", ls="--", lw=1, label="predict the mean (R² = 0)")
axes[1].set_title("Cross-validated R² per model"); axes[1].set_ylabel("R²"); axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=25); axes[1].legend(loc="lower left")
plt.tight_layout(); plt.show()
print(f"MAE spread across models: {reg_cmp['MAE'].max() - reg_cmp['MAE'].min():,.1f}   |   best MAE is {reg_cmp['MAE'].min() / mae_mean_baseline:.0%} of the mean-baseline MAE")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left, the cross-validated MAE of every model against the dashed floor of "predict the mean"; every model is far below it. Right, the same ranking as R². The three linear models sit at the top of the table, ahead of the tree ensembles: next month's spend is close to a linear function of this month's (rent is a fixed share of income, groceries scale with household size), and a linear model with 1,125 rows estimates those slopes with less variance than a forest. KNN trails because distance in twenty-two z-scored dimensions is a poor guide to spend.

**The common misread:** "R² of 0.8 means 80% accurate". R² is the share of *variance* explained relative to the mean; the MAE column is the number a person can check against a bank statement.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Every number in the table is cross-validation on the training partition

As in Part 3: `compare_models` never touches `test_data`. The held-out set is opened next, once, for the three models that survived.

</div>

## 12.4 The held-out set, with two baselines

`predict_model(model)` with no `data=` scores the held-out partition and returns the rows with a `prediction_label` column (the name is shared with classification; here it holds the predicted amount). `report_card(task="regression")` adds `MedAE` (the median absolute error, blind to a few huge misses) and `bias` (the mean signed residual: positive means the model predicts too low on average).


In [ ]:
def holdout_reg(model):
    # true values and predictions on the experiment's held-out set (= test_df)
    p = reg.predict_model(model, verbose=False)
    return p[LABEL_REG].values, p["prediction_label"].values

cards, reg_preds = [], {}
for m in reg_top3:
    name = type(m).__name__.replace("Regressor", "").replace("Regression", "")
    y_true, y_hat = holdout_reg(m)
    cards.append(report_card(y_true, y_hat, name=name, task="regression"))
    reg_preds[name] = y_hat

# two baselines every regression table should carry
cards.append(report_card(y_true, np.full(len(y_true), y_tr.mean()), name="predict the train mean", task="regression"))
cards.append(report_card(y_true, test_df["total_spend"].values, name="next month = this month", task="regression"))
reg_test = pd.concat(cards)
print(f"held-out households: {len(y_true)}   |   mean spend_next: {y_true.mean():,.0f}")
print(reg_test.round(1).to_string())
pretty(reg_test, highlight="min", subset=["MAE", "RMSE", "MedAE", "MAPE_%"])


In [ ]:
_ = reg.predict_model(reg_best, verbose=False)
pc_row = reg.pull().iloc[0]
rc_row = reg_test.iloc[0]
check = pd.DataFrame({"pycaret pull()": [pc_row["MAE"], pc_row["RMSE"], pc_row["R2"], pc_row["MAPE"] * 100],
                      "report_card": [rc_row["MAE"], rc_row["RMSE"], rc_row["R2"], rc_row["MAPE_%"]]},
                     index=["MAE", "RMSE", "R2", "MAPE (%)"])
print(check.round(3).to_string())
print("\nagree to 2 decimals:", bool(np.allclose(check["pycaret pull()"], check["report_card"], atol=1e-2)))
print("bias of the best model (mean of y - y_hat):", f"{rc_row['bias']:+.1f}", "| of 'next month = this month':", f"{reg_test.loc['next month = this month', 'bias']:+.1f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading the table

The two baseline rows frame everything above them. "Predict the train mean" is the floor: R² near zero, an MAE equal to the target's spread. "Next month = this month" is the baseline that matters: it costs nothing, it needs no model, and its MAE is far better than the mean. Every real model has to beat *that* row, and the margin by which it does is the value the model adds. Its `bias` (mean of actual minus predicted) is large and positive: December, the target month, is the planted holiday bump, so "same as last month" predicts too low for everyone; the fitted models learned the shift from the training partition and their bias sits near zero.

`pull()` after `predict_model` reproduces `report_card` (MAPE is reported as a fraction by PyCaret and as a percentage by `report_card`).

**The common misread:** celebrating R² against the mean baseline. The honest lift is against the best *cheap* prediction, which in any month-to-month problem is persistence.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: put persistence in every regression table

For any target with a "last value" (last month's spend, yesterday's demand, the previous reading), the persistence baseline `y_next = y_now` is the row to beat, and its `bias` tells you whether the period you are forecasting is systematically different from the one you are forecasting from. A model that does not clearly beat persistence on the held-out set should not ship, whatever its R².

</div>

## 12.5 `transform_target`: fitting on a transformed scale

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

Spend is right-skewed: many households near the middle, a long tail of big spenders. Least-squares models weight the tail heavily, and tree models split it coarsely. `setup(transform_target=True)` fits a **Yeo-Johnson** power transform to the target (a family of curves that includes the logarithm as a special case, chosen by maximum likelihood), trains every model on the transformed target, and inverts the transform inside `predict_model`, so predictions and metrics come back in currency. Whether it helps depends on the target's shape; the next cell runs both ways and lets the numbers decide.

</div>


In [ ]:
reg_tt = RegressionExperiment()
with timer("second experiment: transform_target=True"):
    reg_tt.setup(train_df.drop(columns=DROP_REG), target=LABEL_REG, test_data=test_df.drop(columns=DROP_REG),
                 normalize=True, index=False, transform_target=True, transform_target_method="yeo-johnson",
                 fold=3, session_id=RANDOM_STATE, n_jobs=4, verbose=False, html=False)
print("pipeline steps with transform_target:", [n for n, _ in reg_tt.pipeline.steps])
y_raw, y_tf = reg_tt.get_config("y_train"), reg_tt.get_config("y_train_transformed")

rows = []
best_id = reg_cmp.index[0]
for exp_, label in [(reg, "raw target"), (reg_tt, "yeo-johnson target")]:
    for mid in [best_id, "lightgbm"]:
        m = exp_.create_model(mid, fold=3, verbose=False)
        cv_mae = exp_.pull().loc["Mean", "MAE"]
        p = exp_.predict_model(m, verbose=False)
        rows.append({"target": label, "model": mid, "CV MAE": cv_mae, "held-out MAE": mean_absolute_error(p[LABEL_REG], p["prediction_label"]),
                     "held-out RMSE": float(np.sqrt(mean_squared_error(p[LABEL_REG], p["prediction_label"])))})
tt_table = pd.DataFrame(rows).set_index(["target", "model"])
print(tt_table.round(1).to_string())
print(f"\nspend_next on the training partition: raw skew {y_raw.skew():.2f}, std {y_raw.std():,.0f}  ->  transformed skew {y_tf.skew():.2f}, std {y_tf.std():.3f}")
print(f"{best_id} regularisation strength (alpha) as PyCaret sets it: {reg_best.get_params().get('alpha', 'n/a')}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].hist(y_raw, bins=40, color="#764ba2"); axes[0].set_title(f"spend_next, raw (skew {y_raw.skew():.2f})"); axes[0].set_xlabel("currency"); axes[0].set_ylabel("households")
axes[1].hist(y_tf, bins=40, color="#f5576c"); axes[1].set_title(f"spend_next after Yeo-Johnson (skew {y_tf.skew():.2f})"); axes[1].set_xlabel("transformed units")
w = 0.38; x = np.arange(2)
for i, (label, color) in enumerate([("raw target", "#764ba2"), ("yeo-johnson target", "#f5576c")]):
    vals = tt_table.loc[label, "held-out MAE"].values
    axes[2].bar(x + (i - 0.5) * w, vals, w, color=color, label=label)
axes[2].set_xticks(x); axes[2].set_xticklabels([best_id, "lightgbm"]); axes[2].set_ylabel("held-out MAE"); axes[2].set_title("Does the target transform help?")
axes[2].set_ylim(tt_table["held-out MAE"].min() * 0.9, tt_table["held-out MAE"].max() * 1.05); axes[2].legend(loc="upper left")
plt.tight_layout(); plt.show()
delta = tt_table.loc["yeo-johnson target", "held-out MAE"] - tt_table.loc["raw target", "held-out MAE"]
print("held-out MAE change from transforming the target (negative = better):", delta.round(1).to_dict())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left and middle: the transform pulls in the right tail and leaves a symmetric target, but look at the axis: the transformed values live on a scale hundreds of times smaller than currency (the printed standard deviations). Right: the held-out MAE of the same two models trained on each version. For LightGBM the transform changes almost nothing: trees only care about the *order* of target values within a leaf, and a monotone transform keeps the order. For Lasso the transform is a disaster: its held-out MAE jumps to the level of the mean baseline. The reason is the L1 penalty, `alpha=1.0` in PyCaret's default, which is set in the target's units; on a target whose whole spread is a fraction of one unit, that penalty shrinks every coefficient to zero and the model predicts a constant.

**The common misread:** applying `transform_target=True` as a reflex whenever the target is money. It helps when the target spans orders of magnitude (prices across a whole market) or when residuals grow with the target, and it must always be judged on the inverted scale, as here.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ A regularisation strength is a number in the target's units

`alpha` in Lasso and Ridge, `C` in an SVM, the leaf penalties in a boosted tree: every one of them is compared against the size of the loss, and the loss is measured in the target's units. Change the target's scale (a power transform, a log, a currency conversion) and the same hyperparameter means something entirely different. The fix is not to avoid the transform; it is to retune (`tune_model`, Task 9) after any change to the target, and to keep a persistence baseline in the table so that a collapsed model is visible at once.

</div>

## 12.6 The regression plot gallery

`plot_model` in the regression module has its own set of pictures: `residuals`, `error`, `feature`, `learning`, `cooks`, `vc`, `manifold`, `rfe`. The helper below is the same route as Part 3 (`save=True`, then show the PNG) written so that Part 6 can reuse it for `interpret_model`, whose files are named by SHAP rather than by PyCaret.


In [ ]:
PLOTS56 = pathlib.Path(tempfile.mkdtemp(prefix="pycaret_p56_"))

def render(fn, ax=None, title=None, **kw):
    # Run any PyCaret plotting call with save=True inside PLOTS56, then imshow the newest PNG. Works in any runtime.
    cwd = os.getcwd(); os.chdir(PLOTS56)
    try:
        fn(save=True, **kw)
        fname = max(PLOTS56.glob("*.png"), key=os.path.getmtime)
    finally:
        os.chdir(cwd)
    img = plt.imread(fname); own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.imshow(img); ax.axis("off"); ax.set_title(title or fname.stem)
    if own:
        plt.tight_layout(); plt.show()
    return fname.name

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
with timer("residuals + error"):
    f1 = render(lambda **k: reg.plot_model(reg_best, plot="residuals", verbose=False, **k), ax=axes[0], title="plot_model(plot='residuals')")
    f2 = render(lambda **k: reg.plot_model(reg_best, plot="error", verbose=False, **k), ax=axes[1], title="plot_model(plot='error')")
plt.tight_layout(); plt.show()
print("saved:", f1, "|", f2, " in", PLOTS56)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `residuals` and `error`

**Residuals (left):** residual against predicted value, training rows and held-out rows in two colours, with a histogram of the residuals on the side and the R² of each partition in the legend. **Can tell you:** whether the errors have structure (a funnel means the error grows with the prediction; a curve means a missing non-linearity) and whether the train and test clouds look alike (a much tighter train cloud is overfitting). **Cannot tell you:** which feature is responsible.

**Prediction error (right):** predicted against actual on the held-out set, with the identity line (perfect predictions) and the best-fit line. **Can tell you:** systematic under- or over-prediction at the high end, where the best-fit line bends away from the identity. **Cannot tell you:** anything about the size of typical errors in currency; read MAE for that.

**The common misread:** a residual histogram that looks Gaussian read as "the model is fine". Symmetric residuals can still depend on a feature; Section 12.7 checks that by city tier.

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
with timer("feature + learning"):
    render(lambda **k: reg.plot_model(reg_best, plot="feature", verbose=False, **k), ax=axes[0], title=f"plot_model(plot='feature') on {type(reg_best).__name__}")
    render(lambda **k: reg.plot_model(reg_best, plot="learning", verbose=False, **k), ax=axes[1], title="plot_model(plot='learning')")
plt.tight_layout(); plt.show()
cols_reg = reg.get_config("X_train_transformed").columns
if hasattr(reg_best, "coef_"):
    reg_coefs = pd.Series(np.ravel(reg_best.coef_), index=cols_reg).sort_values(key=np.abs, ascending=False)
    print(f"top 5 |coefficient| on z-scored features ({type(reg_best).__name__}); units are currency per standard deviation of the feature:")
else:
    reg_coefs = pd.Series(reg_best.feature_importances_, index=cols_reg).sort_values(ascending=False)
    print(f"top 5 split-based importances ({type(reg_best).__name__}):")
print(reg_coefs.head(5).round(1).to_string())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 `feature` and `learning`

**Feature (left):** for a linear model, the coefficients on the z-scored features, so each bar reads "currency of next-month spend per one standard deviation of this feature". The largest bar by far is `avg_spend_prev`, the household's average spend over all earlier months: the long-run average is a better predictor than any single noisy month, including this one. Then come `rent`, `income` and `groceries`, the fixed costs the generator planted. **Cannot tell you:** which features matter *on the held-out set* under permutation (Part 6 computes that) or anything causal.

**Learning curve (right):** training and cross-validation score as the training set grows. Two lines that have met and flattened mean the model is data-saturated: more households would not help, better features might. A gap that is still closing means more data would help.

**The common misread:** a large coefficient on a category share read as "that category drives spending". Coefficients on correlated, standardised features are shared out among them; `rent` and `income` are strongly correlated here, and the split between their two bars is partly arbitrary.

</div>

## 12.7 Residual analysis by city tier

The gallery plots pool every household. A segment breakdown asks whether the model is systematically wrong for some group, which matters for fairness (Part 9) and for where to look for a missing feature.


In [ ]:
p = reg.predict_model(reg_best, verbose=False)
resid = pd.DataFrame({"city_tier": test_df["city_tier"].values, "household_size": test_df["household_size"].values,
                      "y": p[LABEL_REG].values, "y_hat": p["prediction_label"].values})
resid["residual"] = resid["y"] - resid["y_hat"]

seg_cards = pd.concat([report_card(g["y"], g["y_hat"], name=f"tier = {tier}", task="regression")
                       for tier, g in resid.groupby("city_tier")] +
                      [report_card(resid["y"], resid["y_hat"], name="all households", task="regression")])
print(seg_cards.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = ["metro", "city", "town"]
sns.boxplot(data=resid, x="city_tier", y="residual", order=order, ax=axes[0], palette=["#667eea", "#28a745", "#ffc107"])
axes[0].axhline(0, color="k", lw=1, ls="--"); axes[0].set_title("Residual (actual - predicted) by city tier, held-out set"); axes[0].set_ylabel("residual (currency)")
sns.scatterplot(data=resid, x="y_hat", y="residual", hue="city_tier", hue_order=order, ax=axes[1], s=18, alpha=0.7, palette=["#667eea", "#28a745", "#ffc107"])
axes[1].axhline(0, color="k", lw=1, ls="--"); axes[1].set_title("Residual vs predicted, coloured by tier"); axes[1].set_xlabel("predicted spend_next"); axes[1].set_ylabel("residual"); axes[1].legend(title="city_tier")
plt.tight_layout(); plt.show()
worst = seg_cards["MAE"].drop("all households").idxmax()
print(f"largest segment MAE: {worst} ({seg_cards.loc[worst, 'MAE']:,.0f}) vs overall {seg_cards.loc['all households', 'MAE']:,.0f}; "
      f"segment bias ranges {seg_cards['bias'].drop('all households').min():+.0f} to {seg_cards['bias'].drop('all households').max():+.0f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: the residual distribution per tier. The box centres sit close to zero; the `bias` column of the table gives the exact offsets, all a small fraction of the MAE, with metro the largest (the model predicts slightly low for metro households, whose rent is the biggest and noisiest category in currency). Right: the same residuals against the prediction; the spread widens to the right for every tier, which is the funnel shape the `residuals` plot hinted at: the error grows with the amount, as it does for almost every money target. The outliers far above zero are households hit by a surprise expense in the target month, which no feature from this month could have predicted.

**The common misread:** "the model is unfair to metro households" from the wider box alone. Fairness is about *bias* per group (the box centres), not about *variance*; a wider box with a centre at zero means the group is harder to predict, not that the model is tilted against it.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: one `report_card` per segment, on the held-out set

A segment table costs one `groupby` and catches what the pooled MAE hides: a group with a non-zero bias, or a group whose MAE is twice the others. Do it by the columns a stakeholder would ask about (region, size, tenure), and keep the "all households" row at the bottom for scale.

</div>

## 12.8 Task 12 Summary

**✅ Key Takeaways**
- `RegressionExperiment()` keeps the regression module apart from the classification functions of Parts 3-4; the lifecycle calls are identical.
- MAE is the number in currency; RMSE punishes big misses; R² is relative to the mean; MAPE is relative to the target and unstable near zero.
- The baseline to beat is **persistence** (`spend_next = total_spend`), not the mean; its `bias` shows the December shift the models learned.
- `transform_target=True` fits on a Yeo-Johnson scale and inverts inside `predict_model`; judge it on the inverted MAE. Trees barely noticed; Lasso collapsed because its `alpha` is a number in the target's units.
- The `residuals` / `error` plots show the funnel (error grows with the amount); the by-tier table shows small segment biases and no segment that the model gets systematically wrong.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `reg`, `DROP_REG` | the `RegressionExperiment` opened with `test_data=test_df`; the columns dropped before it |
| `reg_top3`, `reg_best`, `reg_cmp` | the three best regressors by CV MAE, the first of them, the comparison table |
| `reg_test`, `holdout_reg(model)` | held-out `report_card` rows for the top 3 plus the mean and persistence baselines; the helper |
| `reg_tt`, `tt_table` | the `transform_target=True` experiment and the with/without comparison |
| `render(fn, ax=, title=)` | runs any PyCaret plot call with `save=True` and shows the PNG; `PLOTS56` holds the files |
| `reg_coefs`, `resid`, `seg_cards` | the best model's coefficients, held-out residuals with segments, the per-tier report cards |

**➡️ Next Up:** Task 13 builds a table that scores R² of nearly 1, on purpose, and learns to distrust it.


# Task 13 · The Leakage Trap

## 13.1 What a leak is

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

**Leakage** is any information in the training features that will not be available, in the same form, when the model is used. The purest case is a feature that is a *function of the label*. In the budget data, `total_spend` for a month is the sum of that month's seven category columns. A model asked to predict this month's total from this month's categories only has to learn to add, and it will do so with an R² of one on any split, on any month, for any household. It is also completely useless: by the time the categories are known, so is the total.

The honest question, the one every other Part answers, is **next** month's total from **this** month's information. The next cell builds the wrong table on purpose and runs the same `compare_models` on it.

</div>


In [ ]:
# the WRONG table: this month's total_spend as the target, this month's categories as features
# (`overspent` is total_spend > 0.65 * income, another function of the label, so it goes too)
DROP_LEAK = [ID_COL, LABEL_CLS, LABEL_REG, "note", "overspent"]
leak = RegressionExperiment()
with timer("leaky experiment: setup + compare_models on 3 models"):
    leak.setup(train_df.drop(columns=DROP_LEAK), target="total_spend", test_data=test_df.drop(columns=DROP_LEAK),
               normalize=True, index=False, fold=3, session_id=RANDOM_STATE, n_jobs=4, verbose=False, html=False)
    leak_best = leak.compare_models(include=["lr", "lightgbm", "rf"], fold=3, sort="R2", verbose=False)
leak_cmp = leak.pull()
print(leak_cmp.to_string())

honest = reg_cmp.loc[leak_cmp.index, ["MAE", "R2"]]
side = pd.DataFrame({"leaky CV R2": leak_cmp["R2"], "honest CV R2 (Task 12)": honest["R2"],
                     "leaky CV MAE": leak_cmp["MAE"], "honest CV MAE (Task 12)": honest["MAE"]})
print("\n" + side.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(side)); w = 0.38
axes[0].bar(x - w / 2, side["leaky CV R2"], w, color="#dc3545", label="leaky: this month's total from this month's categories")
axes[0].bar(x + w / 2, side["honest CV R2 (Task 12)"], w, color="#2196F3", label="honest: next month's total")
axes[0].set_xticks(x); axes[0].set_xticklabels(side.index); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("cross-validated R²")
axes[0].axhline(1, color="k", ls=":", lw=1); axes[0].set_title("R²: the leak scores almost perfectly"); axes[0].legend(loc="lower left", fontsize=9)
axes[1].bar(x - w / 2, side["leaky CV MAE"], w, color="#dc3545", label="leaky"); axes[1].bar(x + w / 2, side["honest CV MAE (Task 12)"], w, color="#2196F3", label="honest")
axes[1].set_xticks(x); axes[1].set_xticklabels(side.index); axes[1].set_ylabel("cross-validated MAE (currency)"); axes[1].set_title("MAE: the leak is off by a few units"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"leaky R² ranges {side['leaky CV R2'].min():.4f} to {side['leaky CV R2'].max():.4f}; honest R² ranges {side['honest CV R2 (Task 12)'].min():.3f} to {side['honest CV R2 (Task 12)'].max():.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Red bars are the leaky table, blue the honest one, for the same three models with the same folds. The linear model on the leak reaches an R² of one to four decimals: the target is a sum, and a linear model is a sum. The tree models come close but not all the way, because trees approximate a plane with steps. On the honest table every model drops to an R² around 0.8 and an MAE in the hundreds. Nothing about the modelling changed; only the target did.

**The common misread:** "the linear model is best, it gets R² of 1". A perfect score is not a good result, it is a warning. In a real project the leaky column is rarely as obvious as `rent + groceries + ...`: it is the "days since last payment" that was computed after the default, or the "account status" that was updated when the label was written.

</div>

## 13.2 Three signals that say "leak"

The next cell computes the three checks that catch most leaks, for both tables, and reports each as a ✅ or ⚠️ flag.

1. **Too good to be true:** a cross-validated R² above 0.99 (or an AUC above 0.99) on real data.
2. **Importance concentrated on the label's ingredients:** the share of the model's importance that sits on the seven category columns, the addends of `total_spend`.
3. **Time order:** the month the features come from versus the month the label comes from. A label measured in the same period as its features is contemporaneous, not predictive.


In [ ]:
def importance_share(model, cols, on):
    # share of |coefficient| or split importance that sits on the columns in `on`
    vals = np.abs(np.ravel(model.coef_)) if hasattr(model, "coef_") else np.asarray(model.feature_importances_, dtype=float)
    s = pd.Series(vals, index=cols)
    return float(s[[c for c in on if c in s.index]].sum() / s.sum())

CATS = ["rent", "groceries", "utilities", "transport", "dining", "entertainment", "other"]
months = sorted(budget.month.unique())
feat_month, target_month = months[-2], months[-1]        # how household_table() in Part 0 built hh

checks = pd.DataFrame({
    "leaky table":  [leak_cmp["R2"].max(), importance_share(leak_best, leak.get_config("X_train_transformed").columns, CATS), f"{feat_month:%b %Y} -> {feat_month:%b %Y}"],
    "honest table": [reg_cmp["R2"].max(), importance_share(reg_best, cols_reg, CATS), f"{feat_month:%b %Y} -> {target_month:%b %Y}"],
}, index=["best CV R2", "importance share on the 7 category columns", "feature month -> label month"])
flag = lambda ok: "✅" if ok else "⚠️"
checks["verdict (leaky | honest)"] = [
    f"{flag(leak_cmp['R2'].max() < 0.99)} | {flag(reg_cmp['R2'].max() < 0.99)}",
    f"{flag(checks.iloc[1, 0] < 0.5)} | {flag(checks.iloc[1, 1] < 0.5)}",
    f"{flag(False)} | {flag(True)}",
]
print(checks.to_string())
print(f"\nrender of plot_model(plot='feature') on the leaky {type(leak_best).__name__} follows; its largest bars are the category columns themselves")
render(lambda **k: leak.plot_model(leak_best, plot="feature", verbose=False, **k), title=f"leaky table: plot_model(plot='feature') on {type(leak_best).__name__}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

The table is the checklist with numbers filled in; the picture is PyCaret's `feature` plot for the leaky winner, where the largest bars are the category columns and, for a linear model on z-scored features, `rent` and `groceries` come first because they have the largest spread in currency. The honest model in Task 12 leaned on `avg_spend_prev` and `income`, history and capacity rather than the addends of a sum.

**The common misread:** treating check 2 as a fixed threshold. Importance concentrated on a few columns is normal when those columns are legitimately predictive; the flag is raised only when the dominant columns are *ingredients of the label*, which is a question about the data dictionary, not about the number.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: write down, per feature, *when* it becomes known

Before the first `setup`, add a column to the data dictionary: "available at prediction time?" with the timestamp at which each feature is fixed. Every feature whose timestamp is at or after the label's is a leak. That one column would have caught `total_spend`, `overspent` and every derived column in this table, before any model was trained.

</div>

## 13.3 The subtler leak: the split, and what is fitted before it

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

Every preprocessing step that *learns* something (an imputation mean, a scaler's mean and standard deviation, the features kept by `feature_selection`, the synthetic rows of `fix_imbalance`) must learn it from the training partition only. `setup(..., test_data=...)` guarantees that: the pipeline is fitted on the train partition and merely applied to the held-out one. The next cell reads the fitted imputer's statistic out of the pipeline to prove it. The trap is upstream: running `setup` on the **whole** table, then scoring rows that a previous split called "test". PyCaret fits on its own 70%, but that 70% overlaps with your test rows.

</div>


In [ ]:
imputer = reg.pipeline.named_steps["numerical_imputer"].transformer      # a fitted sklearn SimpleImputer
i_dining = list(imputer.feature_names_in_).index("dining")
print(f"imputation value for `dining` inside the pipeline: {imputer.statistics_[i_dining]:.3f}")
print(f"mean of `dining` on train_df: {train_df.dining.mean():.3f}   |   on the full table hh: {hh.dining.mean():.3f}   |   on test_df: {test_df.dining.mean():.3f}")
print("=> with test_data=, the pipeline learned from the training partition only:", np.isclose(imputer.statistics_[i_dining], train_df.dining.mean()))

# the trap: setup on the full table (no test_data) and then 'testing' on the notebook's test_df
full_exp = RegressionExperiment()
with timer("setup on the full table, no test_data"):
    full_exp.setup(hh.drop(columns=DROP_REG), target=LABEL_REG, normalize=True, index=False, fold=3,
                   session_id=RANDOM_STATE, n_jobs=4, verbose=False, html=False)
inner_train = full_exp.get_config("X_train")
overlap = pd.merge(inner_train[["income", "rent", "avg_spend_prev"]], test_df[["income", "rent", "avg_spend_prev"]], how="inner").drop_duplicates()
print(f"\nPyCaret's own training partition: {len(inner_train):,} rows; of the notebook's {len(test_df)} test households, "
      f"{len(overlap)} ({len(overlap) / len(test_df):.0%}) are inside it -> any 'test' score on test_df would be partly a training score")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Two leaks hide in the split itself

1. **Fitting before splitting.** The imputation value above equals the *training* mean, not the full-table mean, because the split happened first. A `fillna(df.mean())` or a `StandardScaler().fit(df)` run on the whole table before `setup` would use test rows to set training statistics. The effect is tiny for a mean and large for anything target-aware: `feature_selection`, `fix_imbalance`, target encoding.
2. **A split that PyCaret did not see.** `setup(hh)` without `test_data` made its own 70/30 split, and the printout shows how much of the notebook's `test_df` landed inside it. Every "test" number on those rows would be optimistic. Pass `test_data=` (Part 2, Task 5), or take the held-out rows from `get_config("X_test")` and never from an outside table.

</div>

## 13.4 Rows that share a household, and the time-aware split

`hh` has one row per household, so a random split never puts the same household on both sides. Panel tables (one row per household **per month**, as `budget` is) do. The next cell builds a two-origin panel from `budget` and counts the households a random split places on both sides.


In [ ]:
def origin_table(m):
    # features from month m, target = total_spend of month m + 1
    f = budget[budget.month == months[m]].drop(columns=["month", "note", "referral_code", "overspent"]).copy()
    t = budget[budget.month == months[m + 1]][["household_id", "total_spend"]].rename(columns={"total_spend": "y_next"})
    return f.merge(t, on="household_id").assign(origin=m)

panel2 = pd.concat([origin_table(len(months) - 3), origin_table(len(months) - 2)], ignore_index=True)
tr_rand, te_rand = train_test_split(panel2, test_size=0.25, random_state=RANDOM_STATE)
both = set(tr_rand.household_id) & set(te_rand.household_id)
tr_time, te_time = panel2[panel2.origin == panel2.origin.min()], panel2[panel2.origin == panel2.origin.max()]

sig = inspect.signature(RegressionExperiment.setup).parameters
print(f"two-origin panel: {len(panel2):,} rows = {panel2.household_id.nunique():,} households x 2 origins")
print(f"random 75/25 split: {len(both):,} households ({len(both) / panel2.household_id.nunique():.0%}) appear on BOTH sides")
print(f"time split (train on origin {tr_time.origin.iloc[0]}, test on origin {te_time.origin.iloc[0]}): "
      f"{len(set(tr_time.household_id) & set(te_time.household_id)):,} households on both sides, and the test month is strictly later")
print("\nPyCaret's remedies exist in setup():", {k: (k in sig) for k in ["fold_strategy", "fold_groups", "data_split_shuffle", "test_data"]})
print("  setup(panel, target='y_next', test_data=<later origin>, fold_strategy='groupkfold', fold_groups='household_id')")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading the printout

On a two-origin panel, a random split puts a large share of households on both sides; the model then sees a household's rent, size, income and city on the training side and is tested on the same household one month later. How much that inflates the score depends on how stable the target is per household: on this data the target is next month's total, which is dominated by noise the model cannot memorise, so the damage would be small; on a target that is nearly constant per household (a credit limit, a subscription tier) it is severe. The time split has zero overlap by construction and, just as important, its test rows come from a *later* month, so seasonality and drift are tested rather than averaged away.

**The common misread:** "cross-validation protects against this". Plain `KFold` shuffles rows, not households; only `fold_strategy="groupkfold"` with `fold_groups=` keeps a household on one side, and only a time-ordered `test_data=` tests the future.

</div>

## 13.5 Task 13 Summary

**✅ Key Takeaways**
- A feature that is a function of the label gives a near-perfect score and a useless model; the leaky table scored R² of one for the linear model.
- Three signals: a score too good for real data, importance concentrated on the label's ingredients, and a label measured in the same period as the features.
- With `test_data=`, PyCaret fits every learning step on the training partition only (the imputer's statistic proved it); `setup` on the full table overlaps with any outside test set.
- Panel data needs `fold_groups` (households on one side) and a time-ordered `test_data` (the future on the other).

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `leak`, `leak_best`, `leak_cmp`, `DROP_LEAK` | the leaky experiment (this month's total from this month's categories), its winner and table |
| `side`, `checks` | leaky vs honest CV scores per model; the three-signal checklist |
| `importance_share(model, cols, on)` | share of a model's importance on a set of columns |
| `CATS`, `months`, `feat_month`, `target_month` | the seven category columns and the months behind `hh` |
| `full_exp`, `overlap` | the full-table experiment and the test rows that landed in its training partition |
| `panel2`, `origin_table(m)` | the two-origin panel and its builder |

**➡️ Next Up:** Part 6 opens the classifier and asks it *why*: SHAP values, permutation importance, partial dependence and reason codes.


# Part 6 · Interpretability: SHAP, Reason Codes, What the Model Learned ⭐⭐

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### The Big Picture

A model that predicts well but cannot say why is hard to trust, hard to debug and, in lending or insurance, often illegal to deploy. Part 6 opens the classifier: **SHAP values** give every prediction an additive explanation, **permutation importance** measures what the model actually relies on, **partial dependence** draws the shape of each feature's effect, and **reason codes** turn all of that into a sentence a household could read.

This Part opens its own `ClassificationExperiment` and trains one LightGBM model, so it stands alone.

</div>


# Task 14 · SHAP with `interpret_model`

## 14.1 One model to explain

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

`interpret_model` is PyCaret's door to the `shap` library. It works on **tree models** (LightGBM, XGBoost, random forest, extra trees, decision tree), because it uses `shap.TreeExplainer`, which computes exact Shapley values for trees in polynomial time. Ask it to explain a logistic regression and it raises; that is not a bug but a scope.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Warning: `pycaret.classification` and `pycaret.regression` share function names

`from pycaret.regression import *` in Part 5 and `from pycaret.classification import *` here would leave you with one `setup`, one `create_model`: whichever was imported last. This notebook uses the experiment **objects** (`RegressionExperiment()`, `ClassificationExperiment()`) so both live side by side. Every function you know exists as a method on the object.

</div>


In [ ]:
from pycaret.classification import ClassificationExperiment

DROP_CLS = [ID_COL, LABEL_REG, "note"]
cls6 = ClassificationExperiment()
with timer("classification setup"):
    cls6.setup(train_df.drop(columns=DROP_CLS), target=LABEL_CLS,
               test_data=test_df.drop(columns=DROP_CLS), index=False,
               fold=3, session_id=RANDOM_STATE, n_jobs=4, verbose=False, html=False)
with timer("create_model('lightgbm')"):
    lgbm6 = cls6.create_model("lightgbm", fold=3, verbose=False)
cv6 = cls6.pull()
pred6 = cls6.predict_model(lgbm6, verbose=False, raw_score=True)
card6 = report_card(pred6[LABEL_CLS], pred6["prediction_label"], pred6["prediction_score_1"], name="LightGBM (Part 6)")
display(pretty(card6))
print(f"CV AUC (3 folds): {cv6.loc['Mean', 'AUC']:.3f} ± {cv6.loc['Std', 'AUC']:.3f} | held-out AUC: {card6['roc_auc'].iloc[0]:.3f}")


## 14.2 The summary plot: every feature, every household, one picture

<div style="background-color: #f8f9fa; border-left: 5px solid #6c757d; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📐 The Math (gently)

A **Shapley value** $\phi_i$ is feature $i$'s fair share of the difference between this prediction and the average prediction, averaged over every order in which the features could have been revealed:

$$\phi_i = \sum_{S \subseteq F\setminus\{i\}} \frac{|S|!\,(|F|-|S|-1)!}{|F|!}\,\big[f(S\cup\{i\}) - f(S)\big]$$

The shares are **additive**: base value + $\sum_i \phi_i$ = the model's output for that row (for LightGBM, the log-odds). That additivity is what makes SHAP an explanation and not just an importance score.

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
render(lambda **kw: cls6.interpret_model(lgbm6, plot="summary", **kw), ax=axes[0], title="interpret_model(plot='summary'): SHAP beeswarm")
render(lambda **kw: cls6.interpret_model(lgbm6, plot="correlation", feature="overspend_rate_prev", **kw), ax=axes[1],
       title="interpret_model(plot='correlation', feature='overspend_rate_prev')")
plt.tight_layout(); plt.show()
print("interpret_model draws with shap.TreeExplainer on the transformed training rows; the PNGs were saved and shown inline (works without a display)")


<div style="background-color: #f8f9fa; border-left: 5px solid #6c757d; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left, the **beeswarm**: one row per feature, sorted by mean |SHAP|; each dot is a household, its horizontal position the feature's push on that household's log-odds of overspending, its colour the feature's value (red high, blue low). `overspend_rate_prev` sits on top with a clean colour split: a high past overspend rate pushes right, a low one pushes left. Right, the **dependence plot** for that feature: SHAP value against feature value, coloured by the feature that interacts with it most. The vertical spread at one feature value is the interaction.

**The common misread:** reading the beeswarm as "income is the second most important feature, so increase income to reduce overspending." SHAP describes the model, not the world; it says what the model uses, not what causes the label. Part 5's leakage trap showed how a model can lean on a feature for the wrong reason.

</div>

## 14.3 SHAP by hand: the same numbers, and the additivity check


In [ ]:
import shap
pipe6 = cls6.get_config("pipeline")                       # the fitted preprocessing steps (imputer, encoder, ...)
X6_tr = cls6.get_config("X_train_transformed")
X6_te = cls6.get_config("X_test_transformed")
y6_te = cls6.get_config("y_test")

explainer = shap.TreeExplainer(lgbm6)
sv = explainer.shap_values(X6_te)
sv = sv[1] if isinstance(sv, list) else sv                # older shap returns [class0, class1]; keep the positive class
base = explainer.expected_value[1] if np.ndim(explainer.expected_value) else explainer.expected_value

margin = lgbm6.predict(X6_te, raw_score=True)             # LightGBM's log-odds output
gap = np.abs(base + sv.sum(axis=1) - margin).max()
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=X6_te.columns).sort_values(ascending=False)
print(f"transformed test matrix: {X6_te.shape} | base value (log-odds) {base:.3f} = P {1 / (1 + np.exp(-base)):.3f}")
print(f"additivity: max |base + sum(SHAP) - log-odds| = {gap:.1e}")
print("mean |SHAP| top 8:", ", ".join(f"{k}={v:.3f}" for k, v in mean_abs.head(8).items()))


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: explain the transformed row, then translate back

SHAP values belong to the columns the model saw: `city_tier_metro`, not `city_tier`. `get_config("X_test_transformed")` gives exactly those columns after the pipeline (imputation, one-hot, scaling), so the explainer and the model agree. When you report to a person, sum the SHAP values of the one-hot columns that came from one original feature; the sum is the original feature's share.

</div>

## 14.4 Reason codes: one household, in words

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

`interpret_model(plot="reason", observation=i)` draws shap's force plot, which is an interactive HTML widget. Regulations (and product managers) want the same content as text: the top three features that pushed this decision, with direction. The cell below builds that from the SHAP row, for the most confident true positive and for the most confident false negative, so you can see both a good explanation and an explanation of a mistake.

</div>


In [ ]:
p6 = pred6["prediction_score_1"].to_numpy()
y6 = pred6[LABEL_CLS].to_numpy()
i_tp = int(np.argmax(np.where(y6 == 1, p6, -1)))           # overspent, model most sure
i_fn = int(np.argmin(np.where(y6 == 1, p6, 2)))            # overspent, model most sure it would NOT

def reason_codes(i, k=3):
    row = pd.Series(sv[i], index=X6_te.columns)
    top = row.reindex(row.abs().sort_values(ascending=False).index).head(k)
    lines = []
    for feat, val in top.items():
        x = X6_te.iloc[i][feat]
        lines.append(f"{'raises' if val > 0 else 'lowers'} the risk: {feat} = {x:.2f} (contribution {val:+.2f} log-odds)")
    return lines

for label, i in [("most confident TRUE POSITIVE", i_tp), ("most confident FALSE NEGATIVE", i_fn)]:
    print(f"{label}: test row {i}, actual {y6[i]}, P(overspend) = {p6[i]:.3f}")
    for line in reason_codes(i):
        print("   -", line)
    print()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, i, ttl in zip(axes, [i_tp, i_fn], ["true positive", "false negative"]):
    row = pd.Series(sv[i], index=X6_te.columns)
    top = row.reindex(row.abs().sort_values(ascending=False).index).head(8)[::-1]
    ax.barh(top.index, top.values, color=["#dc3545" if v > 0 else "#2196F3" for v in top.values])
    ax.axvline(0, color="k", lw=0.8); ax.set_title(f"{ttl}: row {i}, P = {p6[i]:.2f} (base {1 / (1 + np.exp(-base)):.2f})"); ax.set_xlabel("SHAP value (log-odds)")
plt.tight_layout(); plt.show()
print("red pushes toward 'overspend', blue pushes toward 'safe'; the bars sum (with the base value) to the model's log-odds for that row")


<div style="background-color: #f8f9fa; border-left: 5px solid #6c757d; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two per-household explanations. For the true positive, the bars agree: a high past overspend rate and a high spend-to-income history all push the same way, and the model is right to be confident. For the false negative, the bars also agree, and that is the interesting part: every feature said "safe", the household overspent anyway, and nothing in the row hinted at it. That is the signature of an unpredictable event (a surprise expense in the target month), not of a model bug.

**The common misread:** expecting the explanation of a wrong prediction to look wrong. SHAP explains what the model did with the inputs it had; a confident, coherent explanation of a miss tells you the *inputs* were missing something.

</div>

## 14.5 Task 14 Summary

**✅ Key Takeaways**
- `interpret_model` wraps `shap.TreeExplainer`; it works for tree models and draws the summary (beeswarm) and correlation (dependence) plots.
- SHAP values are additive: base value + contributions = the model's log-odds, verified to floating-point precision.
- Explain the transformed columns the model saw, then fold one-hot columns back to their original feature for people.
- Reason codes are the top-k SHAP contributions with a direction; a coherent explanation of a miss points at missing inputs, not a broken model.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `cls6`, `lgbm6`, `pred6`, `card6` | the Part-6 experiment, its LightGBM model, held-out predictions and report card |
| `explainer`, `sv`, `base` | the TreeExplainer, the (rows × features) SHAP matrix for the positive class, the base log-odds |
| `X6_tr`, `X6_te`, `y6_te`, `pipe6` | the transformed matrices the model saw, and the fitted pipeline |
| `mean_abs`, `reason_codes(i)` | global importance from SHAP, and the per-row reason-code builder |

**➡️ Next Up:** Task 15 asks what the model relies on when a feature is *removed* (permutation importance), draws the shape of each effect (partial dependence), and writes down what the model learned.


# Task 15 · Permutation Importance, Partial Dependence, and What the Model Learned

## 15.1 Three importances that disagree, and why

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

- **Split importance** (`plot_model(plot="feature")`): how often, or with how much gain, the trees split on a feature. Cheap, but biased toward high-cardinality numeric columns and blind to what happens after training.
- **Permutation importance**: shuffle one column of the held-out set, re-score, and measure the drop. It asks "how much does the *deployed* model need this column?"
- **Mean |SHAP|**: the average size of the feature's contribution across rows.

They rank features differently when features are correlated (the model can lean on either of two twins; permuting one costs little) and when a feature is used often but with small effects.

</div>


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

with timer("permutation importance, 10 shuffles per column, held-out set"):
    pi = permutation_importance(lgbm6, X6_te, y6_te, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=4)
perm = pd.DataFrame({"perm_drop_auc": pi.importances_mean, "perm_std": pi.importances_std}, index=X6_te.columns)
split = pd.Series(lgbm6.booster_.feature_importance(importance_type="gain"), index=lgbm6.booster_.feature_name())
imp = perm.join(split.rename("split_gain")).join(mean_abs.rename("mean_abs_shap"))
imp["rank_perm"] = imp.perm_drop_auc.rank(ascending=False).astype(int)
imp["rank_split"] = imp.split_gain.rank(ascending=False).astype(int)
imp["rank_shap"] = imp.mean_abs_shap.rank(ascending=False).astype(int)
imp = imp.sort_values("perm_drop_auc", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
top = imp.head(10)
axes[0].barh(top.index[::-1], top.perm_drop_auc[::-1], xerr=top.perm_std[::-1], color="#667eea"); axes[0].set_title("permutation importance (AUC drop, held-out)")
axes[1].barh(top.index[::-1], top.split_gain[::-1], color="#999"); axes[1].set_title("split importance (LightGBM gain)")
axes[2].barh(top.index[::-1], top.mean_abs_shap[::-1], color="#764ba2"); axes[2].set_title("mean |SHAP| (held-out)")
plt.tight_layout(); plt.show()
display(pretty(imp[["perm_drop_auc", "split_gain", "mean_abs_shap", "rank_perm", "rank_split", "rank_shap"]].head(10), fmt="{:.3f}"))
disagree = imp[(imp.rank_perm - imp.rank_split).abs() >= 3].index.tolist()
print("features whose permutation and split ranks differ by 3 or more:", disagree or "none")


<div style="background-color: #f8f9fa; border-left: 5px solid #6c757d; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Three rankings of the same model. The left panel has error bars because permutation importance is a random experiment (ten shuffles); a feature whose bar overlaps zero is one the deployed model does not need. The middle panel is what LightGBM counted during training and tends to inflate continuous columns with many split points. The right panel is SHAP's view. The features at the top agree; the disagreements listed under the plot are the correlated pairs and the often-split-but-small-effect columns.

**The common misread:** "importance zero means the feature is useless." A feature can be useless *to this model* because a correlated twin already carries the information; drop both and the score falls.

</div>

## 15.2 Partial dependence: the shape of an effect

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition

Importance says *how much*; partial dependence says *how*. Fix one feature at a value $v$ for every held-out row, predict, average, and repeat over a grid of $v$: the curve is the model's average response to that feature with everything else as it is in the data. **ICE** lines (one per row) show whether the average hides opposite behaviours. Computed by hand here on the transformed matrix, because PyCaret's `pdp` route needs the optional `interpret` package.

</div>


In [ ]:
def partial_dependence(model, X, feature, grid=25, ice_rows=60):
    # average P(overspend) as `feature` sweeps a quantile grid, everything else held at its observed values
    qs = np.quantile(X[feature], np.linspace(0.02, 0.98, grid))
    Xr = X.copy()
    pd_curve, ice = [], []
    rng = np.random.default_rng(RANDOM_STATE)
    rows = rng.choice(len(X), size=min(ice_rows, len(X)), replace=False)
    for v in qs:
        Xr[feature] = v
        p = model.predict_proba(Xr)[:, 1]
        pd_curve.append(p.mean()); ice.append(p[rows])
    return qs, np.array(pd_curve), np.array(ice)

PD_FEATURES = ["overspend_rate_prev", "income", "dining", "avg_spend_prev"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, f in zip(axes, PD_FEATURES):
    qs, curve, ice = partial_dependence(lgbm6, X6_te, f)
    ax.plot(qs, ice, color="#bbb", lw=0.6, alpha=0.6)
    ax.plot(qs, curve, color="#dc3545", lw=2.5, label="partial dependence")
    ax.set_title(f); ax.set_xlabel(f"{f} (transformed scale)"); ax.set_ylabel("P(overspend)")
axes[0].legend(loc="upper left")
plt.suptitle("partial dependence (red) with 60 ICE lines (grey), held-out households", y=1.02)
plt.tight_layout(); plt.show()
qs, curve, _ = partial_dependence(lgbm6, X6_te, "income")
print(f"income: P(overspend) falls from {curve[0]:.3f} at the 2nd percentile to {curve[-1]:.3f} at the 98th; monotone: {bool(np.all(np.diff(curve) <= 1e-9))}")


<div style="background-color: #f8f9fa; border-left: 5px solid #6c757d; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Four effect shapes. `overspend_rate_prev` rises steeply: a household that overspent in most past months is predicted to do it again. `income` falls: the same spending is a smaller share of a larger income. `dining` and `avg_spend_prev` rise, with the tree model's characteristic steps. The grey ICE lines fan out where the effect depends on other features (an interaction) and run parallel where it does not.

**The common misread:** reading a partial-dependence curve where the grid has no data. The curves are drawn between the 2nd and 98th percentiles for that reason; outside that range the model is extrapolating and the curve is fiction.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Warning: the x-axis is the transformed scale

`setup(normalize=...)` and the imputer change the columns. The curves above are on the scale the model sees; to label the axis in dollars, invert the pipeline's scaler (`pipe6` holds it) or compute partial dependence through `predict_model` on raw rows, which is slower but stays in original units.

</div>

## 15.3 What the model learned, in one table


In [ ]:
from scipy.stats import spearmanr

def fold_onehot(series):
    # sum one-hot columns back to their original feature name (city_tier_metro -> city_tier)
    out = {}
    for col, v in series.items():
        key = next((o for o in ["city_tier", "referral_code"] if col.startswith(o + "_")), col)
        out[key] = out.get(key, 0.0) + float(v)
    return pd.Series(out).sort_values(ascending=False)

learned = []
for f in fold_onehot(mean_abs).head(8).index:
    cols = [c for c in X6_te.columns if c == f or c.startswith(f + "_")]
    if len(cols) == 1 and X6_te[cols[0]].nunique() > 5:
        rho = spearmanr(X6_te[cols[0]], sv[:, X6_te.columns.get_loc(cols[0])]).correlation
        direction = "higher -> more risk" if rho > 0.2 else "higher -> less risk" if rho < -0.2 else "non-monotone"
    else:
        direction = "categorical: " + ", ".join(f"{c.split('_', 1)[1]} {'+' if sv[:, X6_te.columns.get_loc(c)].mean() > 0 else '-'}" for c in cols)
    learned.append({"feature": f, "share_of_|SHAP|": float(fold_onehot(mean_abs)[f] / mean_abs.sum()), "direction": direction,
                    "perm_drop_auc": float(sum(imp.loc[c, "perm_drop_auc"] for c in cols))})
learned = pd.DataFrame(learned).set_index("feature")

# the interaction the data generator planted: entertainment = f(dining); check the SHAP interaction strength
inter = explainer.shap_interaction_values(X6_te.iloc[:500])
inter = inter[1] if isinstance(inter, list) else inter
i_d, i_e = X6_te.columns.get_loc("dining"), X6_te.columns.get_loc("entertainment")
M = np.abs(inter).mean(axis=0); np.fill_diagonal(M, 0)
pairs = pd.Series({(X6_te.columns[i], X6_te.columns[j]): M[i, j] for i in range(M.shape[0]) for j in range(i + 1, M.shape[0])}).sort_values(ascending=False)

display(pretty(learned, fmt="{:.3f}"))
print("strongest SHAP interactions (500 held-out rows):", "; ".join(f"{a} x {b} = {v:.3f}" for (a, b), v in pairs.head(3).items()))
print(f"dining x entertainment ranks #{int((pairs > M[i_d, i_e]).sum()) + 1} of {len(pairs)} pairs")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: write the "what it learned" table before the model review

A model review that starts with the AUC ends with the AUC. One that starts with this table (each feature's share, its direction, its permutation cost, the top interactions) surfaces the surprises: a direction that contradicts domain knowledge is a leak or a data error; a feature nobody expected in the top five is worth a conversation; a planted interaction that the model found (dining × entertainment) is evidence the model is reading the data and not memorising it.

</div>

## 15.4 Task 15 Summary

**✅ Key Takeaways**
- Split, permutation and SHAP importances answer different questions; their disagreements point at correlated features and at often-used-but-weak columns.
- Permutation importance is a random experiment: report the error bars, and treat "overlaps zero" as "not needed by this model".
- Partial dependence draws the shape of an effect; ICE lines expose interactions; never read the curve outside the data range.
- The "what the model learned" table (share, direction, permutation cost, interactions) is the artefact to bring to a review.

**🔑 New variables**

| Variable | Meaning |
|---|---|
| `pi`, `perm`, `split`, `imp` | permutation-importance results, LightGBM split gain, and the merged three-way importance table |
| `partial_dependence(model, X, feature)` | the by-hand partial-dependence and ICE curves |
| `fold_onehot(series)` | sums one-hot columns back to their original feature |
| `learned`, `inter`, `pairs` | the what-it-learned table, SHAP interaction values, and the ranked interaction pairs |

**➡️ Next Up:** Part 7 leaves labels behind: clustering households by their spending shares and flagging anomalous ones with `pycaret.clustering` and `pycaret.anomaly`.


# Part 7 · Clustering and Anomaly Detection on Budget Behaviour ⭐⭐

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### No labels this time

Parts 1 to 6 always had a target column. This part throws the target away and asks two different questions of the same households: **which households spend alike** (clustering) and **which households spend strangely** (anomaly detection). PyCaret ships one module for each, `pycaret.clustering` and `pycaret.anomaly`, with the same `setup → create_model → assign_model → plot_model` rhythm you already know. What changes is the evaluation: without labels there is no accuracy, so this part teaches the internal metrics and, more importantly, their limits.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Four modules, one set of function names

`pycaret.classification`, `pycaret.regression`, `pycaret.clustering`, `pycaret.anomaly` and `pycaret.time_series` all export a function called `setup`, `create_model`, `plot_model`, `pull` and so on. A second `from pycaret.anomaly import *` silently **replaces** the clustering functions of the same name, and `pull()` then returns the table of whichever module was imported last. This part and Part 8 use the object-oriented form instead: `ClusteringExperiment()`, `AnomalyExperiment()` and `TSForecastingExperiment()` each own their state, so several experiments can live in one notebook without stepping on each other. The functional form from Parts 1 to 6 is the same code underneath; every `exp.method(...)` here is `method(...)` there.

</div>


# Task 16 · Clustering households by how they spend

## 16.1 What one row means without a label

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: shares, not amounts

Two households with incomes of 3,000 and 9,000 can have the same *shape* of spending: a third on rent, a fifth on groceries, a little on dining. Clustering on raw amounts would group by income, which is not interesting. Clustering on **shares of income** (`rent / income`, `groceries / income`, ...) groups by *behaviour*: the rent-heavy metro household, the big family with a grocery bill, the couple that eats out. Those are the profiles a budgeting product would want to find.

**Clustering** means partitioning rows into groups so that rows inside a group are more similar to each other than to rows outside it. **k-means**, the workhorse, picks $k$ centre points, assigns every row to its nearest centre, moves each centre to the mean of its rows, and repeats until nothing changes.

</div>

### 📐 The Math (gently)

k-means minimises the **inertia**, the total squared distance of every row to its own centre:

$$ J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \| x_i - \mu_j \|^2 $$

where $C_j$ is cluster $j$ and $\mu_j$ its centre. Distances are Euclidean, so a feature measured in thousands would dominate a feature measured in tenths. That is why `setup(normalize=True)` matters here: it puts every share on the same scale before the distances are computed.


In [ ]:
from pycaret.clustering import ClusteringExperiment

CATS = ["rent", "groceries", "utilities", "transport", "dining", "entertainment", "other"]
shares = hh[CATS].div(hh["income"], axis=0)
shares.columns = [f"{c}_share" for c in CATS]
SHARE_COLS = list(shares.columns)

print(f"{len(shares):,} households x {len(SHARE_COLS)} share features (feature month only; targets are not used)")
print(f"missing values: dining_share has {shares.dining_share.isna().sum()} NaN (the 4% missing dining spends); setup() imputes them")
print(shares.describe().T[["mean", "std", "min", "max"]].round(3).to_string())


## 16.2 `setup` for clustering

The clustering `setup` has no `target`. It still does the preprocessing you met in Part 2: imputation (`numeric_imputation="mean"` by default), encoding of categoricals, and, when asked, `normalize=True`, which applies z-score scaling. `verbose=False` and `html=False` keep it quiet; `session_id` fixes the random seed of k-means initialisation so the clusters are reproducible.


In [ ]:
import inspect
sig = inspect.signature(ClusteringExperiment.setup)
print("setup arguments used here are all real:", all(k in sig.parameters for k in ["data", "normalize", "session_id", "verbose", "html", "n_jobs"]))

clu = ClusteringExperiment()
with timer("clustering setup"):
    clu.setup(data=shares, normalize=True, session_id=RANDOM_STATE, verbose=False, html=False, n_jobs=4)
setup_tbl = clu.pull()
print(setup_tbl.to_string(index=False))
Xc = clu.get_config("X_transformed")
print(f"\ntransformed matrix: {Xc.shape}; column means ~0 and stds ~1 after normalize: "
      f"mean={Xc.mean().abs().max():.3f}, std={Xc.std().round(2).min():.2f}-{Xc.std().round(2).max():.2f}")


## 16.3 k-means with four clusters, and the internal metrics

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Three metrics that need no labels

- **Silhouette** (higher is better, range $-1$ to $1$): for each row, how much closer it is to its own cluster than to the nearest other cluster. Near 0 means the clusters overlap; negative means the row probably belongs elsewhere.
- **Calinski-Harabasz** (higher is better): the ratio of between-cluster spread to within-cluster spread. It rewards tight, well-separated blobs and grows with the number of rows, so compare it only across models on the same data.
- **Davies-Bouldin** (lower is better): the average, over clusters, of the worst-case ratio of within-cluster scatter to between-centre distance.

All three are **internal** metrics: they score the geometry of the partition, not whether the partition means anything. A silhouette of 0.3 on standardised shares is respectable for real behaviour data, where clusters are regions of a continuum rather than separate islands.

</div>


In [ ]:
with timer("kmeans k=4"):
    km = clu.create_model("kmeans", num_clusters=4)
km_metrics = clu.pull()
print(km_metrics.to_string())
print("\nThe estimator PyCaret fitted:", km)
clustered = clu.assign_model(km)                      # the input rows + a "Cluster" column
clustered["cluster"] = clustered["Cluster"].str.replace("Cluster ", "").astype(int)
print("\ncluster sizes:", clustered.cluster.value_counts().sort_index().to_dict())


## 16.4 How many clusters? The elbow and the silhouette curve

`num_clusters=4` was a guess. PyCaret's `plot_model(plot="elbow")` and `plot="silhouette"` answer the question, but they draw through Yellowbrick and Plotly, which need a browser to display. The cell below computes the same two curves directly with scikit-learn so that they render anywhere, and so that you see exactly what each point is: one k-means fit per candidate $k$.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

ks = list(range(2, 9))
inertia, sil = [], []
with timer("elbow + silhouette sweep (7 fits)"):
    for k in ks:
        m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xc)
        inertia.append(m.inertia_)
        sil.append(silhouette_score(Xc, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ks, inertia, marker="o", color="#667eea"); axes[0].set_title("Elbow: inertia vs k")
axes[0].set_xlabel("k (number of clusters)"); axes[0].set_ylabel("inertia (within-cluster sum of squares)")
axes[1].plot(ks, sil, marker="o", color="#28a745"); axes[1].set_title("Silhouette vs k")
axes[1].set_xlabel("k"); axes[1].set_ylabel("mean silhouette")
for ax in axes:
    ax.axvline(4, color="#dc3545", ls="--", label="k = 4 (used above)"); ax.legend()
plt.tight_layout(); plt.show()
best_k = ks[int(np.argmax(sil))]
print("inertia by k:", dict(zip(ks, np.round(inertia, 0))))
print("silhouette by k:", dict(zip(ks, np.round(sil, 3))), f"-> best silhouette at k={best_k}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: inertia always falls as $k$ grows, because more centres are always closer to the rows. The "elbow" is where the fall flattens. Right: the mean silhouette, which can go up or down; its peak is the $k$ where clusters are tightest relative to their neighbours. The printed line says which $k$ wins on silhouette. The two curves need not agree, and on behaviour data they often do not: the elbow is a judgement call and the silhouette peak is often at a small $k$ that merges profiles a product would want to keep apart.

The common misread: treating the elbow as a measurement. It is a heuristic for a picture that has no single right answer; the choice of $k$ should be settled by whether the resulting profiles are distinct and actionable (Section 16.6), not by the curve alone.

</div>


## 16.5 Seeing the clusters: a two-dimensional projection

Seven share features cannot be drawn directly. **PCA** (principal component analysis) finds the two directions along which the standardised shares vary most and projects every household onto them. PyCaret's `plot_model(plot="cluster")` does the same and draws with Plotly; the cell below does it with scikit-learn and matplotlib, and shows the per-cluster silhouette as well, which is what `plot="silhouette"` draws.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples

pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xc)
Z = pca.transform(Xc)
labels = clustered["cluster"].values
palette = ["#667eea", "#28a745", "#ffc107", "#dc3545", "#9c27b0", "#17a2b8"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for c in sorted(np.unique(labels)):
    m = labels == c
    axes[0].scatter(Z[m, 0], Z[m, 1], s=10, alpha=0.6, color=palette[c], label=f"cluster {c} (n={m.sum()})")
axes[0].set_title("k-means clusters in PCA space (what plot='cluster' draws)")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)"); axes[0].legend(markerscale=2)

s_vals = silhouette_samples(Xc, labels); y0 = 0
for c in sorted(np.unique(labels)):
    sv = np.sort(s_vals[labels == c])
    axes[1].fill_betweenx(np.arange(y0, y0 + len(sv)), 0, sv, color=palette[c], alpha=0.8, label=f"cluster {c}")
    y0 += len(sv) + 20
axes[1].axvline(s_vals.mean(), color="k", ls="--", label=f"mean = {s_vals.mean():.3f}")
axes[1].set_title("Silhouette per household, by cluster (what plot='silhouette' draws)")
axes[1].set_xlabel("silhouette value"); axes[1].set_ylabel("households (stacked by cluster)"); axes[1].legend()
plt.tight_layout(); plt.show()
print("loadings of PC1 (which shares it mixes):", dict(zip(SHARE_COLS, np.round(pca.components_[0], 2))))
print("loadings of PC2:", dict(zip(SHARE_COLS, np.round(pca.components_[1], 2))))
print(f"households with a negative silhouette (probably misassigned): {(s_vals < 0).sum()} of {len(s_vals)}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: each dot is a household placed by its first two principal components; the loadings printed under the plot say which shares each axis mixes (a large positive loading on `rent_share` means moving right is "more of income goes to rent"). The clusters are contiguous regions, not separate islands: k-means cuts a continuous cloud into pieces. Right: every household's silhouette, sorted within its cluster. Wide, tall shapes are good clusters; a cluster whose shape dips below zero contains households that sit closer to a neighbouring centre.

The common misread: judging the clustering by how separated the blobs look in PCA space. PCA keeps only the two directions of largest variance; two clusters that overlap here can be perfectly separated along a share the picture threw away, and the reverse.

</div>


## 16.6 Profiling the clusters: what makes each one different

Internal metrics say whether the partition is tight. Only a **profile**, the mean of every feature per cluster, says whether it is useful. The overspend rate per cluster is the honest check: the label was never shown to the model, so if clusters differ in overspending, the behaviour they capture is real.


In [ ]:
profile = clustered.groupby("cluster")[SHARE_COLS].mean()
profile["n"] = clustered.cluster.value_counts().sort_index()
profile["overspend_rate_next"] = hh[LABEL_CLS].groupby(clustered.cluster).mean()
profile["income_mean"] = hh["income"].groupby(clustered.cluster).mean()
profile["metro_share"] = (hh["city_tier"] == "metro").groupby(clustered.cluster).mean()
print("cluster profile (rows are clusters):")
print(profile.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
(profile[SHARE_COLS].T).plot.bar(ax=axes[0], color=palette[:4], width=0.8)
axes[0].set_title("Mean share of income per category, by cluster"); axes[0].set_ylabel("share of income")
axes[0].set_xticklabels([c.replace("_share", "") for c in SHARE_COLS], rotation=0); axes[0].legend(title="cluster")
axes[1].bar(profile.index.astype(str), profile.overspend_rate_next, color=palette[:4])
axes[1].axhline(hh[LABEL_CLS].mean(), color="k", ls="--", label=f"overall rate {hh[LABEL_CLS].mean():.1%}")
axes[1].set_title("Overspend rate next month, by cluster (label never seen by k-means)")
axes[1].set_xlabel("cluster"); axes[1].set_ylabel("rate"); axes[1].legend()
plt.tight_layout(); plt.show()
hi, lo = profile.overspend_rate_next.idxmax(), profile.overspend_rate_next.idxmin()
print(f"overspend rate ranges from {profile.overspend_rate_next[lo]:.1%} (cluster {lo}) to {profile.overspend_rate_next[hi]:.1%} (cluster {hi})")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: the fingerprint of each cluster. Read one colour across the categories: a cluster whose bar is tallest on rent and shortest on dining is the "rent-heavy" profile; a cluster tall on groceries and utilities is the "large household" profile; the one tall on dining, entertainment and other is the discretionary spender. Right: the fraction of each cluster that overspends next month, against the overall rate as a dashed line. The spread between the best and worst cluster (printed) is the evidence that these clusters carry information about the future, even though no label was used to form them.

The common misread: reading the cluster with the highest overspend rate as "the risky households". Clusters are coarse; Part 3's classifier separates risk far better than any cluster membership. Clusters are for understanding and segmentation, not for ranking individuals.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: validate clusters with a variable the model never saw

Internal metrics cannot tell a meaningful partition from a meaningless one; the same silhouette is possible on pure noise. Always hold back at least one variable (here `overspent_next`, `income`, `city_tier`) and check that it varies across clusters. If nothing you care about differs between the groups, the clustering is not wrong, it is useless.

</div>


## 16.7 Other algorithms: hierarchical and DBSCAN

k-means assumes round clusters of similar size. **Hierarchical clustering** (`hclust`) merges the two closest groups repeatedly and cuts the resulting tree at $k$ groups; it can find elongated shapes. **DBSCAN** grows clusters from dense regions and labels sparse rows as noise (`-1`), so it needs no $k$ but does need a neighbourhood radius `eps` and `min_samples`. PyCaret exposes both through `create_model` with the same two lines. The comparison table below shows why the internal metrics must be read with the number of clusters in view.


In [ ]:
rows = {"kmeans (k=4)": km_metrics.iloc[0]}
with timer("hclust + dbscan"):
    hc = clu.create_model("hclust", num_clusters=4)
    rows["hclust (k=4)"] = clu.pull().iloc[0]
    db = clu.create_model("dbscan", eps=0.9, min_samples=15)
    rows["dbscan (eps=0.9)"] = clu.pull().iloc[0]
db_labels = clu.assign_model(db)["Cluster"]
n_db = db_labels.nunique() - int((db_labels == "Cluster -1").any())
noise = int((db_labels == "Cluster -1").sum())

cmp_tbl = pd.DataFrame(rows).T[["Silhouette", "Calinski-Harabasz", "Davies-Bouldin"]].astype(float)
cmp_tbl["n_clusters"] = [4, 4, n_db]
cmp_tbl["noise_rows"] = [0, 0, noise]
display(pretty(cmp_tbl, subset=["Silhouette", "Calinski-Harabasz"]))
print(cmp_tbl.round(3).to_string())
print(f"\nDBSCAN found {n_db} clusters and marked {noise} households ({noise / len(db_labels):.1%}) as noise (Cluster -1)")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Internal metrics reward geometry, not meaning

Look at the DBSCAN row. Its silhouette is computed only over the rows it kept, after the awkward ones were labelled noise; a method that discards the hard cases can look better on the very metric that ignores them. And a Calinski-Harabasz score is not comparable across a different number of clusters. When the table shows three algorithms with different $k$ and different noise handling, the only fair comparison is the profile check of Section 16.6, repeated per method.

The parameters `eps=0.9, min_samples=15` were chosen by hand for the standardised seven-dimensional shares; DBSCAN's defaults (`eps=0.5, min_samples=5`) come from two-dimensional toy data and will usually mark most of a real table as noise. Sweep `eps` the way Section 16.4 swept $k$.

</div>

## 16.8 Task 16 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- Cluster on **shares of income**, not amounts, when the question is behaviour; `normalize=True` so that every share counts equally in the distance.
- `ClusteringExperiment` gives `setup → create_model → assign_model → pull` with three internal metrics: silhouette, Calinski-Harabasz, Davies-Bouldin.
- The elbow and silhouette curves narrow the choice of $k$; the **profile** and a held-back variable decide it.
- `hclust` and `dbscan` are one-line swaps; their metrics are not directly comparable to k-means when $k$ or the noise handling differ.
- Use the OOP experiment classes when more than one PyCaret module lives in one notebook.

### 🔑 New variables

| Variable | What it holds |
|---|---|
| `shares`, `SHARE_COLS` | the seven category-share features, one row per household |
| `clu` | the `ClusteringExperiment` (its transformed matrix is `Xc`) |
| `km`, `km_metrics`, `clustered` | the fitted k-means, its metric row, and the rows with a `cluster` column |
| `profile` | mean shares, size, overspend rate, income and metro share per cluster |
| `cmp_tbl` | the k-means / hclust / DBSCAN comparison |

### ➡️ Next Up

Task 17 keeps the same features, adds `total_spend / income`, and asks the opposite question: which households do not belong to any group.

</div>


# Task 17 · Anomaly detection: the households that spend strangely

## 17.1 Outliers as a question, not a nuisance

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: how easy is a row to isolate?

An **anomaly** is a row that a simple description of the data cannot account for. The **Isolation Forest** (`iforest`) turns that into a game: pick a random feature, pick a random split value, and repeat until the row sits alone. Ordinary rows, deep inside the crowd, take many splits to isolate; strange rows take few. The average number of splits, across many random trees, becomes the anomaly score. Two other detectors take other routes: the **local outlier factor** (`lof`) compares a row's local density with that of its neighbours, and the **k-nearest-neighbours** detector (`knn`) uses the distance to the $k$-th neighbour directly.

The features are the seven shares from Task 16 plus one more, `spend_ratio = total_spend / income`, so that a household that spends more than it earns can be caught even when its shape of spending is ordinary.

</div>

### 📐 The Math (gently)

For a row $x$ the Isolation Forest score is

$$ s(x) = 2^{-\, \mathbb{E}[h(x)] \,/\, c(n)} $$

where $h(x)$ is the number of splits needed to isolate $x$ in one tree, $\mathbb{E}[h(x)]$ is its average over the trees, and $c(n)$ is the average path length in a random tree of $n$ rows, a normalising constant. A score near 1 means "isolated quickly, anomalous"; near 0.5 means "ordinary". The `fraction` argument (`contamination` in scikit-learn) sets the score cut-off so that exactly that share of rows is flagged.


In [ ]:
from pycaret.anomaly import AnomalyExperiment

anom_X = shares.copy()
anom_X["spend_ratio"] = hh["total_spend"] / hh["income"]
ANOM_COLS = list(anom_X.columns)

sig = inspect.signature(AnomalyExperiment.setup)
print("setup arguments are real:", all(k in sig.parameters for k in ["data", "normalize", "session_id", "verbose", "html"]))
ano = AnomalyExperiment()
with timer("anomaly setup"):
    ano.setup(data=anom_X, normalize=True, session_id=RANDOM_STATE, verbose=False, html=False, n_jobs=4)
print(ano.pull().to_string(index=False))
print(f"\n{len(anom_X):,} households x {len(ANOM_COLS)} features")


In [ ]:
FRACTION = 0.05
with timer("iforest"):
    iso = ano.create_model("iforest", fraction=FRACTION)
iso_rows = ano.assign_model(iso)                       # adds Anomaly (0/1) and Anomaly_Score
print("columns added by assign_model:", [c for c in iso_rows.columns if c not in ANOM_COLS])
print(f"flagged: {iso_rows.Anomaly.sum()} households = {iso_rows.Anomaly.mean():.1%} (fraction={FRACTION} by construction)")
print("\nthe five highest scores:")
print(iso_rows.sort_values("Anomaly_Score", ascending=False).head(5).round(3).to_string())
print("\nunderlying estimator:", iso)


## 17.2 Seeing the flagged households

PyCaret's `plot_model(plot="tsne")` embeds the rows in two dimensions with **t-SNE** and colours the flagged ones; it draws with Plotly. The cell below runs the same t-SNE from scikit-learn (a non-linear embedding that keeps close neighbours close, so dense clumps stay clumps) and adds a second panel with the score distribution and the cut-off, which is the honest way to see what `fraction` did.


In [ ]:
from sklearn.manifold import TSNE

Xa = ano.get_config("X_transformed")
tsne_kw = {"n_components": 2, "perplexity": 30, "init": "pca", "random_state": RANDOM_STATE}
tsne_kw["max_iter" if "max_iter" in inspect.signature(TSNE).parameters else "n_iter"] = 500
with timer("t-SNE on 1,500 rows"):
    E = TSNE(**tsne_kw).fit_transform(Xa.values)

flag = iso_rows.Anomaly.values == 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(E[~flag, 0], E[~flag, 1], s=8, alpha=0.4, color="#667eea", label="normal")
axes[0].scatter(E[flag, 0], E[flag, 1], s=22, color="#dc3545", label=f"flagged (top {FRACTION:.0%})")
axes[0].set_title("t-SNE embedding of spending shares (what plot='tsne' draws)")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2"); axes[0].legend()
cut = iso_rows.loc[flag, "Anomaly_Score"].min()
axes[1].hist(iso_rows.Anomaly_Score, bins=50, color="#667eea", alpha=0.8, label="all households")
axes[1].axvline(cut, color="#dc3545", ls="--", label=f"cut-off = {cut:.3f} (from fraction={FRACTION})")
axes[1].set_title("Isolation Forest scores"); axes[1].set_xlabel("Anomaly_Score (higher = more isolated)"); axes[1].set_ylabel("households"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"score cut-off {cut:.3f}; scores range {iso_rows.Anomaly_Score.min():.3f} to {iso_rows.Anomaly_Score.max():.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: the red households sit on the fringe of the cloud and in its thin outer arms, which is where a random-split isolation finishes quickly. Right: the score distribution is a single hump with a right tail; the dashed line is the value that happens to cut off the top 5%. Nothing in the histogram singles out that line: there is no gap, no second hump. The cut-off is a policy decision expressed as a number.

The common misread: reading the t-SNE picture as a map with distances. t-SNE preserves neighbourhoods, not distances; the size of a clump and the gap between clumps mean little. Use it to see *whether* the flagged rows are on the fringe, not *how far* out they are.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `fraction` is an assumption, not a measurement

`fraction=0.05` did not discover that 5% of households are anomalous; it *decided* it. The histogram above shows why that matters: the scores fall off smoothly, so `fraction=0.02` or `0.10` would flag a different set with equal confidence. When you have any external signal about how common anomalies are (the surprise-expense rate in Section 17.4, a fraud rate, a known defect rate) set `fraction` from it and report that you did. When you have none, report flagged households ranked by `Anomaly_Score` and let the reader choose the cut.

</div>


## 17.3 Do three detectors agree?

If `iforest`, `lof` and `knn` flag the same households, the anomalies are robust to the definition of "strange". If they disagree, each detector's flags say as much about the algorithm as about the data. The **Jaccard index** of two flagged sets, $|A \cap B| \,/\, |A \cup B|$, measures overlap: 1 is identical, 0 is disjoint. The overspend rate of the flagged rows, against the overall rate, is the external check from Task 16 applied again.


In [ ]:
detectors = {"iforest": iso_rows}
with timer("lof + knn"):
    for name in ["lof", "knn"]:
        mdl = ano.create_model(name, fraction=FRACTION)
        detectors[name] = ano.assign_model(mdl)
flags = {k: set(np.flatnonzero(v.Anomaly.values == 1)) for k, v in detectors.items()}

names = list(flags)
jac = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        jac.loc[a, b] = len(flags[a] & flags[b]) / len(flags[a] | flags[b])
print("Jaccard overlap of the flagged sets:")
print(jac.round(2).to_string())

overall = hh[LABEL_CLS].mean()
rate_tbl = pd.DataFrame({
    "n_flagged": [len(flags[n]) for n in names],
    "overspend_rate_flagged": [hh[LABEL_CLS].iloc[sorted(flags[n])].mean() for n in names],
    "overspend_rate_not_flagged": [hh[LABEL_CLS].drop(index=sorted(flags[n])).mean() for n in names],
    "mean_spend_ratio_flagged": [anom_X.spend_ratio.iloc[sorted(flags[n])].mean() for n in names],
}, index=names)
rate_tbl["lift_vs_overall"] = rate_tbl.overspend_rate_flagged / overall
display(pretty(rate_tbl, subset=["overspend_rate_flagged", "lift_vs_overall"]))
print(rate_tbl.round(3).to_string())
all3 = flags["iforest"] & flags["lof"] & flags["knn"]
print(f"\nflagged by all three: {len(all3)} households; overall overspend rate {overall:.1%}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

The Jaccard table is the agreement between detectors at the same `fraction`. `iforest` and `knn` both look for rows far from the bulk and overlap the most; `lof` looks for rows that are sparse *relative to their neighbourhood*, which can be near the centre of the cloud, so it agrees least with the other two. The rate table is the external check: households flagged by a detector that keys on `spend_ratio` overspend far more often than the rest (the `lift_vs_overall` column), while a detector that spreads its attention across all eight features lifts less. None of the detectors saw the label.

The common misread: picking the detector with the largest lift as "the best anomaly detector". It is the best at finding *overspenders*, which is one kind of strange. The households only `lof` flags may be the more interesting ones for a different question, such as a data-entry error.

</div>


## 17.4 A ground-truth-ish check: the surprise-expense months

The budget generator planted surprise expenses: in a random 6% of household-months, `other` jumps to between 300 and 1,200, far above its usual level. That flag is not stored in `hh`, but `other >= 300` in the feature month recovers it almost exactly (ordinary `other` spends rarely reach 300). It is not the anomaly a product cares about, and the detectors were never told about it, so it works as a *partial* ground truth: a good detector should rank those households near the top. **Precision at $k$** is the fraction of the $k$ highest-scored households that carry the surprise flag; it is the metric to use when only the top of a ranking will ever be acted on.


In [ ]:
surprise = (hh["other"] >= 300).astype(int).values
print(f"households with a surprise expense in the feature month: {surprise.sum()} ({surprise.mean():.1%})")

def precision_at_k(score, truth, k):
    top = np.argsort(-np.asarray(score))[:k]
    return truth[top].mean()

ks_eval = [25, 50, 75, 100]
pak = pd.DataFrame({name: [precision_at_k(det.Anomaly_Score.values, surprise, k) for k in ks_eval]
                    for name, det in detectors.items()}, index=[f"P@{k}" for k in ks_eval])
pak["random"] = surprise.mean()
display(pretty(pak, subset=list(detectors)))
print(pak.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
for name, det in detectors.items():
    order = np.argsort(-det.Anomaly_Score.values)
    ax.plot(np.arange(1, 201), np.cumsum(surprise[order][:200]) / np.arange(1, 201), label=name)
ax.axhline(surprise.mean(), color="k", ls="--", label=f"random ({surprise.mean():.1%})")
ax.set_title("Precision at k against the surprise-expense flag"); ax.set_xlabel("k (top-scored households)"); ax.set_ylabel("precision@k"); ax.legend()
plt.tight_layout(); plt.show()
best_det = pak.loc["P@75", list(detectors)].idxmax()
print(f"best precision@75: {best_det} = {pak.loc['P@75', best_det]:.2f} vs random {surprise.mean():.2f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Each curve is the fraction of surprise-expense households among the top $k$ scores as $k$ grows; the dashed line is what a random ranking would give. All three detectors sit well above random at small $k$, which is evidence that the scores are not noise. The curves fall as $k$ grows because the surprise flag covers only one kind of anomaly; a household flagged for an extreme rent share is a true anomaly and a "miss" on this particular check.

The common misread: calling the detectors wrong because precision@100 is far below 1. The surprise flag is a *partial* truth; the right conclusion is "the detectors rank the known anomalies high, and also flag things the check does not cover", which is exactly what unsupervised detection is for.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: report a ranking and a partial truth, not a flag

An anomaly detector's output that survives contact with a product is a **ranked list** plus **precision at the $k$ you can afford to review**, measured on whatever partial ground truth exists (past investigations, planted cases, a known rare event). A binary `Anomaly` column with `fraction` chosen by feel hides both the ranking and the assumption. Keep `Anomaly_Score`, keep the `fraction` in the report, and re-measure precision@k when the reviewers send labels back.

</div>

## 17.5 Task 17 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- `AnomalyExperiment`: `setup → create_model("iforest", fraction=...) → assign_model` gives `Anomaly` (0/1) and `Anomaly_Score`.
- `fraction` fixes the share of rows flagged; it is a decision, and the smooth score histogram shows there is no natural cut.
- `iforest`, `lof` and `knn` flag overlapping but different sets (the Jaccard table); disagreement is information about what "strange" means.
- Validate with what you have: the overspend rate of flagged rows, and precision at $k$ against a partial ground truth such as the planted surprise expenses.
- t-SNE shows whether flagged rows sit on the fringe; it does not measure how far.

### 🔑 New variables

| Variable | What it holds |
|---|---|
| `anom_X`, `ANOM_COLS` | the seven shares plus `spend_ratio` |
| `ano`, `iso`, `iso_rows` | the `AnomalyExperiment`, the Isolation Forest, and the rows with `Anomaly` / `Anomaly_Score` |
| `detectors`, `flags`, `jac` | per-detector assigned rows, flagged index sets, and their Jaccard overlap |
| `rate_tbl`, `pak` | overspend rates of flagged rows; precision at k against the surprise flag |
| `surprise` | the partial ground truth: `other >= 300` in the feature month |

### ➡️ Next Up

Part 8 changes the unit of analysis from a household to a **month** and forecasts spending ahead with `pycaret.time_series`.

</div>


# Part 8 · Time Series Forecasting with `pycaret.time_series` ⭐⭐⭐

<div style="background: linear-gradient(135deg, #fc466b 0%, #3f5efb 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### Same rhythm, different rules

`pycaret.time_series` keeps the `setup → compare_models → create_model → predict_model` lifecycle, but everything underneath obeys time: the train/test split is a cut at a date, not a shuffle; cross-validation is a set of expanding windows, not folds; and the baselines to beat are not "predict the mean" but "predict last month" and "predict the same month last year". The module is built on `sktime`, so the models are `sktime` forecasters and the metrics are the forecasting family: MASE, RMSSE, MAPE, SMAPE.

The data is thin on purpose. Twenty-four monthly points with a yearly season is the shape of most real business series when someone first asks for a forecast, and the honest lessons of this part come from that thinness.

</div>


# Task 18 · One aggregate series: the mean monthly spend

## 18.1 From 1,500 series to one, and the baselines that matter

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: forecasting is extrapolation with a memory

A forecast model looks at the past of one series and extends it. The simplest, the **naive** forecast, repeats the last value. The **seasonal naive** repeats the value from one season ago (twelve months here). Everything else, exponential smoothing (`ets`, `exp_smooth`), ARIMA, Theta, is a way to blend the recent level, the trend and the seasonal pattern with weights learned from the past. If a model cannot beat the two naive forecasts, it has learned nothing that the calendar did not already know.

The **forecast horizon** `fh` is how many steps ahead to predict. With `fh=3` PyCaret holds out the last three months as the test window, and cross-validates on expanding windows that each end three months before the next.

</div>

### 📐 The Math (gently): MASE

The **mean absolute scaled error** divides the forecast's mean absolute error by the in-sample mean absolute error of the one-step naive forecast:

$$ \text{MASE} = \frac{\frac{1}{h}\sum_{t=1}^{h} |y_{T+t} - \hat{y}_{T+t}|}{\frac{1}{T-1}\sum_{t=2}^{T} |y_t - y_{t-1}|} $$

MASE below 1 means the model beats "repeat last month" on average; MASE above 1 means it does worse. It is unit-free, so it compares series measured in different currencies, and it is the first number to read in every table of this part. RMSSE is the same idea with squared errors; MAPE and SMAPE are percentage errors, which are easy to read and easy to distort when the series passes near zero.


In [ ]:
from pycaret.time_series import TSForecastingExperiment

y_all = ts.groupby("timestamp")["target"].mean().sort_index()
y_all.index = pd.PeriodIndex(y_all.index, freq="M")        # a monthly PeriodIndex is what pycaret.time_series prefers
y_all.name = "mean_spend"
print(f"{len(y_all)} monthly points, {y_all.index[0]} to {y_all.index[-1]}")
print(y_all.round(0).to_string())

naive_scale = y_all.diff().abs().mean()
print(f"\nin-sample naive MAE (the MASE denominator): {naive_scale:.1f}")
print(f"December vs the November before it: {y_all.iloc[11] / y_all.iloc[10] - 1:+.1%} and {y_all.iloc[23] / y_all.iloc[22] - 1:+.1%}  (the planted holiday bump)")


## 18.2 `setup` and `check_stats`

`setup` for time series takes the series, the horizon `fh`, the number of expanding-window folds `fold`, and a seed. It detects the frequency and tests for seasonality, and it reports the seasonal period it will pass to seasonal models. `check_stats()` runs the summary statistics and stationarity tests you would otherwise assemble by hand.


In [ ]:
sig = inspect.signature(TSForecastingExperiment.setup)
print("setup arguments are real:", all(k in sig.parameters for k in ["data", "fh", "fold", "session_id", "verbose", "html", "n_jobs"]))

tse = TSForecastingExperiment()
with timer("time-series setup"):
    tse.setup(data=y_all, fh=3, fold=2, session_id=RANDOM_STATE, verbose=False, html=False, n_jobs=1)
setup_ts = tse.pull()
print(setup_ts.to_string(index=False))
print(f"\ntrain: {len(tse.get_config('y_train'))} points, test: {len(tse.get_config('y_test'))} points, seasonal period used: {tse.get_config('primary_sp_to_use')}")


In [ ]:
stats = tse.check_stats()
print(stats.to_string(index=False))


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Twenty-four points and a twelve-month season

The setup table reports the seasonal period the tests chose. A yearly pattern needs at least two full cycles to be *estimated* and more to be *trusted*; after `fh=3` is held out, the models train on twenty-one points, less than two Decembers. Seasonal models are therefore fitting a pattern they have seen once and a half times. Read every seasonal-model result in this part as "plausible", not "established", and read the stationarity tests above the same way: with this few points they have little power, and a "not stationary" verdict is as likely to come from the trend as from anything deeper.

</div>

## 18.3 `compare_models` on a small set of classic forecasters

The `include=` list keeps the run under a minute: the two naive baselines, two exponential-smoothing variants, ARIMA and Theta. `models()` lists everything the module knows; the cell checks the list against it first, so the notebook still runs if an optional dependency is missing.


In [ ]:
wanted = ["naive", "snaive", "exp_smooth", "ets", "arima", "theta"]
available = tse.models().index.tolist()
include = [m for m in wanted if m in available]
print("available in this install:", include, "| skipped:", [m for m in wanted if m not in available])

with timer("compare_models (6 forecasters, 2 expanding folds)"):
    best_ts = tse.compare_models(include=include, fold=2, sort="MASE")
cmp_ts = tse.pull()
display(pretty(cmp_ts.drop(columns=["Model"]), highlight="min", subset=["MASE", "RMSSE", "MAE", "RMSE", "MAPE", "SMAPE"]))
print(cmp_ts.to_string())
print("\nbest by MASE:", best_ts)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 Reading the comparison table

| Column | What it is | Read it as |
|---|---|---|
| **MASE** | MAE scaled by the in-sample one-step naive MAE | below 1 beats "repeat last month" |
| **RMSSE** | RMSE scaled the same way | like MASE, but large misses weigh more |
| **MAE**, **RMSE** | absolute errors in the series' own units (currency here) | for the business reader |
| **MAPE**, **SMAPE** | percentage errors; SMAPE symmetrises over- and under-forecasts | easy to read, unstable near zero |
| **R2** | one minus error variance over series variance, on the test window | often negative on short windows; do not rank by it |
| **TT (Sec)** | fit time | the price of the model |

Every number is the mean over the `fold=2` expanding windows of the *training* part, exactly as `compare_models` in Part 3 reported cross-validation, not test, scores. The `naive` and `snaive` rows are the bars to clear; a model below both on MASE has found something in the series that the calendar alone did not give.

</div>


## 18.4 The forecast, drawn

`predict_model(best, fh=3)` without new data forecasts the held-out test window, so the forecast can be laid over the actual values. `plot_model(plot="forecast")` draws this with Plotly; the cell below draws it with matplotlib from the same numbers and shows the two naive baselines beside the winner, which is the comparison that matters.


In [ ]:
y_train, y_test = tse.get_config("y_train"), tse.get_config("y_test")
preds = {}
for name in ["naive", "snaive"]:
    if name in include:
        m = tse.create_model(name, verbose=False)
        preds[name] = tse.predict_model(m, fh=3, verbose=False)["y_pred"]
preds["best: " + cmp_ts.iloc[0]["Model"]] = tse.predict_model(best_ts, fh=3, verbose=False)["y_pred"]
test_metrics = tse.pull()
print("test-window metrics of the best model (from pull() after predict_model):")
print(test_metrics.to_string(index=False))

def _t(idx):
    return idx.to_timestamp() if isinstance(idx, pd.PeriodIndex) else idx

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(_t(y_train.index), y_train.values, marker="o", color="#333", label="train (21 months)")
ax.plot(_t(y_test.index), y_test.values, marker="o", color="#28a745", lw=2, label="actual (held-out 3 months)")
for (name, p), col in zip(preds.items(), ["#999", "#ffc107", "#dc3545"]):
    ax.plot(_t(p.index), p.values, marker="s", ls="--", color=col, label=f"forecast: {name}")
ax.axvline(_t(y_test.index)[0] - pd.Timedelta(days=15), color="k", ls=":", label="train / test cut")
ax.set_title("Mean monthly spend: history, held-out actuals and three forecasts (what plot='forecast' draws)")
ax.set_xlabel("month"); ax.set_ylabel("mean total spend per household"); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()
mae = {k: float(np.mean(np.abs(v.values - y_test.values))) for k, v in preds.items()}
print("test-window MAE:", {k: round(v, 1) for k, v in mae.items()}, "| MASE = MAE /", round(float(y_train.diff().abs().mean()), 1))


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

The black line is the twenty-one training months, the green line the three held-out months, and the dashed lines are three forecasts made from the cut. The `naive` forecast is flat at the last training value. The `snaive` forecast copies the same three months of the previous year, which is why it alone can anticipate a December bump. The best model by cross-validation is the third dashed line; the printed MAE values say how each did on this one window.

The common misread: judging a forecaster on a single three-month window. Three points is not a test set; it is one draw. The cross-validation table of Section 18.3 averages two windows and is already thin. On a series this short, the honest statement is "the winner by CV was X, and on the final window it did Y", never "X is the better model".

</div>


## 18.5 Diagnostics and decomposition, by hand

`plot_model(plot="diagnostics")` shows the series, its autocorrelation (ACF), partial autocorrelation (PACF), a histogram and a Q-Q plot; `plot="decomp"` splits the series into trend, seasonal and residual parts. Both draw with Plotly in PyCaret; the cell below builds the same panels from `statsmodels`, which `sktime` already depends on. The **ACF** at lag $k$ is the correlation between $y_t$ and $y_{t-k}$; a spike at lag 12 is the signature of yearly seasonality. The **PACF** removes the effect of the shorter lags in between.


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as sp_stats

y_ts = y_all.copy(); y_ts.index = y_ts.index.to_timestamp()
dec = seasonal_decompose(y_ts, model="additive", period=12)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes[0, 0].plot(y_ts, marker="o", color="#333"); axes[0, 0].set_title("series"); axes[0, 0].set_ylabel("mean spend")
plot_acf(y_ts, lags=12, ax=axes[0, 1]); axes[0, 1].set_title("ACF (lag 12 = last year's same month)")
plot_pacf(y_ts, lags=11, ax=axes[0, 2], method="ywm"); axes[0, 2].set_title("PACF")
axes[1, 0].plot(dec.trend, color="#667eea"); axes[1, 0].set_title("decomposition: trend (12-month moving average)")
axes[1, 1].plot(dec.seasonal, color="#28a745"); axes[1, 1].set_title("decomposition: seasonal (period 12)")
axes[1, 2].plot(dec.resid, marker="o", ls="", color="#dc3545"); axes[1, 2].axhline(0, color="k", lw=0.8); axes[1, 2].set_title("decomposition: residual")
for ax in axes.ravel():
    ax.set_xlabel("month" if ax not in (axes[0, 1], axes[0, 2]) else "lag")
plt.tight_layout(); plt.show()

from statsmodels.tsa.stattools import acf
a = acf(y_ts, nlags=12)
print(f"ACF at lag 1: {a[1]:.2f}, lag 12: {a[12]:.2f}; seasonal amplitude (max - min of the seasonal part): {dec.seasonal.max() - dec.seasonal.min():.0f}")
print(f"trend from first to last defined point: {dec.trend.dropna().iloc[0]:.0f} -> {dec.trend.dropna().iloc[-1]:.0f}")
print(f"residuals: {dec.resid.dropna().shape[0]} points (the moving average loses 6 at each end); Shapiro p = {sp_stats.shapiro(dec.resid.dropna()).pvalue:.2f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Top row: the series, its ACF and PACF. The ACF at lag 1 is high because months are close to their neighbours (a slow, smooth season plus a drift); the value at lag 12 is what a seasonal model relies on, and with only twelve pairs of points to estimate it from, its confidence band is wide. Bottom row: the classical decomposition. The trend is a twelve-month moving average, so it is undefined for the first and last six months, which is exactly the part a forecaster needs most. The seasonal part is the *average* December bump and summer dip across the two years, and the residual is what is left.

The common misread: taking the decomposition's seasonal component as the truth of the series. It is an average over the cycles available, here fewer than two, and every model that uses it inherits that uncertainty.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: always put `naive` and `snaive` in the `include=` list

They cost nothing and they are the scale of every other number: MASE is defined against the naive forecast, and a seasonal series is often forecast surprisingly well by `snaive` alone. A comparison table without them can make a mediocre model look good. When a sophisticated model does not beat `snaive` by a clear margin across folds, ship `snaive`; it is easier to explain and cannot overfit.

</div>

## 18.6 Task 18 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- `TSForecastingExperiment().setup(data=y, fh=3, fold=2)` cuts the last `fh` points as test and cross-validates on expanding windows; `check_stats()` gives the summary and stationarity tests.
- **MASE** is the first column to read: below 1 beats "repeat last month".
- `naive` and `snaive` belong in every `compare_models` call; they are the bars to clear.
- Twenty-four points with a yearly season means seasonal models learn from fewer than two cycles; results are plausible, not established.
- `predict_model(best, fh=3)` on the held-out window gives the forecast to plot and, through `pull()`, the test metrics; the diagnostics and decomposition come from `statsmodels` in two lines each.

### 🔑 New variables

| Variable | What it holds |
|---|---|
| `y_all` | the aggregate series: mean monthly spend per household, monthly `PeriodIndex` |
| `tse`, `setup_ts`, `stats` | the `TSForecastingExperiment`, its setup table and `check_stats()` output |
| `include`, `best_ts`, `cmp_ts` | the forecasters that were available, the winner by MASE, the comparison table |
| `y_train`, `y_test`, `preds`, `test_metrics` | the split, the three forecasts on the test window, the best model's test metrics |
| `dec` | the classical decomposition |

### ➡️ Next Up

Task 19 forecasts individual households, first one series at a time, then all 1,500 at once with a machine-learning model on lagged features.

</div>


# Task 19 · Many households: per-series models vs one panel model

## 19.1 Three households, one `ets` each, with prediction intervals

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: a point forecast without an interval is a guess in disguise

A single household's monthly spend is far noisier than the average over 1,500. A forecast for it must say not only "about 2,400" but "between 2,000 and 2,800 with 90% confidence". Exponential smoothing (`ets`) provides that: the **prediction interval** widens with the horizon because every extra month adds another month of unknown noise. `predict_model(..., return_pred_int=True, coverage=0.9)` returns the interval as `lower` and `upper` columns; `coverage` is its nominal probability (`alpha` is a different argument: it asks for one quantile as the point forecast).

The loop below runs one full `setup → create_model("ets") → predict_model` per household. On three households that is fine. On 1,500 it would take a quarter of an hour and give 1,500 models to keep, which is the problem Section 19.2 solves.

</div>


In [ ]:
wide = ts.pivot(index="item_id", columns="timestamp", values="target").sort_index(axis=1)   # 1,500 households x 24 months
wide.columns = pd.PeriodIndex(wide.columns, freq="M")
wide.index.name = ID_COL
print(f"wide table: {wide.shape[0]:,} households x {wide.shape[1]} months")

pm_sig = inspect.signature(TSForecastingExperiment.predict_model).parameters
HAS_PI = "return_pred_int" in pm_sig
print("predict_model supports return_pred_int:", HAS_PI, "| alpha:", "alpha" in pm_sig, "| coverage:", "coverage" in pm_sig)

SAMPLE_HH = [3, 250, 1200]
hh_runs = {}
with timer("3 households x (setup + ets + predict)"):
    for hid in SAMPLE_HH:
        y_h = wide.loc[hid].copy(); y_h.name = f"household_{hid}"
        e = TSForecastingExperiment()
        e.setup(data=y_h, fh=3, fold=2, session_id=RANDOM_STATE, verbose=False, html=False, n_jobs=1)
        ets_h = e.create_model("ets", verbose=False)
        cv_h = e.pull()
        try:
            fc = e.predict_model(ets_h, fh=3, return_pred_int=True, coverage=0.9, verbose=False)   # 90% interval
            pi_note = "interval from predict_model(return_pred_int=True)"
        except Exception as err:
            fc = e.predict_model(ets_h, fh=3, verbose=False)
            resid_sd = e.get_config("y_train").diff().std()
            fc["lower"], fc["upper"] = fc["y_pred"] - 1.64 * resid_sd, fc["y_pred"] + 1.64 * resid_sd
            pi_note = f"fallback interval +/- 1.64 sd of differences ({type(err).__name__})"
        hh_runs[hid] = {"exp": e, "cv": cv_h, "test": e.pull(), "fc": fc}
        print(f"household {hid}: CV MASE {cv_h['MASE'].iloc[-1]:.2f} | test-window MASE {hh_runs[hid]['test']['MASE'].iloc[0]:.2f} | {pi_note}")
print("\nforecast frame columns:", list(hh_runs[SAMPLE_HH[0]]["fc"].columns))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=False)
for ax, hid in zip(axes, SAMPLE_HH):
    r = hh_runs[hid]; e = r["exp"]; fc = r["fc"]
    ytr, yte = e.get_config("y_train"), e.get_config("y_test")
    ax.plot(_t(ytr.index), ytr.values, marker="o", ms=3, color="#333", label="train")
    ax.plot(_t(yte.index), yte.values, marker="o", color="#28a745", lw=2, label="actual")
    ax.plot(_t(fc.index), fc["y_pred"].values, marker="s", ls="--", color="#dc3545", label="ets forecast")
    ax.fill_between(_t(fc.index), fc["lower"].values, fc["upper"].values, color="#dc3545", alpha=0.15, label="90% interval")
    inside = ((yte.values >= fc["lower"].values) & (yte.values <= fc["upper"].values)).sum()
    ax.set_title(f"household {hid}: test MASE {r['test']['MASE'].iloc[0]:.2f}, {inside}/3 actuals inside the interval")
    ax.set_xlabel("month"); ax.set_ylabel("total spend"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()
widths = {hid: (hh_runs[hid]["fc"]["upper"] - hh_runs[hid]["fc"]["lower"]).round(0).tolist() for hid in SAMPLE_HH}
print("interval width by horizon step (1, 2, 3 months ahead):", widths)


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Three households, each with its own `ets` fitted on its own twenty-one months. The dashed line is the point forecast and the shaded band the 90% prediction interval; the title reports the test-window MASE and how many of the three actuals fell inside the band. The band is wide relative to the month-to-month movement of a single household, and the printed widths grow with the horizon step.

The common misread: judging the intervals by whether all three actuals fall inside. With a 90% band, one actual in ten *should* fall outside; three households times three months is nine draws, so a single miss is expected behaviour and no miss at all would hint that the band is too wide.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Per-series models do not scale, and they cannot learn from each other

Each household's model saw twenty-one points and nothing else. It does not know that December is expensive for *everyone*, that large households have larger grocery swings, or that a surprise expense last month usually does not repeat. A panel model, one model for all series, can learn those shared patterns from 1,500 histories at once. That is the trade at the heart of modern forecasting: per-series statistical models are interpretable and need no feature engineering; one global machine-learning model needs lag features but learns across series.

</div>


## 19.2 A panel alternative: reduced regression with LightGBM

**Reduced regression** turns forecasting into a table: for every household and every origin month, the features are the last six months of spend (the **lags**), the horizon step `h` (1, 2 or 3), and the static household attributes; the target is the spend `h` months after the origin. One `lightgbm` regressor from `pycaret.regression`, the same module as Part 5, is fitted on all households at once. The table below is built so that its test rows forecast months 22 to 24 from origin month 21, exactly the window Task 18 and Section 19.1 held out, so the MASE values are comparable.


In [ ]:
from pycaret.regression import RegressionExperiment

LAGS, H = 6, 3
static = hh.set_index(ID_COL)[["city_tier", "household_size", "owns_car"]]
vals = wide.values                                              # (1500, 24), month index 0..23
n_months = vals.shape[1]
last_train_origin = n_months - 1 - H                            # origin 20 (0-based) -> targets at months 21..23 (0-based) = months 22..24

def make_rows(origins):
    frames = []
    for o in origins:
        for h in range(1, H + 1):
            if o + h >= n_months:
                continue
            f = pd.DataFrame(vals[:, o - LAGS + 1:o + 1], columns=[f"lag_{LAGS - i}" for i in range(LAGS)], index=wide.index)
            f["month_of_target"] = ((o + h) % 12) + 1
            f["h"] = h
            f["origin"] = o
            f["y"] = vals[:, o + h]
            frames.append(f.join(static))
    return pd.concat(frames).reset_index()

panel_train = make_rows(range(LAGS - 1 + 3, last_train_origin))          # origins whose targets all fall on or before month 21
panel_train = panel_train[panel_train.origin + panel_train.h <= last_train_origin]
panel_test = make_rows([last_train_origin])                               # origin month 21 -> forecasts for months 22, 23, 24
print(f"train rows: {len(panel_train):,} (origins {panel_train.origin.min()+1}-{panel_train.origin.max()+1}, targets up to month {last_train_origin+1})")
print(f"test rows:  {len(panel_test):,} (origin month {last_train_origin+1}, horizons 1-3 -> months {last_train_origin+2}-{n_months})")
print(panel_train.head(3).to_string())


In [ ]:
panel = RegressionExperiment()
with timer("panel setup + lightgbm (fold=3)"):
    panel.setup(data=panel_train.drop(columns=[ID_COL, "origin"]), target="y", test_data=panel_test.drop(columns=[ID_COL, "origin"]),
                fold=3, session_id=RANDOM_STATE, verbose=False, html=False, n_jobs=4, index=False)
    lgb_name = "lightgbm" if "lightgbm" in panel.models().index else "gbr"
    lgb_panel = panel.create_model(lgb_name, verbose=False)
print("model used:", lgb_name)
print(panel.pull().to_string())
panel_pred = panel.predict_model(lgb_panel, verbose=False)
print("\ntest-window (months 22-24, all households) metrics from pull():")
print(panel.pull().to_string(index=False))


## 19.3 MASE across approaches

The MASE of each approach is computed per household on the same three test months, with the same denominator PyCaret uses (the mean absolute one-step difference over that household's training months), then averaged. The `naive` and `snaive` forecasts for every household are one line each; `ets` was fitted only for the three sample households, so its column shows those three, and the panel LightGBM is scored on the same three and on all 1,500.


In [ ]:
test_idx = np.arange(last_train_origin + 1, n_months)                                 # months 22..24 (0-based 21..23)
actual = vals[:, test_idx]                                                            # (1500, 3)
scale = np.abs(np.diff(vals[:, :last_train_origin + 1], axis=1)).mean(axis=1)         # per-household MASE denominator
naive_fc = np.repeat(vals[:, [last_train_origin]], H, axis=1)
snaive_fc = vals[:, test_idx - 12]
lgb_fc = (panel_pred.assign(hid=panel_test[ID_COL].values)
          .pivot(index="hid", columns="h", values="prediction_label").loc[wide.index].values)

def mase_rows(fc):
    return np.abs(actual - fc).mean(axis=1) / scale

mase = pd.DataFrame({"naive": mase_rows(naive_fc), "snaive": mase_rows(snaive_fc), f"panel {lgb_name}": mase_rows(lgb_fc)}, index=wide.index)
ets_col = pd.Series({hid: float(hh_runs[hid]["test"]["MASE"].iloc[0]) for hid in SAMPLE_HH}, name="ets (per series)")
mase = mase.join(ets_col)

summary = mase.loc[SAMPLE_HH].T.rename(columns=lambda h: f"household {h}")
summary["mean, all 1,500"] = mase.drop(columns=["ets (per series)"]).mean()
display(pretty(summary, highlight="min"))
print(summary.round(2).to_string())
wins = (mase[f"panel {lgb_name}"] < mase[["naive", "snaive"]].min(axis=1)).mean()
print(f"\npanel model beats the better naive baseline on {wins:.0%} of households; "
      f"share of households where MASE < 1 for the panel model: {(mase[f'panel {lgb_name}'] < 1).mean():.0%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cols = ["naive", "snaive", f"panel {lgb_name}"]
axes[0].boxplot([mase[c].values for c in cols], labels=cols, showfliers=False)
axes[0].axhline(1, color="#dc3545", ls="--", label="MASE = 1 (in-sample naive)"); axes[0].set_title("Per-household MASE on months 22-24, all 1,500 households")
axes[0].set_ylabel("MASE (lower is better)"); axes[0].legend()
imp = pd.Series(lgb_panel.feature_importances_, index=panel.get_config("X_train_transformed").columns).sort_values()
imp.tail(10).plot.barh(ax=axes[1], color="#667eea"); axes[1].set_title(f"Panel {lgb_name}: split importance of the top 10 features"); axes[1].set_xlabel("importance")
plt.tight_layout(); plt.show()
print("median MASE:", mase[cols].median().round(2).to_dict())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: the distribution of per-household MASE for the three approaches that cover all households. The dashed line at 1 is "as good as repeating last month". Right: what the panel model relied on: the most recent lags, the target month (which encodes the December bump the per-series models had to learn from one December each), and the static attributes. The summary table above adds the three sample households, where the per-series `ets` and the panel model can be compared directly.

The common misread: declaring a winner from the three sample households. They are three draws; the "mean, all 1,500" column and the box plot are the evidence, and even those are one three-month window.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: when statistical forecasting wins, and when the panel does

- **One or a few long, clean series** (a company's monthly revenue, a country's demand): `ets`, `arima`, `theta` through `pycaret.time_series`, with `snaive` in the table and prediction intervals in the report. They need no features and are easy to explain.
- **Many short, related series** (every household, every product, every store): a single panel model on lags and static attributes. It learns the calendar and the cross-sectional patterns from all series at once, and it produces one model to deploy instead of thousands. The 2026 frontier extends this idea to pretrained **time-series foundation models** (Chronos, TimesFM, Moirai) that were trained on millions of series and forecast a new one zero-shot; Part 11 places them in the landscape.
- **Either way**, the test is the same: does it beat `snaive` on a held-out window, per series, with an interval that is honest?

</div>

## 19.4 Task 19 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- A per-series `ets` gives a point forecast and a prediction interval that widens with the horizon; `predict_model(..., return_pred_int=True, coverage=...)` returns `lower` / `upper`.
- Per-series models cannot learn shared patterns and do not scale to thousands of series.
- **Reduced regression** turns forecasting into a supervised table (lags + horizon + static features) and one `lightgbm` from `pycaret.regression` forecasts every household at once.
- Compare approaches with per-household MASE on the same held-out window; the median over all series and the share of series with MASE below 1 are the honest summaries.
- Statistical models for a few long series; a panel model (or a foundation model) for many related short ones.

### 🔑 New variables

| Variable | What it holds |
|---|---|
| `wide` | 1,500 households x 24 months of `total_spend` |
| `SAMPLE_HH`, `hh_runs` | the three sample households and their per-series experiments, CV tables, test metrics and forecasts |
| `panel_train`, `panel_test`, `panel`, `lgb_panel`, `panel_pred` | the reduced-regression table, the `RegressionExperiment`, the fitted model and its test predictions |
| `mase`, `summary` | per-household MASE for naive / snaive / panel (+ ets for the sample), and the comparison table |

### ➡️ Next Up

Part 9 takes the classifier from Part 4 to production: finalize, save, load, predict on new households, log experiments, and watch for drift.

</div>


# Part 9 · MLOps: finalize, save, predict, experiment logging, drift, fairness ⭐⭐⭐

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### From a notebook model to a model in service

A model that lives in a notebook cell is not a product. This part turns the classifier into an artefact that can be **refit on all the data, saved to disk, loaded somewhere else, served over HTTP, logged run by run, audited for fairness, and watched for drift**. PyCaret has one function for each step; every step also has a trap, and each trap is shown with a number.

Parts 9 and 10 are self-contained: the first cell runs its own `setup()` and `compare_models()` on three fast models, and both parts reuse that experiment.

</div>


# Task 20 · Finalize, save, load, predict on new data, and the API file

## 20.1 The experiment Parts 9 and 10 share

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Same split, same numbers

The setup below passes `test_data=test_df` so that the held-out set inside PyCaret is exactly the notebook's `test_df`. Because `train_df` and `test_df` both carry a fresh `RangeIndex`, they share index values; `index=False` tells PyCaret to rebuild a single index instead of failing on duplicates. Three quick models are compared on 3 folds and the best one by AUC becomes `best`.

The cell also attaches a **logger**: a small object that receives every parameter table and metric row PyCaret produces. Task 21 explains it; for now it only listens.

</div>


In [ ]:
from pycaret.classification import *
from pycaret.loggers.base_logger import BaseLogger
import os, importlib.util, tempfile, pathlib
import sklearn


class RunLog(BaseLogger):
    """A minimal PyCaret logger: keeps every logged parameter set and metric row in memory (explained in Task 21)."""
    def __init__(self):
        self.runs, self.metrics, self.params, self.artifacts = [], [], [], []
        self._current = None
    def init_experiment(self, exp_name_log, full_name=None, setup=True, **kwargs):
        self._current = full_name; self.runs.append({"experiment": exp_name_log, "run": full_name, "is_setup": setup})
    def log_params(self, params, model_name=None):
        self.params.append({"run": model_name, "n_params": len(params), **{k: str(v)[:30] for k, v in list(params.items())[:4]}})
    def log_metrics(self, metrics, source=None):
        self.metrics.append({"run": self._current, "source": source, **metrics})
    def log_artifact(self, file, type="artifact"):
        self.artifacts.append({"run": self._current, "type": type, "file": os.path.basename(str(file))})
    def log_sklearn_pipeline(self, experiment, prep_pipe, model, path=None):
        self.artifacts.append({"run": self._current, "type": "pipeline", "file": type(model).__name__})


run_log = RunLog()
DROP = [ID_COL, LABEL_REG, "note"]          # id, the regression target, free text (a high-cardinality categorical to PyCaret)
with timer("setup + compare_models(3 models, 3 folds)"):
    exp9 = setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP), index=False,
                 session_id=RANDOM_STATE, fold=3, n_jobs=4, verbose=False, html=False,
                 log_experiment=[run_log], experiment_name="budget_overspend")
    best = compare_models(include=["lr", "rf", "gbc"], fold=3, sort="AUC", verbose=False)
cmp9 = pull()
best_id = cmp9.index[0]                      # PyCaret's short id of the winner, e.g. "lr"; used to retrain later
best_name = cmp9.loc[best_id, "Model"]
print(cmp9[["Model", "Accuracy", "AUC", "Recall", "Prec.", "F1", "TT (Sec)"]].round(4).to_string())
print(f"\nbest by CV AUC: {best_name} (id='{best_id}')  |  train rows {len(get_config('X_train'))}, test rows {len(get_config('X_test'))}")


## 20.2 `finalize_model`: refit on everything, and what that does to the test numbers

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Why finalize at all

During development the held-out rows are kept out of training so they can score the model honestly. Once the model is chosen, those rows are wasted potential: **`finalize_model` refits the same pipeline and the same hyper-parameters on the training set and the held-out set together**, so the shipped model has seen every labelled row.

The price is that the held-out set is no longer held out. Any score computed on it after finalizing is a training score in disguise. The cell below shows the gap.

</div>


In [ ]:
# holdout numbers BEFORE finalize: predict_model(best) with no data argument scores the experiment's test set
_ = predict_model(best, verbose=False); before = pull().iloc[0]

with timer("finalize_model"):
    final = finalize_model(best)              # a full Pipeline (preprocessing + estimator) refit on train + test

_ = predict_model(final, verbose=False); after = pull().iloc[0]
gap = pd.DataFrame({"before finalize (honest holdout)": before, "after finalize (test rows were in training)": after}).drop(index="Model")
print(gap.to_string())
print(f"\nfinal is a {type(final).__name__} with steps: {[name for name, _ in final.steps]}")
print(f"AUC on the 'test' set moved from {float(before['AUC']):.4f} to {float(after['AUC']):.4f} after finalize: the second number is not a test score.")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Never report a metric computed after `finalize_model`

The table above has two columns for the same 375 rows. Accuracy, AUC, recall and F1 all rise in the second column, because the finalized model was fit on those rows (precision can move either way when the threshold stays fixed). The honest numbers are the ones from **before** finalizing, and the honest way to say it is: "the model was selected on cross-validation, scored once on the held-out set, then refit on all labelled data for deployment."

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: freeze the report before you finalize

Save `pull()` from the last honest `predict_model` to a variable (or a file) **before** calling `finalize_model`. Ship the finalized pipeline, report the frozen numbers. In this notebook `before` is that frozen row.

</div>

## 20.3 `save_model` and `load_model`

`save_model` pickles the **whole pipeline**: imputers, encoders, and the estimator, plus the experiment's metadata. `load_model` returns the same pipeline, so raw rows in the original column layout go in and predictions come out, with no preprocessing code to copy. Both write next to the current working directory unless the path is absolute; here they write to a temporary folder.


In [ ]:
MODEL_DIR = pathlib.Path(tempfile.mkdtemp(prefix="pycaret_budget_"))
model_path = MODEL_DIR / "overspend_pipeline"           # save_model adds the .pkl extension itself
saved_pipeline, saved_file = save_model(final, str(model_path), verbose=False)

loaded = load_model(str(model_path), verbose=False)
size_kb = os.path.getsize(saved_file) / 1024
print(f"saved  -> {os.path.basename(saved_file)} ({size_kb:.0f} KB)")
print(f"loaded -> {type(loaded).__name__} with steps:")
for name, step in loaded.steps:
    inner = getattr(step, "transformer", step)
    print(f"   {name:<20s} {type(inner).__name__}")

# the loaded object must reproduce the in-memory one exactly on raw rows
p_mem = predict_model(final, data=test_df.drop(columns=DROP + [LABEL_CLS]), raw_score=True, verbose=False)["prediction_score_1"]
p_disk = predict_model(loaded, data=test_df.drop(columns=DROP + [LABEL_CLS]), raw_score=True, verbose=False)["prediction_score_1"]
print(f"\nmax |P(overspend) in memory - from disk| over {len(p_mem)} test rows: {np.abs(p_mem.values - p_disk.values).max():.2e}")


## 20.4 Predict on rows the model has never seen

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 New data must arrive in the training layout

`predict_model` expects the same raw columns that went into `setup` (before any preprocessing), in any order, without the target. Five new households are generated with a different seed, passed through the same `household_table` builder, and scored. By default the output has `prediction_score`, the probability of the **predicted** class, which is not the positive-class probability for rows predicted 0. `raw_score=True` replaces it with one probability column per class (`prediction_score_0`, `prediction_score_1`). The cell scores the rows both ways to show the difference.

</div>


In [ ]:
new_hh = household_table(make_budget(n_households=5, months=24, seed=7))    # five never-seen households
new_rows = new_hh.drop(columns=DROP + [LABEL_CLS])                               # same layout as setup's data, no target
scored = predict_model(loaded, data=new_rows, raw_score=True, verbose=False)      # prediction_score_0 / _1: one column per class
default = predict_model(loaded, data=new_rows, verbose=False)                     # default: prediction_score = score of the PREDICTED class

view = scored[["city_tier", "income", "total_spend", "overspend_rate_prev", "prediction_label", "prediction_score_1"]].copy()
view.insert(5, "prediction_score", default["prediction_score"].values)
view["true_next"] = new_hh[LABEL_CLS].values
view["spend/income"] = (scored["total_spend"] / scored["income"]).round(2)
print(view.round(3).to_string())
print(f"\nprediction_score is the score of the predicted class; prediction_score_1 is P(overspend): "
      f"row 0 has prediction_score={view.loc[0, 'prediction_score']:.3f} and P(overspend)={view.loc[0, 'prediction_score_1']:.3f}")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `prediction_score` is not the positive-class probability

For a household predicted `0`, `prediction_score` is the probability of class 0, so a row with score 0.96 is a *confident stay-within-budget*, not a 96% overspend risk. Rank or threshold on `prediction_score_1` (from `raw_score=True`); never on `prediction_score`.

</div>

## 20.5 The serving file: `create_api`, `create_docker`, `create_app`

PyCaret can write the deployment scaffolding for you:

| Function | Writes | Needs |
|---|---|---|
| `create_api(model, "name")` | `name.py`, a FastAPI app with a `/predict` endpoint, plus `name.pkl` | `fastapi`, `uvicorn`, `pydantic` |
| `create_docker("name")` | `Dockerfile` and `requirements.txt` around that API | nothing extra |
| `create_app(model)` | launches a Gradio form for manual predictions | `gradio` |

The cell writes the API file into the temporary folder and prints it. It does **not** start the server; the file is shown so you can read exactly what it does. When `fastapi` is not installed, the identical file PyCaret would produce is written by hand from its template, so the output does not depend on the environment.


In [ ]:
# !pip install -q fastapi uvicorn   # needed only to RUN the generated API (pycaret[mlops] installs them)
API_NAME = "overspend_api"
cwd = os.getcwd(); os.chdir(MODEL_DIR)
try:
    create_api(final, API_NAME)                     # writes overspend_api.py and overspend_api.pkl in the current folder
    how = "written by PyCaret's create_api"
except ModuleNotFoundError as e:                    # fastapi/uvicorn missing: write the same file PyCaret generates
    save_model(final, API_NAME, verbose=False)
    X0, y0 = get_config("X").iloc[0].to_dict(), get_config("y").iloc[0]
    api_src = f"""# -*- coding: utf-8 -*-

import pandas as pd
from pycaret.classification import load_model, predict_model
from fastapi import FastAPI
import uvicorn
from pydantic import create_model

# Create the app
app = FastAPI()

# Load trained Pipeline
model = load_model("{API_NAME}")

# Create input/output pydantic models
input_model = create_model("{API_NAME}_input", **{X0})
output_model = create_model("{API_NAME}_output", prediction={y0!r})


# Define predict function
@app.post("/predict", response_model=output_model)
def predict(data: input_model):
    data = pd.DataFrame([data.dict()])
    predictions = predict_model(model, data=data)
    return {{"prediction": predictions["prediction_label"].iloc[0]}}


if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
"""
    pathlib.Path(f"{API_NAME}.py").write_text(api_src, encoding="utf-8")
    how = f"written from PyCaret's template ({str(e).strip().splitlines()[0][:60]}...)"
create_docker(API_NAME)                             # Dockerfile + requirements.txt next to the API file
os.chdir(cwd)

print(f"{API_NAME}.py {how}\n" + "-" * 70)
print((MODEL_DIR / f"{API_NAME}.py").read_text(encoding="utf-8"))
print("-" * 70 + "\nDockerfile:\n" + (MODEL_DIR / "Dockerfile").read_text(encoding="utf-8").strip())
print("\nfiles in the deploy folder:", sorted(p.name for p in MODEL_DIR.iterdir()))
print("\nto serve:  python overspend_api.py   (not run here)   |   to containerise:  docker build -t overspend .")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: read the generated file before shipping it

Three things in the generated API deserve a second look. The `input_model` is built from the **first row of the training data**, so every field's type is inferred from one example (an integer-looking float becomes `int`). The endpoint returns `prediction_label` only, so a consumer who wants a probability must edit it to call `predict_model(..., raw_score=True)`. And the pickle is loaded from the working directory, so the container's `WORKDIR` must contain it. Generated code is a starting point, not a deliverable.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Pickles pin versions

A pipeline saved with one PyCaret and scikit-learn version may fail to load with another. Record `pycaret.__version__` and `sklearn.__version__` next to the artefact (the Dockerfile from `create_docker` pins nothing by default: edit `requirements.txt`).

</div>

## 20.6 Task 20 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- `finalize_model` refits on train **and** held-out rows; the held-out metrics after it are optimistic (the table in 20.2 shows the jump). Freeze the honest numbers first.
- `save_model` / `load_model` move the whole pipeline; a max-difference check of predictions confirms the loaded object is the same model.
- New rows go in as raw columns in the setup layout; use `prediction_score_1` for the positive-class probability.
- `create_api` and `create_docker` write scaffolding, not a finished service; read the generated file.

### 🔑 New variables

| Variable | What it is |
|---|---|
| `exp9`, `best`, `best_id`, `best_name` | the shared experiment, its best model, PyCaret's id and display name |
| `run_log` | the in-memory logger attached to the experiment (Task 21) |
| `before` | the frozen honest holdout row, taken before finalize |
| `final`, `loaded`, `MODEL_DIR` | the finalized pipeline, its reloaded copy, the temp folder with the artefacts |
| `new_rows`, `scored` | five unseen households and their predictions |

➡️ **Next Up:** Task 21 opens `run_log`, shows what PyCaret logs per run, and audits the model for fairness across city tiers.

</div>


# Task 21 · Experiment logging, dashboards and a fairness check

## 21.1 What PyCaret logs, and where

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 A run is parameters + metrics + artefacts

Every `create_model`, `compare_models` or `tune_model` call is a **run**. A run has parameters (the estimator's hyper-parameters and the setup knobs), metrics (the CV table), and artefacts (the fitted pipeline, the results HTML). Logging is the habit of writing all three somewhere durable so that next month you can answer "which settings gave AUC 0.91?" without re-running anything.

`setup(log_experiment=True, experiment_name=...)` sends runs to **MLflow**; `"wandb"`, `"comet_ml"` and `"dagshub"` are the other built-in targets, and a list of `BaseLogger` objects lets you write your own. `RunLog` in 20.1 is exactly that: eight short methods that receive what MLflow would receive. Reading its contents shows the logging contract without a tracking server.

</div>


In [ ]:
# !pip install -q mlflow   # then: setup(..., log_experiment=True, experiment_name="budget_overspend"); get_logs(); !mlflow ui
HAS_MLFLOW = importlib.util.find_spec("mlflow") is not None
print(f"mlflow installed: {HAS_MLFLOW}  ->  {'get_logs() would return the MLflow run table' if HAS_MLFLOW else 'log_experiment=True would raise; the RunLog logger below records the same calls'}")

runs = pd.DataFrame(run_log.runs)
metrics_log = pd.DataFrame(run_log.metrics)
params_log = pd.DataFrame(run_log.params)
arts = pd.DataFrame(run_log.artifacts)
print(f"\nruns logged: {len(runs)} (1 setup run + 1 per model)   metric rows: {len(metrics_log)}   param tables: {len(params_log)}   artefacts: {len(arts)}")
print("\nmetrics per run (what MLflow's run table would show):")
print(metrics_log.set_index("run").round(4).to_string())
print("\nparameters per run (first four of each table):")
print(params_log.set_index("run").to_string())
print("\nartefact types:", arts.groupby("type").size().to_dict())


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: log the setup, not only the model

The first run in the table is the `setup` run and its parameter table is the longest. Two models with identical hyper-parameters differ completely if one was trained with `normalize=True` and the other was not; a log that omits the preprocessing knobs cannot reproduce anything. PyCaret logs them automatically; a home-made training script rarely does.

</div>

## 21.2 `dashboard()` and the functions that are not there

`dashboard(best)` builds an interactive **explainerdashboard** (SHAP, what-if, confusion matrices) and starts a web server; it is not run here because the notebook validator has no browser. A function called `deep_check` does not exist in PyCaret 3.3.2; the checks it might suggest (drift, fairness) are `check_drift` (Task 22) and `check_fairness` (below).

## 21.3 Fairness across city tiers

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Same model, different groups

A model can be accurate overall and still treat one group worse. `check_fairness(model, sensitive_features=[...])` computes every registered metric **per group** of a sensitive column, plus the **selection rate** (the share of the group predicted positive), using `fairlearn`. Two classic summaries:

- **Demographic parity difference**: max selection rate minus min selection rate across groups. Zero means every tier is flagged at the same rate.
- **Equalized odds gaps**: the spread of the true-positive rate and of the false-positive rate across groups. Zero means errors are spread evenly.

Both are computed by hand below, so the audit runs with or without `fairlearn`.

</div>


In [ ]:
# !pip install -q fairlearn   # pycaret[analysis] installs it
SENSITIVE = "city_tier"
plt.close("all")                                                   # PyCaret draws on the current axes: start from a clean figure
try:
    fair = check_fairness(best, sensitive_features=[SENSITIVE])      # draws a 3x3 grid of per-group bars via pandas
    plt.gcf().suptitle(f"check_fairness: every metric per {SENSITIVE} (fairlearn MetricFrame)", y=1.02)
    plt.tight_layout(); plt.show()
    print("check_fairness output (per group):")
    print(fair.round(3).to_string())
except ModuleNotFoundError as e:
    fair = None
    print("fairlearn not installed; the by-hand audit below covers the same ground.")

# the same audit by hand, from the honest holdout predictions
hold = predict_model(best, raw_score=True, verbose=False)
grp = {}
for g, d in hold.groupby(SENSITIVE):
    y, yhat, p = d[LABEL_CLS].values, d["prediction_label"].values, d["prediction_score_1"].values
    grp[g] = {"n": len(d), "base_rate": y.mean(), "selection_rate": yhat.mean(),
              "TPR (recall)": recall_score(y, yhat, zero_division=0),
              "FPR": ((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1),
              "precision": precision_score(y, yhat, zero_division=0),
              "roc_auc (probabilities)": roc_auc_score(y, p)}
by_group = pd.DataFrame(grp).T
dp_diff = by_group["selection_rate"].max() - by_group["selection_rate"].min()
eo_tpr = by_group["TPR (recall)"].max() - by_group["TPR (recall)"].min()
eo_fpr = by_group["FPR"].max() - by_group["FPR"].min()
print("\nby-hand metrics per group:")
print(by_group.round(3).to_string())
print(f"\ndemographic parity difference = {dp_diff:.3f}   equalized-odds gaps: TPR {eo_tpr:.3f}, FPR {eo_fpr:.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Nine small bar charts, one per metric, three bars each (one per city tier). The bars for `Samples` and `Selection Rate` say how many households each tier has and how many are flagged; the rest are accuracy-style metrics per tier. The by-hand table underneath adds the **base rate** per tier, which the PyCaret grid does not show. Metro households overspend far more often (rent takes a bigger share of income there), so a much higher selection rate for metro is the model following the data, not inventing a bias: the demographic parity difference is large *because the base rates differ*. The equalized-odds gaps are the fairer question: are errors of each kind spread evenly? The common misread: taking the `AUC` bars in the grid as ranking quality. `check_fairness` feeds fairlearn **hard labels**, so its "AUC" is the balanced accuracy of the labels; the by-hand column `roc_auc (probabilities)` is the real ranking AUC.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Parity is a choice, not a computation

Forcing equal selection rates here would mean flagging town households that are not at risk and missing metro households that are. Which gap matters (selection rate, TPR, FPR, precision) depends on what happens to a flagged household. Decide that with the people affected before optimising any of them, and read every per-group number with its `n`: a tier with a hundred test rows and a dozen positives moves its TPR by a tenth with one household.

</div>

## 21.4 Task 21 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- A run is parameters + metrics + artefacts; `log_experiment=True` sends them to MLflow, a `BaseLogger` subclass sends them anywhere. The setup run carries the preprocessing knobs; log it.
- `check_fairness` gives every metric per group; add the base rate and a probability AUC by hand to read it correctly.
- Demographic parity and equalized odds usually disagree when base rates differ; choosing between them is a product decision.

### 🔑 New variables

| Variable | What it is |
|---|---|
| `HAS_MLFLOW` | whether `log_experiment=True` would work in this environment |
| `metrics_log`, `params_log` | the logged metric rows and parameter tables from `run_log` |
| `hold` | honest holdout predictions of `best` with both class probabilities |
| `by_group`, `dp_diff`, `eo_tpr`, `eo_fpr` | the by-hand fairness table and its two summary gaps |

➡️ **Next Up:** Task 22 makes a drifted month, measures it with PSI, watches the score distribution move, and retrains.

</div>


# Task 22 · Drift and monitoring

## 22.1 A month that looks different

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Three things can drift

- **Data (covariate) drift**: the inputs move. Incomes rise, dining habits change.
- **Label (prior) drift**: the share of positives moves.
- **Concept drift**: the relation between inputs and label changes; the same income and spend now mean a different risk.

Only the first is visible before labels arrive, so monitoring starts there. The cell fabricates a drifted month from the held-out households: income up 20% for everyone and dining shifted up, everything else untouched, outcomes kept as they were.

</div>

### 📐 The Population Stability Index

For one feature, bin the reference values into $B$ quantile bins, then count the share of reference rows $r_b$ and current rows $c_b$ in each bin:

$$ \text{PSI} = \sum_{b=1}^{B} (c_b - r_b)\,\ln\frac{c_b}{r_b} $$

Every term is non-negative (it is a symmetric KL-style divergence), so PSI is zero only when the two histograms agree. The usual reading: below 0.1 stable, 0.1 to 0.25 watch, above 0.25 act. These cut-offs are conventions from credit scoring, not theorems.


In [ ]:
reference = test_df.drop(columns=DROP).copy()                       # last month's scored rows, labels known
drifted = reference.copy()
drifted["income"] = (drifted["income"] * 1.20).round(0)                # everyone earns 20% more
drifted["dining"] = (drifted["dining"] * 1.35 + 40).round(0)           # dining shifts up
NUMERIC = [c for c in reference.columns if c != LABEL_CLS and pd.api.types.is_numeric_dtype(reference[c]) and reference[c].nunique() > 2]


def psi(ref, cur, bins=10):
    # Population Stability Index on quantile bins of the reference; NaNs dropped, empty bins smoothed.
    ref, cur = pd.Series(ref).dropna(), pd.Series(cur).dropna()
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref); c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))


psi_tab = pd.DataFrame({"PSI": {f: psi(reference[f], drifted[f]) for f in NUMERIC}}).sort_values("PSI", ascending=False)
psi_tab["status"] = np.select([psi_tab.PSI > 0.25, psi_tab.PSI > 0.10], ["act", "watch"], "stable")
print(psi_tab.round(4).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
colors = psi_tab.status.map({"act": "#dc3545", "watch": "#ffc107", "stable": "#2196F3"})
ax.bar(psi_tab.index, psi_tab.PSI, color=colors)
ax.axhline(0.10, ls="--", color="#ffc107", label="0.10 watch"); ax.axhline(0.25, ls="--", color="#dc3545", label="0.25 act")
ax.set_title("PSI per numeric feature: reference month vs drifted month"); ax.set_ylabel("PSI"); ax.set_xlabel("feature")
ax.tick_params(axis="x", rotation=45); ax.legend()
plt.tight_layout(); plt.show()
print(f"features flagged: {psi_tab.index[psi_tab.status != 'stable'].tolist()}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

One bar per numeric feature, sorted by PSI, with the two conventional lines. The two features that were changed stand far above the lines; every untouched feature sits near zero, which is the sanity check that the statistic is not noisy on 375 rows with ten bins. `income` moved by a constant factor and still scores high, because a 20% shift moves most rows across at least one decile boundary. The common misread: PSI says *that* a feature moved, not *whether it matters*. A feature the model barely uses can drift with no effect on predictions, and a small PSI on a dominant feature can matter a lot; the next cell checks the predictions themselves.

</div>

## 22.2 The score distribution moves before the labels arrive

Predictions are available the moment a new month is scored, weeks before its outcomes. Comparing the distribution of `prediction_score_1` on the reference month and the drifted month is the earliest model-level alarm.


In [ ]:
X_ref, X_cur = reference.drop(columns=[LABEL_CLS]), drifted.drop(columns=[LABEL_CLS])
# `best`, not `final`: the reference month is the held-out set, and `final` has already trained on it (20.2)
p_ref = predict_model(best, data=X_ref, raw_score=True, verbose=False)["prediction_score_1"].values
p_cur = predict_model(best, data=X_cur, raw_score=True, verbose=False)["prediction_score_1"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(0, 1, 21)
axes[0].hist(p_ref, bins=bins, alpha=0.6, color="#2196F3", label=f"reference (mean {p_ref.mean():.3f})")
axes[0].hist(p_cur, bins=bins, alpha=0.6, color="#dc3545", label=f"drifted (mean {p_cur.mean():.3f})")
axes[0].set_title("P(overspend) before and after the drift"); axes[0].set_xlabel("P(overspend next month)"); axes[0].set_ylabel("households"); axes[0].legend()
flag_ref, flag_cur = (p_ref >= 0.5).mean(), (p_cur >= 0.5).mean()
axes[1].bar(["reference", "drifted"], [flag_ref, flag_cur], color=["#2196F3", "#dc3545"])
axes[1].set_title("Share of households flagged (threshold 0.5)"); axes[1].set_ylabel("flagged share")
for i, v in enumerate([flag_ref, flag_cur]): axes[1].text(i, v + 0.005, f"{v:.1%}", ha="center")
plt.tight_layout(); plt.show()
score_psi = psi(p_ref, p_cur)
print(f"mean score {p_ref.mean():.3f} -> {p_cur.mean():.3f}   flagged share {flag_ref:.1%} -> {flag_cur:.1%}   PSI of the score itself: {score_psi:.3f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left: two histograms of the predicted overspend probability, reference month in blue and drifted month in red. The red mass slides left: households with 20% more income and the same rent look safer to the model, so fewer cross 0.5 (right panel). The PSI of the score itself, printed under the plot, is a single number that summarises the slide and is the metric to alert on. The common misread: a falling flagged share is not good news by itself. If the outcomes did not improve as much as the inputs suggest, every household that dropped below the threshold is a missed overspend, which is what the next cell measures once labels arrive.

</div>

## 22.3 When the labels arrive: performance on the drifted month

A month later the outcomes are known. Scoring the same rows again with `report_card` turns the alarm into a measurement (with `best`, the model that never saw these rows). The drifted table keeps the households' real outcomes, which is the pessimistic case: inputs moved, behaviour did not.


In [ ]:
perf = pd.concat([
    report_card(reference[LABEL_CLS], (p_ref >= 0.5).astype(int), p_ref, name="reference month"),
    report_card(drifted[LABEL_CLS], (p_cur >= 0.5).astype(int), p_cur, name="drifted month (same outcomes)"),
])
print(perf.round(3).to_string())
drop_recall = float(perf.loc["reference month", "recall"] - perf.loc["drifted month (same outcomes)", "recall"])
print(f"\nrecall fell by {drop_recall:.3f}; log loss rose from {perf.loc['reference month', 'log_loss']:.3f} to {perf.loc['drifted month (same outcomes)', 'log_loss']:.3f}")

# PyCaret's own drift report needs the `evidently` package; it writes an HTML file (opened in a browser, not shown here)
# !pip install -q evidently   # pycaret[mlops]
try:
    drift_html = check_drift(reference_data=reference, current_data=drifted, target=LABEL_CLS)
    print(f"check_drift wrote {drift_html}")
    os.remove(drift_html)
except ModuleNotFoundError as e:
    print("check_drift exists in PyCaret 3.3.2 but needs evidently:", str(e).strip().splitlines()[0][:70])
pretty(perf, highlight="max")        # the styled version of the same table


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Ranking can survive while decisions fail

Compare `roc_auc` with `recall` in the two rows. AUC barely moves, because a uniform income change reorders few households; recall collapses, because the whole score distribution slid under the fixed 0.5 threshold. A dashboard that only tracks AUC would have missed this month. Track a threshold metric and a calibration metric (`log_loss`, `brier`) next to the ranking metric.

</div>

## 22.4 The monitoring loop and the retrain path

| Stage | Question | Signal | Tool in this notebook |
|---|---|---|---|
| 1. Data quality | Are the columns, types and ranges what the pipeline expects? | schema check, missing-rate per column | `predict_model` on a schema-checked frame |
| 2. Drift | Did the inputs move? Did the scores move? | PSI per feature, PSI of the score, flagged share | `psi`, `check_drift` |
| 3. Performance | Once labels arrive, is the model still good enough? | `report_card` on the new month vs the frozen `before` row | `report_card` |
| 4. Retrain trigger | Is the drop larger than the noise of one month? | recall or log loss beyond a tolerance for two months in a row | `ClassificationExperiment` retrain below |

The retrain path re-runs `setup` on the new labelled data with the **same configuration** and re-creates the same estimator by its id. It uses the object-oriented `ClassificationExperiment` so the functional experiment (`exp9`) that Part 10 needs is left untouched.


In [ ]:
from pycaret.classification import ClassificationExperiment

retrain_data = pd.concat([train_df.drop(columns=DROP), drifted], ignore_index=True)     # old labelled rows + the new month
with timer(f"retrain '{best_id}' on {len(retrain_data)} rows"):
    exp_re = ClassificationExperiment()
    exp_re.setup(retrain_data, target=LABEL_CLS, session_id=RANDOM_STATE, fold=3, n_jobs=4, verbose=False, html=False)
    best_re = exp_re.create_model(best_id, fold=3, verbose=False)
cv_re = exp_re.pull()
print(f"retrained {best_name} CV (3 folds, including the drifted month):")
print(cv_re.loc[["Mean", "Std"], ["Accuracy", "AUC", "Recall", "Prec.", "F1"]].round(4).to_string())

# does the retrained model recover on drifted-looking rows? score a fresh drifted month built from NEW households
fresh = household_table(make_budget(n_households=400, months=24, seed=99)).drop(columns=DROP)
fresh["income"] = (fresh["income"] * 1.20).round(0); fresh["dining"] = (fresh["dining"] * 1.35 + 40).round(0)
p_old = predict_model(final, data=fresh.drop(columns=[LABEL_CLS]), raw_score=True, verbose=False)["prediction_score_1"].values
p_new = exp_re.predict_model(best_re, data=fresh.drop(columns=[LABEL_CLS]), raw_score=True, verbose=False)["prediction_score_1"].values
recover = pd.concat([report_card(fresh[LABEL_CLS], (p_old >= 0.5).astype(int), p_old, name="old model on new drifted month"),
                     report_card(fresh[LABEL_CLS], (p_new >= 0.5).astype(int), p_new, name="retrained model on new drifted month")])
print("\n" + recover[["recall", "precision", "f1", "roc_auc", "log_loss"]].round(3).to_string())
print(f"\nsame setup knobs, same estimator id '{best_id}', new data: that is the whole retrain recipe.")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: retrain by id and config, not by hand

`create_model(best_id)` inside a fresh `setup` with the same knobs reproduces the deployed model's recipe on new data. Keep the recipe (the setup call, the id, `session_id`) under version control next to the pickle; the pickle is the output, the recipe is the source. When a retrain is triggered, the comparison that matters is the old and new model **on the same new labelled month**, which is what the last table does.

</div>

## 22.5 Task 22 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- PSI per feature finds *which* inputs moved; PSI of the score and the flagged share show *whether the model cares*.
- Labels arrive late; the score histogram is the earliest model-level alarm.
- Under input drift, AUC can hold while recall collapses: monitor threshold and calibration metrics too.
- The retrain path is `setup` with the same knobs plus `create_model(best_id)`; use `ClassificationExperiment` for side experiments so the main one stays intact.

### 🔑 New variables

| Variable | What it is |
|---|---|
| `reference`, `drifted` | the held-out month and its drifted copy (income ×1.2, dining shifted) |
| `psi`, `psi_tab`, `score_psi` | the PSI function, its per-feature table, and the PSI of the score |
| `p_ref`, `p_cur`, `perf` | scores on both months and the report cards once labels arrived |
| `exp_re`, `best_re`, `recover` | the retrain experiment, the retrained model and the old-vs-new comparison |

➡️ **Next Up:** Part 10 turns to the errors themselves: which households the model gets wrong, in which segments, and why.

</div>


# Part 10 · Error analysis and diagnostics ⭐⭐

<div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### One AUC hides a hundred stories

A single held-out score says how good the model is on average. It does not say **which** households it misses, whether one city tier carries all the mistakes, whether more data would help, or whether the labels can be trusted. This part answers those questions with the experiment from Part 9 (`best`, `exp9`) and ends with a checklist that is computed, not asserted.

</div>


# Task 23 · Where it fails

## 23.1 The confusion matrix on the held-out set

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Two kinds of wrong

A **false negative** is a household that overspent but was not flagged: a missed warning. A **false positive** is a flagged household that stayed within budget: a false alarm. Which is worse depends on what a flag triggers, and Part 4 priced them. Here the point is to look at the two groups separately, because they usually have different causes.

</div>


In [ ]:
from sklearn.metrics import confusion_matrix

hold = predict_model(best, raw_score=True, verbose=False)              # honest holdout rows with all raw columns
y_true, y_hat, p1 = hold[LABEL_CLS].values, hold["prediction_label"].values, hold["prediction_score_1"].values
cm = confusion_matrix(y_true, y_hat)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mat, title, fmt in [(axes[0], cm, "counts", "d"), (axes[1], cm / cm.sum(axis=1, keepdims=True), "row-normalised (recall view)", ".2f")]:
    sns.heatmap(mat, annot=True, fmt=fmt, cmap="Blues", cbar=False, ax=ax,
                xticklabels=["pred: within", "pred: overspend"], yticklabels=["true: within", "true: overspend"])
    ax.set_title(f"{best_name}: confusion matrix ({title})"); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
plt.tight_layout(); plt.show()
print(f"TN {tn}  FP {fp}  FN {fn}  TP {tp}   |   recall {tp / (tp + fn):.3f}, precision {tp / (tp + fp):.3f}, "
      f"missed overspends {fn} of {tp + fn}, false alarms {fp} of {tn + fp} safe households")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Left, the raw counts; right, each row divided by its total, so the bottom-right cell is recall and the top-right cell is the false-positive rate. The top row is dark because most households stay within budget and the model is confident about them. The bottom row is where the money is: the bottom-left cell is the missed overspends. The common misread: reading the row-normalised matrix as precision. Precision is a column-wise quantity (of the flagged, how many overspent) and it is printed under the plot.

</div>

## 23.2 The hardest mistakes

Sorting the false negatives by ascending score and the false positives by descending score puts the most confident mistakes first. These are the rows worth reading one by one: they are either label noise, a missing feature, or a pattern the model has not learned.


In [ ]:
hold["spend_to_income"] = (hold["total_spend"] / hold["income"]).round(3)
show_cols = ["city_tier", "household_size", "income", "total_spend", "spend_to_income", "overspent", "overspend_rate_prev", "avg_spend_prev", "prediction_score_1"]
fneg = hold[(y_true == 1) & (y_hat == 0)].sort_values("prediction_score_1").head(5)
fpos = hold[(y_true == 0) & (y_hat == 1)].sort_values("prediction_score_1", ascending=False).head(5)
print(f"5 most confident FALSE NEGATIVES (overspent next month, lowest P(overspend)) of {fn}:")
print(fneg[show_cols].round(3).to_string())
print(f"\n5 most confident FALSE POSITIVES (stayed within budget, highest P(overspend)) of {fp}:")
print(fpos[show_cols].round(3).to_string())
print(f"\nmedian spend/income: FN {fneg.spend_to_income.median():.2f}  FP {fpos.spend_to_income.median():.2f}  "
      f"all overspenders {hold.loc[y_true == 1, 'spend_to_income'].median():.2f}  all safe {hold.loc[y_true == 0, 'spend_to_income'].median():.2f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Reading the two tables

The confident false negatives look like safe households: spend well under income this month, a low history of overspending. What made them overspend next month is not in the features, most often a **surprise expense** that the generator plants in six percent of household-months. No feature predicts a surprise, so these are an irreducible error at this feature set, not a model bug. The confident false positives are the mirror image: households that usually overspend and did not this time. The fix for the first group is a new signal (or accepting the floor); the fix for the second is often the threshold, not the model.

</div>

## 23.3 Error rate by segment

An average error rate can hide a segment where the model is much worse. The cell computes the error rate, the missed-overspend rate and the base rate per `city_tier`, `household_size` and `owns_car`.


In [ ]:
hold["error"] = (y_true != y_hat).astype(int)
hold["missed"] = ((y_true == 1) & (y_hat == 0)).astype(int)
seg_rows = []
for col in ["city_tier", "household_size", "owns_car"]:
    for g, d in hold.groupby(col):
        pos = d[LABEL_CLS].sum()
        seg_rows.append({"segment": f"{col}={g}", "n": len(d), "base_rate": d[LABEL_CLS].mean(), "error_rate": d["error"].mean(),
                         "missed_rate (FN/positives)": d["missed"].sum() / pos if pos else np.nan})
seg = pd.DataFrame(seg_rows).set_index("segment")
print(seg.round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(seg)); w = 0.27
ax.bar(x - w, seg["base_rate"], w, label="base rate (share overspending)", color="#9c27b0")
ax.bar(x, seg["error_rate"], w, label="error rate", color="#ffc107")
ax.bar(x + w, seg["missed_rate (FN/positives)"], w, label="missed rate (FN / positives)", color="#dc3545")
ax.set_xticks(x); ax.set_xticklabels(seg.index, rotation=45, ha="right"); ax.set_ylabel("rate")
ax.set_title("Where the errors live: base rate, error rate and missed rate per segment"); ax.legend()
plt.tight_layout(); plt.show()
worst = seg["error_rate"].idxmax(); best_seg = seg["error_rate"].idxmin()
print(f"highest error rate: {worst} ({seg.loc[worst, 'error_rate']:.3f}, n={seg.loc[worst, 'n']})   lowest: {best_seg} ({seg.loc[best_seg, 'error_rate']:.3f})")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Three bars per segment. The purple bar is how often that segment overspends, the yellow bar is how often the model is wrong there, and the red bar is the share of that segment's overspenders it misses. The yellow bar mostly tracks the purple one: segments with more positives have more errors, because errors concentrate in the minority class. The red bar is the fairer comparison across segments, and it is highest where positives are rare (town and city tiers, one-person households) and lowest where they are common (metro, five-person households): the model has few examples of the rare-segment pattern to learn from. The common misread: calling the segment with the highest error rate "the problem segment". It is usually the segment with the most positives, and a model that never flags anyone has a low error rate wherever overspending is rare.

</div>

## 23.4 Calibration by segment

A model can be calibrated overall and miscalibrated in every segment (the errors cancel). One reliability curve per city tier, five bins each, shows whether "P = 0.4" means the same thing in a metro and in a town.


In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
cal_rows = []
for g, d in hold.groupby("city_tier"):
    frac, mean_p = calibration_curve(d[LABEL_CLS], d["prediction_score_1"], n_bins=5, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", label=f"{g} (n={len(d)})")
    cal_rows.append({"city_tier": g, "n": len(d), "mean P(overspend)": d["prediction_score_1"].mean(), "observed rate": d[LABEL_CLS].mean(),
                     "brier": brier_score_loss(d[LABEL_CLS], d["prediction_score_1"])})
ax.set_xlabel("mean predicted P(overspend) in bin"); ax.set_ylabel("observed overspend rate in bin")
ax.set_title(f"Reliability curve per city tier ({best_name}, 5 quantile bins)"); ax.legend()
plt.tight_layout(); plt.show()
cal = pd.DataFrame(cal_rows).set_index("city_tier")
cal["over/under (mean P - observed)"] = cal["mean P(overspend)"] - cal["observed rate"]
print(cal.round(3).to_string())


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

One curve per city tier; the dashed diagonal is perfect calibration. Points above the line are bins where the model under-predicts risk, below it over-predicts. With five bins and about a hundred rows per tier, each point averages roughly twenty households, so wobbles of a tenth are noise. The table underneath is the coarser, more reliable summary: the mean predicted probability against the observed rate per tier, and the Brier score per tier. The common misread: treating a curve that crosses the diagonal several times as "badly calibrated". Read the gap in the table first, then the curve; segment reliability curves need thousands of rows to be smooth.

</div>

## 23.5 Would more data help? The learning curve

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Bias or variance

A learning curve plots the score against the number of training rows, once on the training folds and once on validation. If the two curves are close and flat, the model is limited by **bias** (it cannot represent the pattern, or the features lack the signal): more rows will not help. If the training curve is high and the validation curve is still rising, the model is limited by **variance**: more rows help. `plot_model(plot="learning")` draws it with scikit-learn's `learning_curve`, here for a **decision tree** (`dt`), the classic high-variance model; the second cell computes the same idea by hand for `best` at 200, 500 and 1000 rows on the fixed held-out set, so the two models can be compared.

</div>


In [ ]:
import matplotlib.image as mpimg

with timer("decision tree + learning curve"):
    dt = create_model("dt", fold=3, verbose=False)                        # an unpruned tree: the textbook high-variance model
    dt_cv = pull().loc["Mean", ["AUC", "F1"]]
    plt.close("all")                                                     # plot_model draws on the current axes: start clean
    os.chdir(MODEL_DIR)
    lc_png = plot_model(dt, plot="learning", save=True, verbose=False)   # writes "Learning Curve.png" and returns its name
    os.chdir(cwd)
img = mpimg.imread(MODEL_DIR / lc_png)
fig, ax = plt.subplots(figsize=(9, 5.5)); ax.imshow(img); ax.axis("off"); ax.set_title("plot_model(dt, plot='learning'): decision tree")
plt.tight_layout(); plt.show()
print(f"rendered {lc_png} ({img.shape[1]}x{img.shape[0]} px); the score on the y-axis is PyCaret's default for classification (Accuracy)")
print(f"decision tree CV: AUC {dt_cv['AUC']:.4f}, F1 {dt_cv['F1']:.4f}  vs  {best_name}: AUC {cmp9.loc[best_id, 'AUC']:.4f}, F1 {cmp9.loc[best_id, 'F1']:.4f}")


In [ ]:
from sklearn.base import clone

Xtr_t, ytr_t = get_config("X_train_transformed"), get_config("y_train_transformed")
Xte_t, yte_t = get_config("X_test_transformed"), get_config("y_test_transformed")
order = np.random.default_rng(RANDOM_STATE).permutation(len(Xtr_t))
lc_rows = []
with timer("hand-made learning curve (3 fits)"):
    for n in [200, 500, 1000]:
        idx = order[:n]
        m = clone(best).fit(Xtr_t.iloc[idx], ytr_t.iloc[idx])
        p_tr, p_te = m.predict_proba(Xtr_t.iloc[idx])[:, 1], m.predict_proba(Xte_t)[:, 1]
        lc_rows.append({"train rows": n, "train AUC": roc_auc_score(ytr_t.iloc[idx], p_tr), "test AUC": roc_auc_score(yte_t, p_te),
                        "train PR-AUC": average_precision_score(ytr_t.iloc[idx], p_tr), "test PR-AUC": average_precision_score(yte_t, p_te)})
lc = pd.DataFrame(lc_rows).set_index("train rows")
print(lc.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lc.index, lc["train AUC"], marker="o", label="train AUC"); ax.plot(lc.index, lc["test AUC"], marker="o", label="held-out AUC")
ax.set_xlabel("training rows"); ax.set_ylabel("ROC-AUC"); ax.set_title(f"Learning curve by hand: {best_name} on the fixed held-out set"); ax.legend()
plt.tight_layout(); plt.show()
gain = float(lc["test AUC"].iloc[-1] - lc["test AUC"].iloc[0])
gain_late = float(lc["test AUC"].iloc[-1] - lc["test AUC"].iloc[1])
print(f"held-out AUC {gain:+.4f} from 200 to 1000 rows, {gain_late:+.4f} from 500 to 1000; train-test gap at 1000 rows: {float(lc['train AUC'].iloc[-1] - lc['test AUC'].iloc[-1]):+.4f}")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two learning curves for two models. PyCaret's plot is the decision tree: the training accuracy sits at 1.0 for every size (an unpruned tree memorises its rows) while the cross-validation curve hovers around 0.84, barely above the 0.83 of a model that flags nobody. That gap is variance, and it does not close as rows are added: a tree of unlimited depth keeps memorising, so what it needs is a depth limit (the next plot), not more data. The hand-made curve is the logistic regression on the fixed held-out set: it rises from 200 to 500 rows and then flattens, and the training curve sits on top of the held-out one with no gap. A linear model on twenty-odd features has almost no variance to shrink, so more rows will not move it; its remaining error is bias, the surprise expenses no feature predicts. The common misread: a flat accuracy curve on an imbalanced target. Accuracy is about 0.83 for a model that flags nobody, so an accuracy curve can look flat while AUC is still moving; that is why the hand-made curve uses AUC.

</div>

## 23.6 Bias and variance made visible: the validation curve

`plot_model(plot="vc")` varies one hyper-parameter (for a decision tree, `max_depth`) and plots the training and cross-validation score against it. Left of the crossover the tree is too shallow (bias); right of it the training score keeps rising while validation falls (variance).


In [ ]:
with timer("validation curve over max_depth"):
    plt.close("all")
    os.chdir(MODEL_DIR)
    vc_png = plot_model(dt, plot="vc", save=True, verbose=False)         # re-fits the tree at each depth, 3 folds each
    os.chdir(cwd)
img = mpimg.imread(MODEL_DIR / vc_png)
fig, ax = plt.subplots(figsize=(9, 5.5)); ax.imshow(img); ax.axis("off"); ax.set_title("plot_model(dt, plot='vc'): score vs max_depth for a decision tree")
plt.tight_layout(); plt.show()
print(f"the unpruned tree (CV AUC {dt_cv['AUC']:.4f}) lies beyond the right edge of this curve; the best depth is where the validation line peaks")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Two lines against `max_depth`. The training score climbs toward a perfect fit as the tree deepens, because a deep tree can memorise every training row. The cross-validation score is highest at depth one or two and falls from there: on this data a stump on the spend-to-income signal already captures most of what a tree can, and every extra level fits noise, so the crossover sits at the far left. The vertical distance between the lines is the variance; the height of the validation line at the left is the bias of a shallow tree, and here that bias is small. The common misread: choosing the depth where the training score peaks. The training score always peaks at the deepest tree; the depth to pick is where the **validation** line is highest, and Part 4's `tune_model` searched it automatically.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ `plot_model` draws from cross-validation on the training set

Both `learning` and `vc` re-fit the model several times inside the training data. The held-out set is not involved, which is correct (it must stay untouched for the final number) but means these curves say nothing about the test score. The hand-made curve in 23.5 is the one that uses the held-out set, and it is used once, to read a trend, not to pick anything.

</div>

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: one error table per release

Before shipping, save the four things this task produced: the confusion matrix counts, the top confident mistakes with their features, the per-segment error table and the per-segment calibration table. Comparing them release to release catches regressions that the headline AUC hides: a new feature that fixes town households and breaks metro ones leaves the average unchanged.

</div>

## 23.7 Task 23 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- Split the errors into missed overspends and false alarms; they have different causes and different fixes.
- The most confident false negatives look safe on every feature: they are surprise expenses, an error floor at this feature set.
- Error rate per segment follows the base rate; compare the missed rate instead.
- Learning curves say whether more rows help (not here: the tree needs pruning, the linear model is at its bias floor); validation curves say whether the model is too simple or too complex. Both come from CV inside training.

### 🔑 New variables

| Variable | What it is |
|---|---|
| `hold`, `y_true`, `y_hat`, `p1`, `cm` | honest holdout rows with predictions, and the confusion matrix |
| `fneg`, `fpos` | the five most confident false negatives and false positives |
| `seg`, `cal` | error and calibration tables per segment |
| `dt`, `dt_cv`, `lc`, `gain_late` | the decision tree and its CV row, the hand-made learning curve, its late-stage gain |

➡️ **Next Up:** Task 24 runs the diagnostics checklist: label noise, duplicates, overlap, leakage-like correlations, base rates, and a computed ✅/⚠️ table.

</div>


# Task 24 · The diagnostics checklist

## 24.1 How much does label noise cost?

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Labels are data too

Real labels come from forms, thresholds and people, and some are wrong. The cheapest way to learn how sensitive a model is: flip a share of the training labels at random, retrain with the same recipe, and score on the clean held-out set. The drop tells you what ten percent noise costs, which is also what cleaning ten percent of the labels would buy. The retrain uses a side `ClassificationExperiment` so `exp9` stays intact.

</div>


In [ ]:
NOISE = 0.10
rng = np.random.default_rng(RANDOM_STATE)
noisy_train = train_df.drop(columns=DROP).copy()
flip = rng.random(len(noisy_train)) < NOISE
noisy_train.loc[flip, LABEL_CLS] = 1 - noisy_train.loc[flip, LABEL_CLS]
with timer(f"retrain '{best_id}' with {NOISE:.0%} flipped labels"):
    exp_noise = ClassificationExperiment()
    exp_noise.setup(noisy_train, target=LABEL_CLS, test_data=test_df.drop(columns=DROP), index=False,
                    session_id=RANDOM_STATE, fold=3, n_jobs=4, verbose=False, html=False)
    noisy_model = exp_noise.create_model(best_id, fold=3, verbose=False)
pn = exp_noise.predict_model(noisy_model, raw_score=True, verbose=False)      # scores the same clean test_df
p_noisy = pn["prediction_score_1"].values
noise_tab = pd.concat([report_card(y_true, y_hat, p1, name="clean labels"),
                       report_card(pn[LABEL_CLS].values, (p_noisy >= 0.5).astype(int), p_noisy, name=f"{NOISE:.0%} labels flipped")])
print(noise_tab.round(3).to_string())
auc_drop = float(noise_tab.loc["clean labels", "roc_auc"] - noise_tab.loc[f"{NOISE:.0%} labels flipped", "roc_auc"])
f1_drop = float(noise_tab.loc["clean labels", "f1"] - noise_tab.loc[f"{NOISE:.0%} labels flipped", "f1"])
print(f"\n{flip.sum()} of {len(flip)} training labels flipped (positive rate {train_df[LABEL_CLS].mean():.3f} -> {noisy_train[LABEL_CLS].mean():.3f}); "
      f"held-out AUC dropped by {auc_drop:.4f}, F1 by {f1_drop:.4f}")
pretty(noise_tab, highlight="max")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Random flips are the kind case

Symmetric random noise mostly blurs the decision boundary, and a linear or averaged model shrugs off a lot of it: the AUC drop above is modest. Real label noise is **systematic** (one city's records are entered late, one clerk rounds differently) and correlates with the features, which moves the boundary instead of blurring it. Treat this cell as a lower bound on the damage.

</div>

## 24.2 Duplicates, overlap, leakage-like correlations and base rates

Four cheap checks that catch most silent data bugs:

1. **Duplicate rows** in training inflate the weight of some households and make CV optimistic (the same row can land in both folds).
2. **Train/test overlap** on the feature columns means the held-out score is partly a training score.
3. **Feature-target correlation**: a feature whose correlation with the label is near 1 is usually a copy of the label (Part 5 built one on purpose).
4. **Base rate per segment, train vs test**: a stratified split keeps the overall rate equal; per segment it should be close, and a big gap means the test set is not representative for that segment.


In [ ]:
feat_cols = [c for c in train_df.columns if c not in DROP + [LABEL_CLS]]
n_dup_train = int(train_df.duplicated(subset=feat_cols).sum())
n_dup_all = int(hh.duplicated(subset=feat_cols).sum())
overlap = train_df[feat_cols].merge(test_df[feat_cols].drop_duplicates(), how="inner", on=feat_cols)
n_overlap = len(overlap)

num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(train_df[c])]
corr_target = train_df[num_cols + [LABEL_CLS]].corr()[LABEL_CLS].drop(LABEL_CLS).abs().sort_values(ascending=False)
LEAK_T = 0.90
print("duplicate rows on the feature columns: train", n_dup_train, "| whole table", n_dup_all)
print("train/test overlap on the feature columns:", n_overlap, "rows")
print(f"\n|correlation| with {LABEL_CLS}, top 6 (leak threshold {LEAK_T}):")
print(corr_target.head(6).round(3).to_string())

base = pd.DataFrame({"train": train_df.groupby("city_tier")[LABEL_CLS].mean(), "test": test_df.groupby("city_tier")[LABEL_CLS].mean()})
base["gap"] = (base["train"] - base["test"]).abs()
base["n_test"] = test_df.groupby("city_tier").size()
print("\npositive rate per city tier, train vs test:")
print(base.round(3).to_string())
max_gap = float(base["gap"].max())


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: check overlap on the features, not on an id

Two households with identical feature rows are the same row to the model, whatever their ids say. The overlap check above merges on **every feature column**; a check on `household_id` alone would miss exports where the same household was re-keyed. Zero here is expected because the generator draws every household once; on real exports a non-zero count is common and always worth a look.

</div>

## 24.3 `check_drift`: does it exist, and does it run?

PyCaret 3.3.2 does ship `check_drift` (it appeared in Task 22). It builds an HTML report with the `evidently` package and returns the file name; without `evidently` it raises `ModuleNotFoundError`. The checklist records which of the two happened.


In [ ]:
import pycaret.classification as pc
has_check_drift = hasattr(pc, "check_drift")
try:
    fname = check_drift(reference_data=train_df.drop(columns=DROP), current_data=test_df.drop(columns=DROP), target=LABEL_CLS)
    drift_status = f"ran, wrote {os.path.basename(fname)}"; os.remove(fname)
except ModuleNotFoundError:
    drift_status = "present in 3.3.2, needs `pip install evidently`"
print(f"check_drift in pycaret.classification: {has_check_drift}  ->  {drift_status}")


## 24.4 The checklist, computed

Every row below is a boolean evaluated from a number this part computed, with the number shown. A ⚠️ is not a failure; it is a row to explain before the model ships.


In [ ]:
checks = [
    ("Held-out numbers frozen before finalize", True, f"AUC {float(before['AUC']):.4f} saved in `before`"),
    ("Saved pipeline reproduces in-memory predictions", bool(np.abs(p_mem.values - p_disk.values).max() < 1e-9), f"max diff {np.abs(p_mem.values - p_disk.values).max():.1e}"),
    ("Equalized-odds TPR gap across city tiers < 0.20", bool(eo_tpr < 0.20), f"TPR gap {eo_tpr:.3f}, FPR gap {eo_fpr:.3f}"),
    ("No feature with PSI > 0.25 in the latest month", bool((psi_tab.PSI <= 0.25).all()), f"max PSI {psi_tab.PSI.max():.3f} ({psi_tab.PSI.idxmax()}) on the drifted month"),
    ("Recall on the latest labelled month within 0.10 of reference", bool(drop_recall <= 0.10), f"recall drop {drop_recall:.3f} on the drifted month"),
    ("Held-out AUC drop under 10% label noise < 0.05", bool(auc_drop < 0.05), f"AUC drop {auc_drop:.4f}"),
    ("No duplicate feature rows in training", n_dup_train == 0, f"{n_dup_train} duplicates"),
    ("No train/test overlap on the feature columns", n_overlap == 0, f"{n_overlap} overlapping rows"),
    (f"No feature with |corr(target)| > {LEAK_T}", bool(corr_target.max() < LEAK_T), f"max |corr| {corr_target.max():.3f} ({corr_target.idxmax()})"),
    ("Per-tier base rate gap train vs test < 0.05", bool(max_gap < 0.05), f"max gap {max_gap:.3f}"),
    ("Learning curve flat from 500 to 1000 rows (data is not the bottleneck)", bool(abs(gain_late) < 0.01), f"held-out AUC {gain_late:+.4f} from 500 to 1000 rows"),
    ("check_drift available", has_check_drift and "ran" in drift_status, drift_status),
]
checklist = pd.DataFrame(checks, columns=["check", "passed", "evidence"])
checklist["status"] = np.where(checklist["passed"], "✅", "⚠️")
print(checklist[["status", "check", "evidence"]].to_string(index=False))
n_ok = int(checklist["passed"].sum())
flagged = checklist.loc[~checklist["passed"], "check"].tolist()
print(f"\n{n_ok} of {len(checklist)} checks pass. Rows to explain before shipping: {flagged}")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ A checklist is only as honest as its thresholds

The cut-offs above (0.20 for the TPR gap, 0.25 for PSI, 0.10 for the recall drop, 0.05 for the noise cost) are reasonable defaults, not laws. Set them **before** looking at the numbers, write them next to the model card, and change them only with a reason you would say out loud. A threshold moved after the fact to turn a ⚠️ into a ✅ is the most common way a checklist stops working.

</div>

## 24.5 Task 24 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- Flipping 10% of the training labels and retraining with the same recipe prices label quality; random flips are the kind case.
- Duplicates, feature-level train/test overlap, near-perfect feature-target correlations and per-segment base-rate gaps are four one-line checks that catch most silent data bugs.
- `check_drift` exists in PyCaret 3.3.2 and needs `evidently`; PSI by hand covers the same question offline.
- A computed ✅/⚠️ table with the evidence next to each row is the artefact to attach to every release.

### 🔑 New variables

| Variable | What it is |
|---|---|
| `noisy_train`, `exp_noise`, `noisy_model`, `noise_tab` | the flipped-label training set, its side experiment, model and report card |
| `n_dup_train`, `n_overlap`, `corr_target`, `base` | the four data checks |
| `has_check_drift`, `drift_status` | whether `check_drift` exists and what happened when it ran |
| `checklist` | the final ✅/⚠️ table with evidence |

➡️ **Next Up:** Part 11 puts PyCaret next to AutoGluon, RAPIDS and tabular foundation models, and gives the 2026 verdict on when low-code wins.

</div>


# Part 11 · SOTA 2026: where low-code fits next to AutoGluon, RAPIDS and foundation models 🔬

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; color: white; margin: 10px 0;">

### Where this sits

Ten parts taught you to drive PyCaret. This part answers the question every project eventually asks: **is this the right tool, and what do I graduate to when it is not?** Task 25 puts PyCaret next to the three things that compete with it in 2026 (a full AutoML system, a GPU stack, and tabular foundation models) and runs each idea on the budget data. Task 26 gives the honest verdict, the release status of the library you just learned, and a decision flowchart you can defend in a design review.

Every claim below that has a date was checked in September 2026. Every number was computed in a cell.

</div>


# Task 25 · Low-code ML in the 2026 landscape

## 25.1 Four ways to fit a tabular model

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: the same job, four philosophies

Every tool in this table turns a DataFrame into a fitted pipeline. They differ in **who makes the decisions**: you (scikit-learn), you with a menu (PyCaret), the system (AutoGluon), or the hardware budget (RAPIDS, which makes the *same* decisions faster). None of them is "best"; each wins a region of the data-size × control × latency space that Task 26 maps.

| | **Hand-built scikit-learn** | **PyCaret 3.3** | **AutoGluon 1.6** | **RAPIDS cuML 26.x** |
|---|---|---|---|---|
| Setup effort | `ColumnTransformer` + estimator, every choice explicit | one `setup()` with about sixty knobs, sensible defaults | one `TabularPredictor.fit()`; presets decide almost everything | `import cuml` or `cuml.accel` on top of sklearn code |
| Model zoo | everything, one at a time | 18 classifiers via `compare_models` in one call | GBMs, neural nets, and tabular foundation models (TabPFN, TabICL, TabDPT, Nori) | GPU ports of the classical sklearn estimators (LR, RF, kNN, SVM, k-means, DBSCAN, UMAP, HDBSCAN) |
| Ensembling | you write it | `blend_models`, `stack_models`, `ensemble_model` (Part 4) | automatic multi-layer stacking + bagging, the default | none built in; you stack by hand |
| GPU | via LightGBM / XGBoost flags | `setup(use_gpu=True)` swaps engines (25.2) | automatic for GBMs and neural models | the whole point |
| Deployment | pickle + your own service | `save_model`, `create_api`, `create_docker`, `create_app` (Part 9) | `predictor.save()` + cloud containers | same as sklearn, needs a GPU host |
| Text, images, time series | text via TF-IDF; images no | free text is a categorical (Part 2); separate `time_series` module (Part 8) | multimodal predictor; text/image/tabular fusion; `TimeSeriesPredictor` with Chronos/Toto foundation models | none |
| When it wins | you need every knob, a small dependency set, or a custom loss | you need a comparison, a plot gallery and an explanation in an afternoon | you need the best number and can spend the compute | your sklearn code is too slow and you have an NVIDIA GPU |

</div>

The one comparison that is worth a cell rather than a table is **effort against accuracy**. The cell below fits the hand-built baseline from Part 1 and PyCaret's LightGBM on the same split and times both. Two things to watch: how close the numbers are, and where the time goes.

One argument in the `setup` call is new: `index=False`. `train_df` and `test_df` each start their index at 0, and PyCaret concatenates the two when `test_data=` is given; without `index=False` it stops with "duplicate indices". Every `setup` in this Part passes it.


In [ ]:
# ----------------------------------------------------------------- hand-built sklearn vs PyCaret on one split
from pycaret.classification import *
import inspect
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier

DROP = [ID_COL, LABEL_REG, "note"]                  # id, the other target, free text (Part 2 explains the note column)
X_tr, y_tr = train_df.drop(columns=DROP + [LABEL_CLS]), train_df[LABEL_CLS]
X_te, y_te = test_df.drop(columns=DROP + [LABEL_CLS]), test_df[LABEL_CLS]
num_cols = X_tr.select_dtypes("number").columns.tolist()
cat_cols = [c for c in X_tr.columns if c not in num_cols]

# (a) hand-built: every decision is a line of code
with timer("sklearn hand-built HGB (fit + predict)"):
    prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), num_cols),
                              ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                                ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])
    hand = Pipeline([("prep", prep), ("hgb", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]).fit(X_tr, y_tr)
    p_hand = hand.predict_proba(X_te)[:, 1]

# (b) PyCaret: the decisions live in setup(); n_jobs=4 keeps the shared machine polite
assert {"session_id", "verbose", "html", "n_jobs", "test_data", "use_gpu"} <= set(inspect.signature(setup).parameters)
with timer("PyCaret setup + create_model('lightgbm', fold=3) + predict_model"):
    exp = setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP),
                session_id=RANDOM_STATE, fold=3, n_jobs=4, index=False, verbose=False, html=False, system_log=False)
    lgbm = create_model("lightgbm", fold=3, verbose=False)
    cv_lgbm = pull()
    pred = predict_model(lgbm, data=test_df.drop(columns=DROP), raw_score=True, verbose=False)
    p_pc = pred["prediction_score_1"].values

cards = pd.concat([report_card(y_te, (p_hand >= 0.5).astype(int), p_hand, name="sklearn HGB (hand-built)"),
                   report_card(y_te, (p_pc >= 0.5).astype(int), p_pc, name="PyCaret lightgbm")])
display(pretty(cards[["accuracy", "recall", "f1", "roc_auc", "pr_auc", "log_loss"]]))
print(f"CV AUC of the PyCaret model on training folds: {cv_lgbm.loc['Mean', 'AUC']:.3f}  |  test AUC: {cards.loc['PyCaret lightgbm', 'roc_auc']:.3f}")
print(f"Test AUC gap between the two: {abs(cards.roc_auc.iloc[0] - cards.roc_auc.iloc[1]):.3f}")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Reading the two rows

The two report cards sit within a few hundredths of AUC of each other, because both are gradient-boosted trees on the same 20 columns. What PyCaret bought for its extra seconds is not accuracy; it is the **cross-validated table** (`cv_lgbm`), a pipeline with imputation and encoding you did not write, and the whole plot, tune, ensemble, interpret and deploy vocabulary of Parts 3 to 9 on top of that one object. The hand-built pipeline bought a dependency list of one package and total control.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Low-code is not low-decision

`setup()` made about sixty choices for you in that cell (median imputation, one-hot encoding, a stratified split, three folds). They were reasonable; they were not *yours*. Part 2 walked through every knob for a reason: when a low-code result surprises you, the explanation is almost always a default you did not know you accepted.

</div>

## 25.2 `setup(use_gpu=True)`: what actually changes

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Intuition: the flag swaps engines, it does not rewrite models

PyCaret does not contain GPU code. `use_gpu=True` tells each model container to look for a GPU **engine** and use it if one is importable:

| Model id | CPU engine | GPU engine (when `use_gpu=True`) |
|---|---|---|
| `lr`, `knn`, `rbfsvm`, `svm`, `ridge`, `rf`, `kmeans`, `dbscan` | scikit-learn | **RAPIDS cuML** (needs `cuml >= 23.08`) |
| `xgboost` | `tree_method="hist"`, `device="cpu"` | `device="gpu"` (the same library) |
| `lightgbm` | default | `device="gpu"`, then `device="cuda"`, tried by fitting a 2 × 2 dummy at container-build time |
| `catboost` | `task_type="CPU"` | `task_type="GPU"`, `border_count=32` |
| everything else (`dt`, `et`, `nb`, `ada`, `gbc`, `lda`, `qda`, ...) | scikit-learn | unchanged; still runs on CPU |

`use_gpu=True` **warns and falls back** when cuML is missing; `use_gpu="force"` raises instead. The related `engine=` argument on `setup`, `create_model` and `compare_models` lets you pick `"sklearnex"` (Intel's CPU accelerator) or `"cuml"` per model. The cell below runs the flag on this machine and shows what the model registry reports.

</div>


In [ ]:
# ----------------------------------------------------------------- use_gpu=True, guarded: shows the mechanism with or without a GPU
gpu_note = []
try:
    import cuml  # noqa: F401
    gpu_note.append(f"cuML {cuml.__version__} found: cuML-backed engines will be used")
except ImportError:
    gpu_note.append("cuML not installed: PyCaret logs a warning and keeps the scikit-learn engines (use_gpu='force' would raise)")

try:
    exp_gpu = setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP),
                    session_id=RANDOM_STATE, fold=3, n_jobs=4, use_gpu=True, index=False, verbose=False, html=False, system_log=False)
    zoo = models(internal=True)                        # the registry, one row per model container
    gpu_col = [c for c in zoo.columns if "GPU" in c][0]
    on_gpu = zoo[zoo[gpu_col].astype(str).isin(["True", "gpu", "cuda"])]
    print(f"model containers: {len(zoo)}  |  reporting a GPU engine: {len(on_gpu)}  ({', '.join(on_gpu.index) or 'none'})")
    print(f"engine chosen for 'lr': {get_engine('lr')}   allowed engines for 'lr': {get_allowed_engines('lr')}")
    print(f"engine chosen for 'lightgbm': {get_engine('lightgbm')}  (None means the library's own, non-swappable engine)")
    display(zoo.loc[["lr", "knn", "rf", "lightgbm", "xgboost", "catboost", "dt"], ["Name", "Turbo", gpu_col]] if "xgboost" in zoo.index
            else zoo.loc[[m for m in ["lr", "knn", "rf", "lightgbm", "catboost", "dt"] if m in zoo.index], ["Name", "Turbo", gpu_col]])
except Exception as e:                                 # a broken CUDA install must not stop the notebook
    print(f"use_gpu=True raised {type(e).__name__}: {str(e)[:120]}")
for n in gpu_note:
    print("•", n)
print("• Zero-code alternative (RAPIDS 25.02+): `%load_ext cuml.accel` before importing sklearn, and PyCaret's sklearn engines run on the GPU untouched.")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: decide GPU by the fit that hurts, not by the flag

On this table (about eleven hundred training rows) every fit in the course took seconds, and a GPU would spend more time moving data than computing. GPU acceleration pays when one of three things is true: **rows beyond a few hundred thousand**, **kNN / SVM / UMAP / HDBSCAN** (whose cost grows faster than linearly), or **a `tune_model` loop** that repeats a fit hundreds of times. Measure with `timer` first; if `compare_models` finishes inside a coffee break, the flag is a distraction. When it does not, install RAPIDS (`cuml-cu12`, monthly releases; 26.08 shipped on 6 August 2026), and the same notebook runs with `use_gpu=True`.

</div>

<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ A GPU model is a different model

cuML's `LogisticRegression` and `RandomForest` are re-implementations, not wrappers. Defaults, tie-breaking and numerical paths differ, so a model tuned on CPU can score differently on GPU and vice versa. Re-run `compare_models` after flipping the engine; never carry a CPU-tuned grid across.

</div>

## 25.3 Foundation models for tables inside a PyCaret experiment

<div style="background-color: #f3e5f5; border-left: 5px solid #9c27b0; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🔬 SOTA: tabular in-context learning

A **tabular foundation model** is a transformer pre-trained on millions of *synthetic* tables. At prediction time it receives the whole training set as context and the test rows as a query, and produces class probabilities in one forward pass. There is no gradient step on your data; fitting is memorisation of the context, which is why it takes seconds.

- **TabPFN v2** (Hollmann et al., *Nature*, January 2025): built for tables with **up to about 10,000 rows and 500 features**; on that regime it outperformed tuned GBMs in the paper's benchmark. The `tabpfn` package reached 9.0.0 on 15 September 2026; weights download from Hugging Face on first use.
- **TabICL** (Qu et al., ICML 2025): a column-then-row attention design that scales in-context learning to **about 100,000 rows** (the paper reports runs to 500,000). `tabicl` 2.2.0 shipped on 2 September 2026.
- **TabArena** (Erickson et al., June 2025, [tabarena.ai](https://tabarena.ai)): a *living* benchmark with a public leaderboard. Its 2026 standings put tabular foundation models at the top of the **single-model** ranking on small and medium tables, while tuned and ensembled gradient boosting and modern MLPs (TabM, RealMLP) stay competitive, and GBMs remain the default above the foundation models' size limits.

</div>

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 💻 The mechanism, not the package

`create_model` accepts **any object with the scikit-learn interface** (`fit`, `predict`, `predict_proba`, `get_params`), not only a string id. That is the whole integration: PyCaret wraps your object in its pipeline, cross-validates it and gives it the same `pull()`, `plot_model`, `blend_models` and `save_model` treatment as a built-in. The cell tries `tabpfn`, then `tabicl`, and otherwise falls back to a small estimator class written in the cell, so the mechanism runs everywhere.

</div>


In [ ]:
# ----------------------------------------------------------------- a foundation model (or a stand-in) as a PyCaret estimator
# !pip install -q tabpfn      # TabPFN v2+ (needs torch; downloads weights on first fit)
# !pip install -q tabicl      # TabICL (needs torch)
from sklearn.base import BaseEstimator, ClassifierMixin

class TabularStandIn(BaseEstimator, ClassifierMixin):
    # A minimal sklearn-compatible classifier: anything with this shape can go into create_model(estimator=...).
    def __init__(self, max_iter=150, learning_rate=0.06):
        self.max_iter, self.learning_rate = max_iter, learning_rate
    def fit(self, X, y):
        self.model_ = HistGradientBoostingClassifier(max_iter=self.max_iter, learning_rate=self.learning_rate,
                                                     random_state=RANDOM_STATE).fit(X, y)
        self.classes_ = self.model_.classes_
        return self
    def predict(self, X): return self.model_.predict(X)
    def predict_proba(self, X): return self.model_.predict_proba(X)

fm_name, fm = None, None
try:
    from tabpfn import TabPFNClassifier
    fm, fm_name = TabPFNClassifier(device="cpu"), "TabPFN"
except Exception as e:
    print(f"tabpfn unavailable ({type(e).__name__}); trying tabicl")
    try:
        from tabicl import TabICLClassifier
        fm, fm_name = TabICLClassifier(device="cpu"), "TabICL"
    except Exception as e2:
        print(f"tabicl unavailable ({type(e2).__name__}); using the stand-in class defined above")
        fm, fm_name = TabularStandIn(), "TabularStandIn (HGB inside)"

exp = setup(train_df.drop(columns=DROP), target=LABEL_CLS, test_data=test_df.drop(columns=DROP),
            session_id=RANDOM_STATE, fold=3, n_jobs=4, index=False, verbose=False, html=False, system_log=False)
with timer(f"create_model(estimator={fm_name}, fold=3)"):
    fm_model = create_model(fm, fold=3, verbose=False)          # an object, not a string id
    cv_fm = pull()
cv_lgbm_again = None
lgbm2 = create_model("lightgbm", fold=3, verbose=False); cv_lgbm_again = pull()

cv_table = pd.DataFrame({fm_name: cv_fm.loc["Mean", ["AUC", "F1", "Recall", "Prec."]],
                         "lightgbm": cv_lgbm_again.loc["Mean", ["AUC", "F1", "Recall", "Prec."]]}).T.astype(float)
display(pretty(cv_table))
pred_fm = predict_model(fm_model, data=test_df.drop(columns=DROP), raw_score=True, verbose=False)
card_fm = report_card(y_te, pred_fm["prediction_label"].astype(int), pred_fm["prediction_score_1"], name=fm_name)
print(f"{fm_name}: CV AUC {cv_table.loc[fm_name, 'AUC']:.3f} vs lightgbm {cv_table.loc['lightgbm', 'AUC']:.3f}  |  test AUC {card_fm.roc_auc.iloc[0]:.3f}, PR-AUC {card_fm.pr_auc.iloc[0]:.3f}")
print(f"pipeline steps around the estimator: {[name for name, _ in get_config('pipeline').steps]} + {type(fm_model).__name__}")


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ What just happened

PyCaret took an object it had never heard of, put it behind its own imputation and encoding steps, cross-validated it three times and returned the same table it returns for `"lightgbm"`. If the cell found `tabpfn` or `tabicl`, the row you see is a real foundation model; if not, it is the stand-in class, and the *code path is identical*. Swap the object and re-run; nothing else in the notebook changes. That is why the `estimator=` hook matters more than any individual model: it makes the low-code lifecycle a frame that the 2026 model of the year drops into.

</div>

## 25.4 Where each family wins: the size map

The figure places the four families on the axis that decides most of it in 2026, **training rows**, using the limits the papers state: TabPFN v2's 10,000-row design point, TabICL's 100,000-row regime, and the point beyond which a GPU pays. The budget table's own size is marked.


In [ ]:
# ----------------------------------------------------------------- qualitative size map with the stated design limits
fig, ax = plt.subplots(figsize=(11, 4.6))
bands = [("Tabular foundation models (TabPFN v2)",      50,     10_000,  "#9c27b0"),
         ("Tabular foundation models (TabICL)",         1_000,  100_000, "#ba68c8"),
         ("Gradient boosting (LightGBM / CatBoost / XGBoost)", 200, 50_000_000, "#2196F3"),
         ("PyCaret compare_models sweet spot",          200,    2_000_000, "#28a745"),
         ("GPU pays off (RAPIDS cuML, use_gpu=True)",   300_000, 50_000_000, "#ffc107")]
for i, (label, lo, hi, col) in enumerate(bands):
    ax.barh(i, np.log10(hi) - np.log10(lo), left=np.log10(lo), color=col, alpha=0.75, edgecolor="white", height=0.62, label=label)
n_train = len(train_df)
ax.axvline(np.log10(n_train), color="#dc3545", ls="--", lw=2)
ax.text(np.log10(n_train) + 0.05, len(bands) - 0.55, f"this notebook: {n_train:,} training rows", color="#dc3545", fontsize=10, fontweight="bold")
ax.set_yticks(range(len(bands))); ax.set_yticklabels([b[0] for b in bands], fontsize=9.5)
ticks = [2, 3, 4, 5, 6, 7]; ax.set_xticks(ticks); ax.set_xticklabels([f"{10**t:,}" for t in ticks])
ax.set_xlim(1.7, 7.7); ax.set_xlabel("training rows (log scale)"); ax.set_title("Which family fits which data size (design limits stated by the papers and vendors, not a benchmark)")
ax.invert_yaxis(); ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
plt.tight_layout(); plt.show()
print(f"At {n_train:,} rows this table sits inside every band except the GPU one, which starts at {bands[-1][1]:,} rows.")


<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 📊 What this shows

Each bar is the row range where a family is designed to operate; the dashed line is this notebook's training set. At a thousand rows you are inside the foundation-model band, the gradient-boosting band and PyCaret's comfortable range at once, which is exactly why 25.3 could compare them in seconds. Move right by two decades and the purple bars end: TabPFN v2 is not built for a million rows, and TabICL is stretching. Gradient boosting runs the full width, and the GPU band begins where a CPU fit stops finishing inside a meeting.

**The common misread:** treating the bars as accuracy. They are *applicability* ranges. Within the overlap, TabArena's 2026 leaderboard says foundation models often win per single model, and tuned GBM ensembles stay close; outside the purple bars the question does not arise.

</div>

## 25.5 Task 25 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- Hand-built scikit-learn, PyCaret, AutoGluon and RAPIDS solve the same fitting problem with different owners of the decisions; on this table the hand-built GBM and PyCaret's LightGBM landed within a few hundredths of AUC.
- `use_gpu=True` swaps **engines** (cuML, XGBoost/LightGBM/CatBoost device flags); it does not touch the tree, Bayes or discriminant models, and it warns and falls back when cuML is missing.
- `create_model(estimator=<object>)` accepts any sklearn-compatible object, which is how TabPFN, TabICL or your own class enter the PyCaret lifecycle.
- Tabular foundation models lead TabArena's single-model ranking on small and medium tables in 2026; gradient boosting remains the default beyond their size limits and for latency-bound serving.

### 🔑 New variables

| Variable | Meaning |
|---|---|
| `hand`, `p_hand` | the hand-built sklearn pipeline and its test probabilities |
| `lgbm`, `cv_lgbm`, `cards` | PyCaret's LightGBM, its CV table, and the two-row report card |
| `zoo`, `on_gpu` | the model registry (`models(internal=True)`) and the rows reporting a GPU engine |
| `TabularStandIn`, `fm`, `fm_name`, `fm_model` | the fallback estimator class, the chosen foundation model (or stand-in), and its fitted PyCaret model |
| `cv_table`, `card_fm` | CV comparison against LightGBM, and the foundation model's test report card |

### ➡️ Next Up

Task 26 turns these facts into a decision flowchart, states PyCaret's release status honestly, and names what to graduate to.

</div>


# Task 26 · The honest 2026 verdict

## 26.1 The decision flowchart

<div style="background-color: #e7f3fe; border-left: 5px solid #2196F3; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🧠 Four questions, in this order

```
START: a tabular prediction problem
│
├─ 1. Is the data tabular only?
│      no  → images / free text / audio / sequences  →  deep learning (PyTorch, Keras) or AutoGluon MultiModal
│      yes ↓
├─ 2. How many training rows?
│      under ~10k   → try a tabular foundation model (TabPFN v2) *and* a GBM; keep the calibrated winner (Part 4)
│      10k to ~1M   → gradient boosting; PyCaret compare → tune → blend is the fast path
│      over ~1M     → RAPIDS cuML / GPU GBMs, or AutoGluon with a time budget
│      ↓
├─ 3. Do you need the best number, or an explainable one you control?
│      best number         → AutoGluon (`presets="best_quality"` or `"extreme"`), then interpret its leaderboard
│      control / custom loss / custom CV → hand-built scikit-learn pipeline
│      an explanation by tomorrow → PyCaret (setup → compare → interpret → finalize, Parts 1-9)
│      ↓
└─ 4. What does serving need?
       sub-millisecond latency, tiny image → a single GBM or linear model, exported as a plain pickle (Part 9)
       batch scoring, no latency constraint → an ensemble is fine (blend / stack / AutoGluon)
       a GPU at inference               → foundation models are acceptable; otherwise they are not
```

The order matters. Modality rules out most tools before size does; size rules out foundation models before accuracy is discussed; and serving constraints veto ensembles that won on the leaderboard.

</div>

## 26.2 PyCaret's release status, computed rather than remembered

The cell prints the versions actually running here, then the facts about where the project stands.


In [ ]:
# ----------------------------------------------------------------- the versions in this kernel
import sys, pycaret, sklearn, importlib.metadata as im
rows = {"python": sys.version.split()[0], "pycaret": pycaret.__version__, "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__, "numpy": np.__version__}
for pkg in ["lightgbm", "catboost", "xgboost", "shap", "imbalanced-learn", "mlflow", "cuml", "tabpfn", "tabicl"]:
    try:
        rows[pkg] = im.version(pkg)
    except im.PackageNotFoundError:
        rows[pkg] = "not installed"
ver = pd.DataFrame.from_dict(rows, orient="index", columns=["version"])
display(ver)
py_major_minor = tuple(int(x) for x in rows["python"].split(".")[:2])
print(f"PyCaret {rows['pycaret']} on Python {rows['python']}: " + ("inside the tested 3.9-3.11 window" if (3, 9) <= py_major_minor <= (3, 11) else "outside the 3.x-tested window; expect dependency conflicts"))
print(f"PyPI metadata for pycaret {rows['pycaret']} declares requires_python '>=3.9'; the 4.0 alphas declare '>=3.11'.")


<div style="background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ⚠️ Release status, as of September 2026

- **3.3.2 is the latest stable release and it dates from 28 April 2024.** It was tested on Python 3.9 to 3.11; on 3.12 the pinned dependencies frequently fail to install (PyCaret issue #4173). Fresh Colab runtimes move to newer Pythons, so the `!pip install pycaret` cell at the top of this notebook is the one most likely to break first. Pin a Python 3.11 kernel or a conda env when it does.
- **PyCaret 4.0 exists only as alphas** (4.0.0a8 on PyPI, 8 May 2026; Python 3.11 to 3.13; scikit-learn 1.7+, NumPy 2, pandas 2). It is a rewrite: a "lean core" with roughly half the dependencies, an **OOP-only** API (`ClassificationExperiment(target=..., session_id=...).fit(df)`), and a "control plane" web app around it. The functional `from pycaret.classification import *` style used in this notebook is not in the alpha. Do not build production on an alpha; do learn the OOP form from Part 1, because it is the one that survives.
- **scikit-learn moves faster than PyCaret pins it.** 3.3.x pins `scikit-learn < 1.5` at install time; every sklearn release since then has to be reconciled by hand or by waiting.

What to watch: the 4.0 changelog for a first beta, the `requires_python` line on PyPI, and whether `compare_models` gains native foundation-model containers (today you bring your own object, as in 25.3).

</div>

## 26.3 Graduate to

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### 🏆 Best practice: keep PyCaret as the first hour, not the last

| You need | Graduate to | What carries over from this notebook |
|---|---|---|
| The best accuracy on a fixed compute budget | **AutoGluon 1.6** (`TabularPredictor.fit(train, presets="best_quality")`; 1.6.0 shipped 5 August 2026 with TabPFN-3, TabICLv2, TabDPT-Turbo and Nori built in) | metrics (Part 3), calibration and thresholds (Part 4), the leakage and drift checks (Parts 5, 9) |
| Scale: millions of rows, kNN/SVM/UMAP | **RAPIDS cuML 26.x** (`cuml.accel` for zero-code sklearn, or `use_gpu=True` here) | everything; PyCaret's sklearn engines become cuML engines |
| Control: custom losses, custom CV, a minimal dependency list | **scikit-learn** pipelines, LightGBM / CatBoost directly | the pipeline object from Part 2 is an sklearn `Pipeline`; `get_config("pipeline")` hands it to you |
| Images, text, audio, long sequences | **deep learning** (PyTorch, Keras 3), or AutoGluon MultiModal | evaluation discipline (Parts 3, 10) and MLOps habits (Part 9) |
| Forecasting many series with covariates | AutoGluon `TimeSeriesPredictor` (Chronos, Toto-2), or `sktime` directly (PyCaret's `time_series` module is built on it) | the MASE bar and the naive baselines from Part 8 |

</div>

## 26.4 Task 26 Summary

<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 15px; border-radius: 8px; margin: 10px 0;">

### ✅ Key Takeaways

- Decide in this order: modality, then rows, then accuracy-versus-control, then serving constraints. Each question removes tools before the next one is asked.
- PyCaret 3.3.2 (April 2024) is the stable release; it expects Python 3.9 to 3.11 and an older scikit-learn pin. PyCaret 4.0 is alpha-only, OOP-only and Python 3.11+.
- PyCaret's durable value is the *lifecycle vocabulary* and the `estimator=` hook; the models you put through it will change.
- Graduate deliberately: AutoGluon for the number, RAPIDS for the scale, scikit-learn for the control, deep learning for other modalities. Every evaluation habit from Parts 3 to 10 goes with you.

### 🔑 New variables

| Variable | Meaning |
|---|---|
| `rows`, `ver` | the versions running in this kernel, as a dict and a table |
| `py_major_minor` | the kernel's Python (major, minor) tuple, used to judge the support window |

### ➡️ Next Up

The closing section: a cheat sheet per module, the metric table, the pitfalls this course showed you, and where to go next.

</div>


---

# 🏆 Cheat Sheet

## The lifecycle in one line per step

```
setup → compare_models → create_model → tune_model → ensemble_model / blend_models / stack_models
      → calibrate_model → optimize_threshold → interpret_model → finalize_model → predict_model → save_model
```

## Functions per module (functional API; every module also has an `*Experiment` class with the same methods)

| Step | `pycaret.classification` | `pycaret.regression` | `pycaret.clustering` | `pycaret.anomaly` | `pycaret.time_series` |
|---|---|---|---|---|---|
| Prepare | `setup(data, target, test_data=, fold=, session_id=, normalize=, fix_imbalance=, ...)` | `setup(..., transform_target=)` | `setup(data, normalize=)` (no target) | `setup(data)` (no target) | `setup(y, fh=, fold=, seasonal_period=)` |
| Survey | `compare_models(include=, fold=, sort="AUC", n_select=)` | `compare_models(sort="MAE")` | — (one model at a time) | — | `compare_models(include=["naive","snaive","ets","arima","theta"])` |
| Fit one | `create_model("lightgbm" \| estimator_object, fold=)` | `create_model("ridge")` | `create_model("kmeans", num_clusters=)` | `create_model("iforest", fraction=)` | `create_model("ets")` |
| Improve | `tune_model(m, n_iter=, optimize=, choose_better=True)` · `ensemble_model` · `blend_models` · `stack_models` · `calibrate_model` · `optimize_threshold` | same, minus calibration and threshold | — | — | `tune_model`, `blend_models` |
| Look | `plot_model(m, plot="auc"\|"pr"\|"confusion_matrix"\|"threshold"\|"calibration"\|"learning"\|"feature"\|"boundary"\|"vc")` · `interpret_model(m, plot="summary"\|"reason"\|"pdp")` | `plot_model(plot="residuals"\|"error"\|"feature"\|"learning")` | `plot_model(plot="elbow"\|"silhouette"\|"cluster"\|"distribution")` | `plot_model(plot="tsne")` | `plot_model(plot="forecast"\|"diagnostics"\|"decomp")` · `check_stats()` |
| Label | `predict_model(m, data=, raw_score=True, probability_threshold=)` | `predict_model(m, data=)` | `assign_model(m)` · `predict_model(m, data=)` | `assign_model(m)` → `Anomaly`, `Anomaly_Score` | `predict_model(m, fh=, return_pred_int=True)` |
| Ship | `finalize_model` · `save_model` / `load_model` · `create_api` · `create_docker` · `create_app` | same | `save_model` / `load_model` | same | same |
| Audit | `pull()` · `get_config("pipeline" \| "X_train_transformed")` · `get_metrics()` / `add_metric()` · `check_fairness(m, sensitive_features=)` · `get_logs()` | same | `pull()` | `pull()` | `pull()` |

## Arguments you will type every time

`session_id=RANDOM_STATE` (reproducibility) · `verbose=False, html=False` (scripts and CI) · `test_data=` (one held-out set for every number) · `fold=3` while exploring, `fold=10` for the final table · `n_jobs=4` on shared machines · `raw_score=True` for the positive-class probability.

## Metrics: when to use which

| Family | Metric | Use it when | Do not trust it when |
|---|---|---|---|
| Threshold (classification) | Accuracy | classes are balanced and errors cost the same | the positive rate is low (this notebook's is about one in six) |
| | Precision / Recall / F1 | one error type matters more; F1 when both do | you have not chosen the threshold deliberately (Part 4) |
| | MCC, Kappa | you want one balanced number that survives imbalance | you need to explain it to a stakeholder in one sentence |
| Ranking | ROC-AUC | you compare rankers regardless of threshold | positives are rare and you care about the top of the list |
| | PR-AUC (average precision) | positives are rare; the top-k matters | the base rate differs between datasets you compare |
| Probabilistic | Log loss | probabilities feed a decision or a downstream model | a few confident mistakes dominate (it is unbounded) |
| | Brier score | you want a bounded, calibration-aware error | classes are extremely imbalanced (the base rate hides in it) |
| Calibration | reliability curve, ECE | probabilities will be read as probabilities | there are too few positives per bin |
| Regression, scale | MAE | errors cost linearly; the median matters | outliers must be punished more |
| | RMSE | large errors must be punished more | the target has a heavy tail and RMSE is chasing three rows |
| Regression, relative | MAPE | the audience thinks in percent | the target crosses zero or is small |
| | R² | you compare against "predict the mean" | you compare across datasets with different variance |
| Forecasting | MASE, RMSSE | you compare series with different scales; **the naive forecast scores 1.0** | seasonality is absent and the in-sample naive error is tiny |
| | SMAPE | the audience wants a percentage with a bounded range | actuals are near zero |
| Clustering, internal | Silhouette | you choose k without labels | clusters are non-convex (DBSCAN shapes) |
| | Calinski-Harabasz, Davies-Bouldin | a fast internal sanity check | you treat them as evidence the clusters are real |
| Anomaly | precision@k | a human will inspect the top k | `fraction` was chosen by assumption, which it always is |

---

# ⚠️ Common Pitfalls to Avoid

1. **Reading `compare_models` scores as test scores.** They are cross-validation means on the training data. Report on `test_data` once, with `predict_model` (Task 2, Task 6).
2. **Running `setup` on the full table before splitting.** Imputation and scaling statistics then see the test rows; pass `test_data=` so the split is yours (Task 5, Task 13).
3. **Predicting this month's total from this month's categories.** R² near 1 and importance concentrated on a column that sums to the label are a leak, not a result (Task 13).
4. **Trusting `prediction_score` as the positive-class probability.** It is the score of the *predicted* class; use `raw_score=True` and read `prediction_score_1` (Task 2, Task 7).
5. **Accuracy on a one-in-six positive rate.** Predicting "no" for everyone scores well above 80 percent; use PR-AUC, recall at a chosen threshold, and the cost matrix (Task 7, Task 11).
6. **Assuming `tune_model` improves the model.** With `choose_better=True` PyCaret returns whichever is better; without it a worse tuned model is returned silently (Task 9).
7. **Blending or stacking without calibrating.** Averaged probabilities are not calibrated; read the reliability diagram before thresholding (Task 10, Task 11).
8. **Treating `fraction` in anomaly detection, or `k` in clustering, as measured.** They are assumptions; profile the flagged rows and the clusters against something you know (Tasks 16-17).
9. **Forecasting 24 monthly points without a naive baseline.** MASE below 1.0 is the bar; a seasonal model that does not beat seasonal-naive learned nothing (Task 18).
10. **Shipping the un-finalized model.** `finalize_model` refits on train plus test; the test numbers no longer describe it, but the version you save should have seen all the data (Task 20).
11. **Skipping the drift check because the model was good at launch.** A scaled income or a shifted dining spend moves the score distribution long before labels arrive; PSI per feature and a retrain trigger are part of the deliverable (Task 22).
12. **Picking a tool by leaderboard.** Modality, row count and serving constraints decide before accuracy does (Task 26).

---

# 🚀 What's Next

**Do one real thing end to end.** Take a table you actually care about and run the whole lifecycle in one sitting: `setup` with `test_data=`, `compare_models` on a short `include=` list, `tune_model` on the winner, `calibrate_model`, `optimize_threshold` against a cost you can defend, `interpret_model` for two rows, `finalize_model`, `save_model`, and a `predict_model` on rows the model never saw. Then:

1. **Re-run the same table through AutoGluon** with a 5-minute budget and compare its leaderboard to your `compare_models` table. Where they disagree is where you learn.
2. **Drop a foundation model into 25.3** on a Colab GPU (`pip install tabpfn`) and see whether the row count puts it ahead.
3. **Turn Part 9 into a job.** Schedule the drift cell to run weekly on new rows and page yourself when PSI crosses your threshold.
4. **Read the OOP form once more** (Part 1, `ClassificationExperiment`). It is the API that continues into PyCaret 4.
5. **Hand-build the winning pipeline in plain scikit-learn** using `get_config("pipeline")` as the answer key. When you can rebuild it, you own it.

### Where to go deeper in this repository

| Topic | Notebook |
|---|---|
| The full-AutoML alternative | `colabs/06_data_eval_mlops/autogluon_zero_to_hero.ipynb` |
| The GPU stack | `colabs/06_data_eval_mlops/nvidia_rapids_zero_to_hero.ipynb` |
| Metrics from first principles | `colabs/08_data_mining/logistic_regression_zero_to_hero.ipynb` |
| Anomaly detection in depth | `colabs/08_data_mining/anomaly_detection_zero_to_hero.ipynb` |
| MLOps end to end | `colabs/06_data_eval_mlops/mlops_level3_end_to_end_tutorial.ipynb` |

---

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 34px; border-radius: 20px; color: white; text-align: center; margin: 24px 0;">

## 🎓 Course Complete

You started with a household budget export and a question: who overspends next month?

You finish able to set up a preprocessing pipeline deliberately, compare a model zoo in one call and read the table critically, tune, blend, stack and calibrate, pick a threshold for a cost you can defend, explain a prediction to the household it concerns, forecast the series, find the odd households, ship the model with a saved pipeline, watch it for drift, and know exactly when to reach for a different tool.

The most valuable habit is not a PyCaret call. It is the question to ask of every number:

### *"Which split did this come from, and which default did I accept to get it?"*

Now go and run the lifecycle on something real.

</div>

---

# 📚 References

**PyCaret**
- PyCaret documentation, 3.x: [pycaret.gitbook.io/docs](https://pycaret.gitbook.io/docs) (setup parameters, model ids, the plot and interpret galleries)
- PyCaret source and releases: [github.com/pycaret/pycaret](https://github.com/pycaret/pycaret/releases). 3.3.2 (28 April 2024) is the latest stable release; the 4.0.0 alphas (a8, 8 May 2026) are a Python 3.11+ OOP-only rewrite
- Python 3.12 install failures on 3.3.x: PyCaret issue #4173

**scikit-learn and the pieces PyCaret wraps**
- scikit-learn user guide: `Pipeline`, `ColumnTransformer`, model evaluation, calibration, `permutation_importance`
- LightGBM, CatBoost and XGBoost documentation (the GPU flags in 25.2 are theirs)
- `imbalanced-learn` (SMOTE and the `fix_imbalance` family), `sktime` (under `pycaret.time_series`)

**Explainability**
- Lundberg and Lee, *A Unified Approach to Interpreting Model Predictions* (NeurIPS 2017); the SHAP library used by `interpret_model`

**Tabular foundation models and benchmarks**
- Hollmann et al., *Accurate predictions on small data with a tabular foundation model* (TabPFN v2), *Nature*, January 2025; [github.com/PriorLabs/TabPFN](https://github.com/PriorLabs/TabPFN)
- Qu et al., *TabICL: A Tabular Foundation Model for In-Context Learning on Large Data*, ICML 2025; [arXiv:2502.05564](https://arxiv.org/abs/2502.05564)
- Erickson et al., *TabArena: A Living Benchmark for Machine Learning on Tabular Data*, June 2025; [arXiv:2506.16791](https://arxiv.org/abs/2506.16791), leaderboard at [tabarena.ai](https://tabarena.ai)

**The neighbours**
- AutoGluon 1.6 (1.6.0 released 5 August 2026): [auto.gluon.ai](https://auto.gluon.ai), tabular foundation models tutorial and the `extreme` preset
- RAPIDS cuML 26.x, `cuml.accel` zero-code-change acceleration: [docs.rapids.ai/api/cuml](https://docs.rapids.ai/api/cuml/stable/)

---

<div style="text-align: center; color: #5F6368; font-size: 0.9em; padding: 12px;">
Built for the CMPE 258 zero-to-hero Colab series · verified against <b>PyCaret 3.3.2</b> · every cell runs on a CPU without an API key
</div>
